In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/hauuto/catbreed-id.git"
BRANCH = os.environ.get("CATBREED_BRANCH", "main")

# Kaggle: /kaggle/working. Máy khác (VPS): đặt CATBREED_BASE, mỗi lượt chạy song song một thư mục riêng
BASE_DIR = Path(os.environ.get("CATBREED_BASE", "/kaggle/working"))
INPUT_DIR = Path(os.environ.get("CATBREED_INPUT", "/kaggle/input"))
WORK_DIR = BASE_DIR / "catbreed-id"
OUTPUTS_DIR = BASE_DIR / "outputs"
HF_REPO_ID = "catbreed-id/catbreed-experiments"

In [ ]:
import subprocess
import sys

if not WORK_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(WORK_DIR)],
        check=True,
    )
else:
    changes = subprocess.check_output(
        ["git", "status", "--porcelain"],
        cwd=WORK_DIR,
        text=True,
    ).strip()

    if changes:
        raise RuntimeError(
            "Repo có thay đổi chưa commit. Kiểm tra trước khi cập nhật."
        )

    subprocess.run(
        ["git", "switch", BRANCH],
        cwd=WORK_DIR,
        check=True,
    )
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", BRANCH],
        cwd=WORK_DIR,
        check=True,
    )

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "-r", str(WORK_DIR / "requirements.txt"),
    ],
    check=True,
)

GIT_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=WORK_DIR,
    text=True,
).strip()

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print("Code sẵn sàng:", GIT_COMMIT)

In [ ]:
import os

from huggingface_hub import HfApi
try:  # Kaggle: lấy từ Secrets; máy khác: biến môi trường HF_TOKEN
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except ImportError:
    if not os.environ.get("HF_TOKEN"):
        raise RuntimeError("Ngoài Kaggle cần đặt biến môi trường HF_TOKEN.")

api = HfApi(token=os.environ["HF_TOKEN"])
RUN_OWNER = api.whoami()["name"]

# Xác nhận tài khoản truy cập được kho kết quả.
api.repo_info(repo_id=HF_REPO_ID, repo_type="dataset")

print("Tài khoản:", RUN_OWNER)
print("Kho kết quả:", HF_REPO_ID)
print("Runner sẵn sàng")

In [ ]:
import importlib
import json
from datetime import datetime, timezone
from uuid import uuid4

import torch
import yaml

if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

importlib.invalidate_caches()
from src.training.artifacts import save_stage, sync_results

run_id = (
    datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    + "_smoke_"
    + uuid4().hex[:8]
)
run_dir = OUTPUTS_DIR / run_id
run_dir.mkdir(parents=True, exist_ok=False)

info = {
    "run_id": run_id,
    "owner": RUN_OWNER,
    "git_commit": GIT_COMMIT,
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "Kiểm tra workflow, không huấn luyện",
    "is_smoke_test": True,
    "data_version": None,
    "split_id": None,
    "training_seed": None,
    "status": "running",
}

def write_info():
    (run_dir / "info.json").write_text(
        json.dumps(info, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

write_info()

(run_dir / "config.yaml").write_text(
    yaml.safe_dump({
        "mode": "smoke",
        "stages": ["frozen", "finetuned"],
        "hf_repo_id": HF_REPO_ID,
    }),
    encoding="utf-8",
)

(run_dir / "history.csv").write_text(
    "stage,event\nfrozen,smoke_checkpoint\nfinetuned,smoke_checkpoint\n",
    encoding="utf-8",
)

(run_dir / "requirements_freeze.txt").write_text(
    subprocess.check_output(
        [sys.executable, "-m", "pip", "freeze"],
        text=True,
    ),
    encoding="utf-8",
)

sync_status = {}

for stage, value in [("frozen", 1.0), ("finetuned", 2.0)]:
    sync_status[stage] = save_stage(
        run_dir=run_dir,
        stage=stage,
        checkpoint={
            "is_smoke_test": True,
            "tensor": torch.tensor([value]),
        },
        metrics={"is_smoke_test": True, "stage": stage},
        repo_id=HF_REPO_ID,
    )

info["status"] = "completed"
info["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
write_info()

sync_status["final"] = sync_results(run_dir, HF_REPO_ID)

print("\nRun ID:", run_id)
for step, success in sync_status.items():
    print(f"{step}: {'Đã đồng bộ' if success else 'Chưa đồng bộ'}")

if not all(sync_status.values()):
    raise RuntimeError(
        f"Có bước chưa đồng bộ. Giữ thư mục để upload lại: {run_dir}"
    )

print("Workflow thử hoàn thành")

## 1. Cấu hình thực nghiệm

- **Làm:** khai báo mọi tham số của giao thức trong `CONFIG`.
- `profile` là `smoke` (chạy thử ngắn, không đẩy kết quả lên Hugging Face) hoặc `official` (chạy đủ, cần GPU và commit sạch).


In [ ]:
CONFIG = {
    "profile": "smoke",  # smoke: chạy thử ngắn, không đẩy lên Hugging Face; official: chạy đủ giao thức;
    # sweep: chỉ pha A cho từng điểm của sweep_grid, không khóa lựa chọn, không giải mã test hay OOD test
    "sweep_grid": [],  # profile sweep: [{"name": "...", "overrides": {khóa CONFIG: giá trị}, "models": [...] (tùy chọn)}]
    "model_overrides": {},  # ghi đè theo mô hình, ví dụ {"efficientnet_b0": {"backbone_lr": 1e-4}}
    "baseline_batchnorm": False,  # True: thêm BatchNorm sau mỗi Conv của CNN baseline
    "head_hidden": 0,  # 0: Dropout → Linear (§2.5.1); > 0: Dropout → Linear(d, h) → ReLU → Dropout → Linear(h, 12)
    "fine_tune_fraction": None,  # None: dùng fine_tune_blocks; số trong (0, 1]: mở khóa các khối cuối đến khi đạt tỷ lệ tham số này
    "train_fraction": 1.0,  # chỉ profile sweep: lấy mẫu phân tầng theo giống một phần tập train (đường học)
    "train_fraction_seed": 42,  # seed lấy mẫu tập train, cố định cho mọi seed huấn luyện
    "protocol_id": "Nhom09_QD1-QD10_2026-10-18",
    "models": ["resnet50", "mobilenet_v2", "efficientnet_b0", "cnn_baseline"],
    "training_seeds": [42, 43, 44],  # QĐ4
    "run_augmentation_ablation": True,  # 3 backbone × 3 seed huấn luyện lại không tăng cường
    "owner": "Tai",
    "data_input_root": str(INPUT_DIR),
    "far_ood_root": str(INPUT_DIR),
    "official_archives_root": str(INPUT_DIR),  # tìm images.tar.gz, annotations.tar.gz, dtd-r1.0.1.tar.gz gốc nếu đã gắn
    "official_data_dir": str(BASE_DIR / "official_data"),  # nơi tải (nếu chưa gắn) và giải nén archive gốc
    "unseen_cat_root": str(INPUT_DIR),  # tùy chọn; không có thư mục lớp thì bỏ qua
    "unseen_cat_classes": ["Tabby", "Mixed Breed", "Local"],  # FeliBreed
    "output_root": str(OUTPUTS_DIR),
    "clean_kaggle_output": True,  # cuối lượt chỉ giữ tệp zip trong Output; thư mục chưa đồng bộ được giữ lại
    "image_size": 224,
    "resize_size": 256,  # §2.4.2
    "batch_size": 32,
    "workers": 4,  # tiến trình nạp ảnh huấn luyện; đổi số này làm đổi dãy tăng cường của mỗi seed
    "io_threads": 16,  # luồng đọc tệp khi kiểm tra và giải mã ảnh
    "pretrained": True,
    "frozen_epochs": 10,
    "finetuned_epochs": 20,
    "baseline_epochs": 30,
    "frozen_lr": 1e-3,
    "head_lr": 1e-4,
    "backbone_lr": 1e-5,
    "baseline_lr": 1e-3,
    "weight_decay": 1e-4,
    "dropout": 0.3,
    "early_stopping_patience": 7,
    "scheduler_patience": 3,
    "fine_tune_blocks": 2,
    "knn_k": 10,
    "ece_bins": 15,
    "bootstrap_repeats": 1000,  # §2.7.5
    "bootstrap_seed": 42,
    "deployment_max_megabytes": 25,  # §2.10.1
    "progress_interval_seconds": 15,
    "command_timeout_seconds": 300,
    "sync_timeout_seconds": 600,
    "download_timeout_seconds": 600,
    "network_read_timeout_seconds": 30,
    "hf_repo_id": "catbreed-id/catbreed-experiments",
    "smoke_images_per_class": 2,
    "smoke_ood_images": 24,
    "smoke_overrides": {
        "batch_size": 8,
        "frozen_epochs": 1,
        "finetuned_epochs": 1,
        "baseline_epochs": 1,
        "bootstrap_repeats": 50,
    },
}
SUBMISSION = {}  # Metadata nguồn của lượt chạy, không chứa token
AUTO_RUN = True


## 2. Thư viện, kiểu hình và hàm dùng chung

- **Làm:** nạp thư viện, ghi nhật ký, thanh tiến độ, đọc tệp song song, hằng số 12 giống mèo, kiểu hình chung, hàm gom bảng.
- **Xuất:** một dòng phiên bản Python, PyTorch và thiết bị.


In [ ]:
# ─── 2.1 Thư viện và thiết bị ────────────────────────────────────────────────────────────────────────
import hashlib
import importlib
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import traceback
import tarfile
import zipfile
import zlib
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from contextlib import contextmanager
from dataclasses import dataclass, field
from datetime import datetime, timezone
from io import BytesIO
from itertools import combinations
from pathlib import Path
from urllib.request import urlopen
from uuid import uuid4
from xml.etree import ElementTree as ET

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
from cycler import cycler
from IPython.display import FileLink, display
from matplotlib.patches import Patch, Rectangle
from matplotlib.ticker import MaxNLocator
from PIL import Image
from scipy.optimize import minimize_scalar
from scipy.special import logsumexp, softmax
from scipy.stats import binomtest, rankdata
from sklearn.covariance import LedoitWolf
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF
from tqdm import tqdm
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # Phụ lục B: chế độ tất định
IS_KAGGLE = os.path.isdir("/kaggle/working")
# Môi trường runner của nhóm: Kaggle, hoặc máy khác đặt CATBREED_RUNNER=1 (được đồng bộ HF, chạy sweep/official)
ON_KAGGLE = IS_KAGGLE or os.environ.get("CATBREED_RUNNER") == "1"
print(
    "Python", sys.version.split()[0],
    "| torch", torch.__version__,
    "| torchvision", torchvision.__version__,
    "| thiết bị", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    flush=True,
)


# ─── 2.2 Nhật ký, thanh tiến độ và đọc tệp song song ─────────────────────────────────────────────────
def log_event(step, status="info", **details):
    # Sự kiện JSON chỉ ghi vào tệp; màn hình dành cho thanh tiến độ và dòng tóm tắt
    root = Path(CONFIG["output_root"])
    root.mkdir(parents=True, exist_ok=True)
    record = {"time": datetime.now(timezone.utc).isoformat(), "step": step, "status": status, **details}
    with (root / "progress_events.jsonl").open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(record, ensure_ascii=False, allow_nan=False) + "\n")


@contextmanager
def timed_step(label):
    started = time.monotonic()
    log_event(label, "started")
    try:
        yield
    except BaseException as exc:
        log_event(label, "failed", error_type=type(exc).__name__)
        raise
    log_event(label, "completed", elapsed_seconds=round(time.monotonic() - started, 1))


def run_step(label, function, *args, **kwargs):
    with timed_step(label):
        return function(*args, **kwargs)


def progress(items, label, total=None, leave=False):
    if total is None and hasattr(items, "__len__"):
        total = len(items)
    started = previous = time.monotonic()
    log_event(label, "started", total=total)
    with tqdm(total=total, desc=label, file=sys.stdout, ascii=True, mininterval=1, leave=leave) as bar:
        for completed, item in enumerate(items, 1):
            yield bar, item
            bar.update(1)
            now = time.monotonic()
            if completed == total or now - previous >= CONFIG["progress_interval_seconds"]:
                log_event(
                    label,
                    "progress",
                    completed=completed,
                    total=total,
                    elapsed_seconds=round(now - started, 1),
                    metrics=bar.postfix,
                )
                previous = now


def parallel_map(function, items, label, leave=False):
    # Đọc tệp bằng nhiều luồng; kết quả giữ đúng thứ tự của items
    with ThreadPoolExecutor(CONFIG["io_threads"]) as pool:
        results = progress(pool.map(function, items), label, total=len(items), leave=leave)
        return [result for _, result in results]


def run_command(arguments, label, capture=False, timeout=None):
    timeout = CONFIG["command_timeout_seconds"] if timeout is None else timeout
    with timed_step(label):
        return subprocess.run(
            arguments, check=True, text=True, capture_output=capture, timeout=timeout
        )


# ─── 2.3 Giống mèo, hằng số giao thức và hàm tiện ích ────────────────────────────────────────────────
CAT_BREEDS = [
    "Abyssinian",
    "Bengal",
    "Birman",
    "Bombay",
    "British_Shorthair",
    "Egyptian_Mau",
    "Maine_Coon",
    "Persian",
    "Ragdoll",
    "Russian_Blue",
    "Siamese",
    "Sphynx",
]
CLASS_TO_IDX = {name: index for index, name in enumerate(CAT_BREEDS)}
CLASS_INDICES = list(range(len(CAT_BREEDS)))
EXPECTED_CAT_COUNTS = dict(
    zip(CAT_BREEDS, [198, 200, 200, 184, 200, 190, 200, 200, 200, 200, 199, 200])
)
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
SPLIT_POLICY, SPLIT_SEED = "official_test_trainval_80_20", 42  # QĐ1
EXPECTED_SPLIT_ID = "3b283d422973d497d268d935ef0fe4a76bbd2d10f4b5efdef20a8d859949cfcb"
PROBE_DOGS_PER_BREED, PROBE_SEED = 20, 42  # §2.3.1: tập thăm dò 25 giống × 20 ảnh chó trainval
EXPECTED_PROBE_ID = "47a50e31b6f609c32aaadb30faa60614bcf5ac518c71fbde68ad8fe045947daf"
OOD_SPLITS = ["near_ood", "far_ood", "unseen_cat"]
MODEL_NAMES = ["resnet50", "mobilenet_v2", "efficientnet_b0", "cnn_baseline"]
MODEL_LABELS = {
    "resnet50": "ResNet50",
    "mobilenet_v2": "MobileNetV2",
    "efficientnet_b0": "EfficientNet-B0",
    "cnn_baseline": "CNN baseline",
}
STAGE_LABELS = {"frozen": "đóng băng", "finetuned": "tinh chỉnh", "scratch": "từ đầu"}
OOD_LABELS = {
    "probe_ood": "Tập thăm dò (chó trainval)",
    "near_ood": "Near-OOD (chó test)",
    "far_ood": "Far-OOD (DTD)",
    "unseen_cat": "Mèo ngoài 12 giống",
}


def configuration_label(model, stage):
    return MODEL_LABELS[model] + " " + STAGE_LABELS[stage]


def json_hash(data):
    return hashlib.sha256(json.dumps(data, sort_keys=True).encode()).hexdigest()


def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(
        json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8"
    )
    temporary.replace(path)


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def save_info(run_dir, info):
    write_json(run_dir / "info.json", {key: value for key, value in info.items() if key != "history"})


def file_hash(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def run_stamp():
    return datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def read_rgb(path):
    # Đọc header thực tế, kể cả PNG mang đuôi .jpg. Alpha ghép nền xám có quy định rõ.
    with Image.open(path) as source:
        source.load()
        if source.mode in ["RGBA", "LA"] or "transparency" in source.info:
            rgba = source.convert("RGBA")
            return Image.alpha_composite(
                Image.new("RGBA", rgba.size, (128, 128, 128, 255)), rgba
            ).convert("RGB")
        return source.convert("RGB")


def notebook_source_hash(notebook):
    content = [(c["cell_type"], "".join(c["source"])) for c in notebook["cells"]]
    return hashlib.sha256(json.dumps(content, ensure_ascii=False).encode()).hexdigest()


# ─── 2.4 Kiểu hình chung và hàm vẽ dùng lại ──────────────────────────────────────────────────────────
# Kiểu hình chung: nền trắng, lưới mảnh, bỏ khung trên và phải. Màu lấy từ một bảng phân biệt được khi mù màu;
# các chuỗi còn khác nhau bằng ký hiệu hoặc kiểu nét nên vẫn đọc được khi in trắng đen.
BLUE, ORANGE, AQUA, YELLOW, MAGENTA = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"
INK, MUTED, NEUTRAL, GRID = "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
plt.rcParams.update(
    {
        "figure.dpi": 100,
        "figure.facecolor": "white",
        "savefig.facecolor": "white",
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.labelsize": 11,
        "axes.labelcolor": MUTED,
        "axes.edgecolor": NEUTRAL,
        "axes.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.axisbelow": True,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "legend.fontsize": 9.5,
        "legend.frameon": False,
        "lines.linewidth": 2,
        "lines.markersize": 5,
        "axes.prop_cycle": cycler(color=[BLUE, ORANGE, AQUA, YELLOW, MAGENTA]),
    }
)
MODEL_COLORS = {"resnet50": BLUE, "mobilenet_v2": ORANGE, "efficientnet_b0": AQUA, "cnn_baseline": NEUTRAL}
MODEL_MARKERS = {"resnet50": "o", "mobilenet_v2": "s", "efficientnet_b0": "^", "cnn_baseline": "D"}
STAGE_LINES = {"frozen": "--", "finetuned": "-", "scratch": "-"}


def configuration_style(model, stage):
    # Màu và ký hiệu theo mô hình; giai đoạn đóng băng vẽ nét đứt, ký hiệu rỗng
    color = MODEL_COLORS[model]
    return {
        "color": color,
        "marker": MODEL_MARKERS[model],
        "linestyle": STAGE_LINES[stage],
        "markerfacecolor": "white" if stage == "frozen" else color,
        "markeredgecolor": color,
        "markeredgewidth": 1.5,
    }


def thousands(value):
    return f"{int(value):,}".replace(",", ".")  # 1.183 theo cách viết tiếng Việt


def figure_title(fig, text):
    fig.suptitle(text, x=0.01, ha="left", fontsize=13, fontweight="bold")


def legend_above(axis, columns=None, handles=None, row=()):
    # Chú thích nằm phía trên vùng vẽ, ngay dưới tiêu đề, nên không bao giờ che dữ liệu.
    # row: các ô cùng hàng, được nâng tiêu đề lên cùng mức để tiêu đề thẳng hàng.
    if handles is None:
        handles = axis.get_legend_handles_labels()[0]
    columns = columns or len(handles)
    axis.legend(handles=handles, loc="lower left", bbox_to_anchor=(0, 1), ncol=columns, borderaxespad=0.3)
    pad = 12 + 17 * math.ceil(len(handles) / columns)
    for panel in {axis, *row}:
        panel.set_title(panel.get_title(loc="left"), pad=pad)


def line_label(axis, value, text, vertical=True):
    # Ghi nhãn ngay cạnh đường mốc thay cho ô chú thích
    if vertical:
        axis.text(value, 0.98, " " + text, transform=axis.get_xaxis_transform(), va="top", fontsize=9.5, color=MUTED)
    else:
        axis.text(0.995, value, text, transform=axis.get_yaxis_transform(), ha="right", va="bottom", fontsize=9.5, color=MUTED)


def image_axis(axis):
    axis.set(xticks=[], yticks=[])
    axis.grid(False)
    for spine in axis.spines.values():
        spine.set_visible(False)


def draw_box(axis, box):
    # Viền trắng dày, nét đứt đen bên trong: hộp nổi trên cả nền sáng lẫn nền tối
    x0, y0, x1, y1 = box
    axis.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="white", linewidth=2.5))
    axis.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor=INK, linewidth=1, linestyle="--"))


def mean_markers(axis, x, y, error, style, label, horizontal=False):
    # Thanh sai số vẽ trước, ký hiệu vẽ đè lên để ký hiệu rỗng không bị thanh cắt ngang
    bars = {"xerr": error} if horizontal else {"yerr": error}
    axis.errorbar(x, y, fmt="none", ecolor=style["color"], elinewidth=1.2, capsize=2.5, **bars)
    axis.plot(x, y, markersize=7, label=label, zorder=3, **{**style, "linestyle": "none"})


def dot_rows(axis, labels, series):
    # Mỗi hàng một cấu hình; mỗi chuỗi gồm (nhãn, trung bình, độ lệch chuẩn, kiểu ký hiệu)
    positions = np.arange(len(labels))
    for shift, (label, mean, std, style) in zip(np.linspace(-0.16, 0.16, len(series)), series):
        mean_markers(axis, mean, positions + shift, np.nan_to_num(std), style, label, horizontal=True)
    axis.set_yticks(positions, labels)
    axis.invert_yaxis()
    axis.grid(axis="y", visible=False)


def show_figure(study, fig, name, section, description):
    # Mọi hình: lưu PNG 200 dpi, hiện ngay dưới cell, đóng figure và ghi mục lục hình
    path = study.directory / "figures" / (name + ".png")
    path.parent.mkdir(exist_ok=True)
    fig.savefig(path, dpi=200, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    study.figures.append({"file": "figures/" + path.name, "section": section, "description": description})


# ─── 2.5 Gom bảng kết quả, trung bình ± độ lệch chuẩn qua seed ───────────────────────────────────────
def collected(study, name):
    return pd.concat(study.tables[name], ignore_index=True)


def write_tables(study):
    for name in study.tables:
        collected(study, name).to_csv(study.directory / (name + ".csv"), index=False)


def report_table(study, name):
    return pd.read_csv(study.directory / "report_tables" / (name + ".csv"))


def mean_std(frame, keys, metrics):
    grouped = frame.astype({metric: float for metric in metrics}).groupby(keys, sort=False)[metrics]
    table = grouped.agg(["mean", "std"])  # Độ lệch chuẩn mẫu (ddof = 1) qua các seed
    table.columns = [f"{metric}_{statistic}" for metric, statistic in table.columns]
    table.insert(0, "n_seeds", grouped.size())
    return table.reset_index()


def mean_std_text(table, metrics, digits=4):
    shown = table.drop(columns=[f"{metric}_{s}" for metric in metrics for s in ["mean", "std"]])
    for metric in metrics:
        shown[metric] = [
            f"{mean:.{digits}f} ± {std:.{digits}f}" if pd.notna(std) else f"{mean:.{digits}f}"
            for mean, std in zip(table[metric + "_mean"], table[metric + "_std"])
        ]
    return shown


## 3. Nối với bốn cell gốc và khởi tạo study

- **Làm:** kiểm tra commit, nối hàm lưu và đồng bộ của nhóm (chỉ bật ở `official`), kiểm tra cấu hình, tạo thư mục study.
- **Xuất:** đường dẫn study, profile và thiết bị.


In [ ]:
# ─── 3.1 Nối với kết quả của bốn cell gốc ────────────────────────────────────────────────────────────
GROUP_SAVE_STAGE = None
GROUP_SYNC_RESULTS = None
RUNNER_PROVENANCE = {"used_group_runner": False, "reason": "Chưa nối với 4 cell gốc"}


def prepare_group_runner():
    global GROUP_SAVE_STAGE, GROUP_SYNC_RESULTS, RUNNER_PROVENANCE
    if not ON_KAGGLE:
        print("Kiểm chứng local: không chạy 4 cell kết nối gốc hoặc upload HF.", flush=True)
        return
    if not all(
        name in globals() for name in ["GIT_COMMIT", "RUN_OWNER", "save_stage", "sync_results"]
    ):
        raise RuntimeError("Chạy đủ 4 cell gốc phía trên trước khi chạy phần bổ sung.")
    if CONFIG["hf_repo_id"] != HF_REPO_ID:
        raise RuntimeError("Kho HF trong CONFIG phải khớp HF_REPO_ID của cell gốc.")
    changes = run_command(
        ["git", "-C", str(WORK_DIR), "status", "--porcelain"], "Kiểm tra Git", capture=True
    ).stdout.strip()
    if changes:
        raise RuntimeError("Repo có thay đổi chưa commit. Kiểm tra trước khi đổi phiên bản code.")
    actual_commit = run_command(
        ["git", "-C", str(WORK_DIR), "rev-parse", "HEAD"], "Đọc commit", capture=True
    ).stdout.strip()
    requested = SUBMISSION.get("git_commit")
    if requested and requested != actual_commit:
        run_command(
            ["git", "-C", str(WORK_DIR), "fetch", "origin", requested], "Lấy commit đã chốt"
        )
        run_command(
            ["git", "-C", str(WORK_DIR), "checkout", "--detach", requested], "Chọn commit đã chốt"
        )
        actual_commit = run_command(
            ["git", "-C", str(WORK_DIR), "rev-parse", "HEAD"], "Xác nhận commit", capture=True
        ).stdout.strip()
    notebook_at_commit = WORK_DIR / "notebooks/kaggle_runner.ipynb"
    matches = False
    if notebook_at_commit.is_file() and SUBMISSION.get("notebook_source_sha256"):
        matches = (
            notebook_source_hash(json.loads(notebook_at_commit.read_text(encoding="utf-8")))
            == SUBMISSION["notebook_source_sha256"]
        )
    if CONFIG["profile"] in ["official", "sweep"] and not matches:
        raise RuntimeError(
            "Source notebook chưa khớp commit trên GitHub. Push nhánh đã commit rồi cập nhật notebook trước khi train."
        )
    importlib.invalidate_caches()
    artifacts = importlib.import_module("src.training.artifacts")
    if Path(artifacts.__file__).resolve() != (WORK_DIR / "src/training/artifacts.py").resolve():
        raise RuntimeError("Module src được nạp từ đường dẫn khác; restart session rồi chạy lại.")
    artifacts = importlib.reload(artifacts)

    def bounded_sync(run_dir, repo_id=HF_REPO_ID):
        try:
            run_command(
                [
                    sys.executable,
                    str(WORK_DIR / "scripts/sync_run.py"),
                    "--run-dir",
                    str(Path(run_dir).resolve()),
                    "--repo-id",
                    repo_id,
                ],
                "Đồng bộ Hugging Face",
                timeout=CONFIG["sync_timeout_seconds"],
            )
            return True
        except (subprocess.SubprocessError, OSError) as exc:
            log_event(
                "Đồng bộ Hugging Face",
                "failed",
                error_type=type(exc).__name__,
                output_retained=True,
            )
            return False

    artifacts.sync_results = bounded_sync
    if CONFIG["profile"] in ["official", "sweep"]:  # Lượt thử (smoke) không đẩy gì lên kho Hugging Face của nhóm
        GROUP_SAVE_STAGE, GROUP_SYNC_RESULTS = artifacts.save_stage, artifacts.sync_results
    RUNNER_PROVENANCE = {
        "used_group_runner": True,
        "bootstrap_git_commit": GIT_COMMIT,
        "bootstrap_owner": RUN_OWNER,
        "runner_git_commit": actual_commit,
        "runner_repo": REPO_URL,
        "sync_timeout_seconds": CONFIG["sync_timeout_seconds"],
        "hf_sync_enabled": GROUP_SYNC_RESULTS is not None,
        "notebook_source_matches_commit": matches,
        "artifact_module_sha256": file_hash(WORK_DIR / "src/training/artifacts.py"),
        "sync_script_sha256": file_hash(WORK_DIR / "scripts/sync_run.py"),
    }
    print(
        "Đã nối phần bổ sung với runner nhóm:", actual_commit,
        "| đẩy kết quả lên Hugging Face:", "có" if GROUP_SYNC_RESULTS else "không (chỉ lượt official)",
        flush=True,
    )


# ─── 3.2 Đồng bộ Hugging Face, chỉ ở lượt official ───────────────────────────────────────────────────
def get_hf_api():
    # Chỉ lượt official trên Kaggle mới đẩy kết quả lên kho Hugging Face của nhóm
    if not ON_KAGGLE or CONFIG["profile"] not in ["official", "sweep"]:
        return None
    if not os.environ.get("HF_TOKEN"):
        raise RuntimeError("Pipeline Kaggle cần HF_TOKEN đã nạp ở cell gốc thứ 3.")
    CONFIG["hf_authenticated_owner"] = RUN_OWNER
    return api  # Đã xác thực ở cell gốc thứ 3.


SYNC_STATUS = {}  # tên thư mục → lần đồng bộ gần nhất có thành công không


def sync_run(run_dir, api, step):
    if api is None:
        print(step + ": không đồng bộ (chỉ lượt official trên Kaggle đẩy lên Hugging Face)", flush=True)
        return False
    if GROUP_SYNC_RESULTS is None:
        raise RuntimeError("Chưa nối hàm đồng bộ của nhóm.")
    success = bool(GROUP_SYNC_RESULTS(run_dir, CONFIG["hf_repo_id"]))
    SYNC_STATUS[Path(run_dir).name] = success
    print(
        step + (": Đã đồng bộ" if success else ": Chưa đồng bộ; giữ output để upload lại"),
        flush=True,
    )
    return success


# ─── 3.3 Study: kiểm tra cấu hình và tạo thư mục kết quả ─────────────────────────────────────────────
@dataclass
class Study:
    root: Path
    directory: Path
    api: object
    device: torch.device
    requirements: str
    info: dict
    manifests: dict = field(default_factory=dict)  # Đủ dữ liệu, chỉ dùng thống kê từ manifest
    frames: dict = field(default_factory=dict)  # Các tập dùng trong lượt chạy (smoke rút gọn)
    audit: dict = field(default_factory=dict)
    provenance: dict = field(default_factory=dict)
    cache: dict = field(default_factory=dict)
    tables: dict = field(default_factory=lambda: defaultdict(list))
    figures: list = field(default_factory=list)
    selection: dict = field(default_factory=dict)


def check_override_keys(overrides, where):
    unknown = set(overrides) - set(CONFIG)
    blocked = set(overrides) & SWEEP_FIXED_KEYS
    if unknown:
        raise ValueError(where + ": khóa không có trong CONFIG: " + ", ".join(sorted(unknown)))
    if blocked:
        raise ValueError(where + ": không được ghi đè khóa cố định: " + ", ".join(sorted(blocked)))


def check_architecture_values(values, where):
    hidden = values.get("head_hidden", CONFIG["head_hidden"])
    fraction = values.get("fine_tune_fraction", CONFIG["fine_tune_fraction"])
    train_fraction = values.get("train_fraction", CONFIG["train_fraction"])
    if not isinstance(hidden, int) or hidden < 0:
        raise ValueError(where + ": head_hidden phải là số nguyên ≥ 0.")
    if fraction is not None and not 0 < fraction <= 1:
        raise ValueError(where + ": fine_tune_fraction phải là None hoặc nằm trong (0, 1].")
    if not 0 < train_fraction <= 1:
        raise ValueError(where + ": train_fraction phải nằm trong (0, 1].")
    if CONFIG["profile"] == "official" and train_fraction != 1.0:
        raise ValueError(where + ": lượt official phải dùng toàn bộ tập train (train_fraction = 1).")


def validate_overrides():
    check_architecture_values({}, "CONFIG")
    if not set(CONFIG["model_overrides"]) <= set(MODEL_NAMES):
        raise ValueError("model_overrides chỉ nhận tên mô hình: " + ", ".join(MODEL_NAMES))
    for model_name, overrides in CONFIG["model_overrides"].items():
        check_override_keys(overrides, "model_overrides[" + model_name + "]")
        check_architecture_values(overrides, "model_overrides[" + model_name + "]")


def validate_sweep_grid():
    grid = CONFIG["sweep_grid"]
    names = [point.get("name") for point in grid]
    if not grid or not all(isinstance(name, str) and name for name in names):
        raise ValueError("Profile sweep cần sweep_grid không rỗng, mỗi điểm có name.")
    if len(set(names)) != len(names):
        raise ValueError("Tên điểm trong sweep_grid bị trùng.")
    for point in grid:
        if set(point) - {"name", "overrides", "models"}:
            raise ValueError("Điểm " + point["name"] + " có khóa lạ.")
        check_override_keys(point.get("overrides", {}), "sweep_grid[" + point["name"] + "]")
        check_architecture_values(point.get("overrides", {}), "sweep_grid[" + point["name"] + "]")
        if not set(point.get("models", CONFIG["models"])) <= set(CONFIG["models"]):
            raise ValueError("Điểm " + point["name"] + " dùng mô hình ngoài CONFIG['models'].")


# Các khóa quyết định dữ liệu, tiền xử lý và đầu ra; sweep và model_overrides không được đổi
SWEEP_FIXED_KEYS = {
    "profile", "protocol_id", "models", "training_seeds", "image_size", "resize_size", "pretrained",
    "data_input_root", "far_ood_root", "unseen_cat_root", "unseen_cat_classes", "output_root",
    "official_archives_root", "official_data_dir",
    "hf_repo_id", "sweep_grid", "model_overrides", "smoke_overrides",
}


def validate_settings():
    if CONFIG["profile"] not in ["smoke", "official", "sweep"]:
        raise ValueError("profile không hợp lệ.")
    if not set(CONFIG["models"]) <= set(MODEL_NAMES):
        raise ValueError("Chỉ hỗ trợ các mô hình: " + ", ".join(MODEL_NAMES))
    if not set(CONFIG["training_seeds"]) <= {42, 43, 44}:
        raise ValueError("QĐ4 chỉ dùng seed 42, 43, 44.")
    validate_overrides()
    if CONFIG["profile"] == "smoke":
        return
    if CONFIG["profile"] == "official" and (
        sorted(CONFIG["models"]) != sorted(MODEL_NAMES) or sorted(CONFIG["training_seeds"]) != [42, 43, 44]
    ):
        raise RuntimeError("Official chạy đủ 4 cấu hình × 3 seed (§2.5.4).")
    if CONFIG["profile"] == "sweep":
        validate_sweep_grid()
    if (
        not torch.cuda.is_available()
        or (CONFIG["image_size"], CONFIG["resize_size"]) != (224, 256)
        or not CONFIG["pretrained"]
    ):
        raise RuntimeError("Official cần GPU, tiền xử lý 256 → 224 và trọng số ImageNet.")
    if (
        not SUBMISSION.get("git_commit")
        or not SUBMISSION.get("notebook_source_sha256")
        or SUBMISSION.get("git_dirty", True)
    ):
        raise RuntimeError("Official cần commit sạch và SHA source của notebook đã commit.")
    if not RUNNER_PROVENANCE.get("notebook_source_matches_commit"):
        raise RuntimeError("Source notebook phải khớp commit đã push trước khi chạy chính thức.")


def start_study():
    prepare_group_runner()
    validate_settings()
    if CONFIG["profile"] == "smoke":
        CONFIG.update(CONFIG["smoke_overrides"])
    root = Path(CONFIG["output_root"])
    directory = root / f"{run_stamp()}_study_{uuid4().hex[:8]}"
    (directory / "report_tables").mkdir(parents=True)
    requirements = run_command(
        [sys.executable, "-m", "pip", "freeze"], "Ghi phiên bản thư viện", capture=True
    ).stdout
    info = {
        "run_id": directory.name,
        "kind": "study",
        "profile": CONFIG["profile"],
        "is_smoke_test": CONFIG["profile"] == "smoke",
        "protocol_id": CONFIG["protocol_id"],
        "git_commit": SUBMISSION.get("git_commit"),
        "notebook_source_sha256": SUBMISSION.get("notebook_source_sha256"),
        "runner_provenance": dict(RUNNER_PROVENANCE),
        "started_at_utc": utc_now(),
        "status": "phase_a",
    }
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    study = Study(root, directory, get_hf_api(), device, requirements, info)
    save_info(directory, info)
    write_json(directory / "config.yaml", CONFIG)  # JSON hợp lệ cũng là YAML
    (directory / "requirements_freeze.txt").write_text(requirements, encoding="utf-8")
    print("Study:", directory, "| profile:", CONFIG["profile"], "| thiết bị:", device, flush=True)
    return study


# ─── Chạy ────────────────────────────────────────────────────────────────────────────────────────────
if AUTO_RUN:
    STUDY = start_study()


## 4. Dữ liệu: kiểm tra, chia tập và các tập OOD

- **Làm:** kiểm tra SHA-256 ảnh Oxford và DTD; chia mèo thành 950 train, 238 validation, 1.183 test; lấy 500 ảnh chó thăm dò; tách 2.486 ảnh chó near-OOD.
- **Xuất:** thanh tiến độ kiểm tra ảnh, mã nhận dạng của cách chia và của tập thăm dò.


In [ ]:
# ─── 4.1 Tìm dataset Oxford đã chuẩn bị và kiểm tra checksum ─────────────────────────────────────────
OFFICIAL_ARCHIVES = {  # URL và MD5 trùng với torchvision 0.26 (oxford_iiit_pet.py, dtd.py)
    "oxford_images": (
        "https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz",
        "5c4f3ee8e5d25df40f4fd59a7f44e54c",
    ),
    "oxford_annotations": (
        "https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz",
        "95a8c909bbe2e81eed6a22bccdf3f68f",
    ),
    "dtd": (
        "https://www.robots.ox.ac.uk/~vgg/data/dtd/download/dtd-r1.0.1.tar.gz",
        "fff73e5086ae6bdbea199a49dfb8a4c1",
    ),
}


def archive_digests(path):
    md5, sha256 = hashlib.md5(), hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            md5.update(chunk)
            sha256.update(chunk)
    return md5.hexdigest(), sha256.hexdigest()


def official_archive(name, config):
    # Dùng archive gốc đã gắn sẵn (nếu có) hoặc tải thẳng từ Oxford VGG; luôn kiểm MD5 lúc chạy
    url, expected_md5 = OFFICIAL_ARCHIVES[name]
    filename = url.rsplit("/", 1)[1]
    root = Path(config["official_archives_root"]).expanduser()
    found = sorted(root.rglob(filename)) if root.is_dir() else []
    if found:
        path = found[0]
    else:
        path = Path(config["official_data_dir"]) / "archives" / filename
        if not path.is_file():
            path.parent.mkdir(parents=True, exist_ok=True)
            partial = path.with_name(filename + ".part")
            with timed_step("Tải " + filename):
                with urlopen(url, timeout=config["network_read_timeout_seconds"]) as response:
                    with open(partial, "wb") as output:
                        shutil.copyfileobj(response, output, 1 << 20)
            partial.replace(path)
    md5, sha256 = archive_digests(path)
    if md5 != expected_md5:
        raise ValueError(f"{filename}: MD5 {md5} khác MD5 chính thức {expected_md5}.")
    return path, {
        "filename": filename,
        "url": url,
        "bytes": path.stat().st_size,
        "md5": md5,
        "sha256": sha256,
        "obtained_from": "attached_input" if found else "downloaded",
    }


def extract_once(archive, destination):
    marker = destination / (".extracted_" + archive.name)
    if marker.is_file():
        return
    destination.mkdir(parents=True, exist_ok=True)
    with timed_step("Giải nén " + archive.name):
        with tarfile.open(archive) as tar:
            tar.extractall(destination, filter="data")
    marker.write_text(utc_now(), encoding="utf-8")


def locate_dataset(config):
    # Dựng dữ liệu Oxford-IIIT Pet từ hai archive chính thức; manifest do notebook tự tạo, không dùng bản sao
    folder = Path(config["official_data_dir"]) / "oxford-iiit-pet"
    sources = {}
    for key, name in [("images", "oxford_images"), ("annotations", "oxford_annotations")]:
        archive, sources[key] = official_archive(name, config)
        extract_once(archive, folder)
    rows = read_annotation_list(folder / "annotations/list.txt")
    hashes = parallel_map(
        lambda row: file_hash(folder / "images" / (row["image_id"] + ".jpg")), rows, "SHA-256 ảnh Oxford"
    )
    manifest = pd.DataFrame(
        {
            "image_id": [row["image_id"] for row in rows],
            "sha256": hashes,
            "species_id": [row["species"] for row in rows],
        }
    )
    (folder / "manifests").mkdir(exist_ok=True)
    manifest.to_csv(folder / "manifests/all_labeled_images.csv", index=False)
    counts = (int((manifest.species_id == 1).sum()), int((manifest.species_id == 2).sum()))
    if counts != (2371, 4978):
        raise ValueError(f"list.txt có {counts[0]} ảnh mèo, {counts[1]} ảnh chó; cần 2.371 và 4.978.")
    checksums = {
        name: file_hash(folder / name)
        for name in [
            "annotations/list.txt",
            "annotations/trainval.txt",
            "annotations/test.txt",
            "manifests/all_labeled_images.csv",
        ]
    }
    write_json(folder / "manifests/file_checksums.json", checksums)
    return folder, {
        "source": "Oxford VGG, archive chính thức (robots.ox.ac.uk), MD5 kiểm lúc chạy",
        "source_archives": sources,
        "listed_images": len(manifest),
        "jpg_files_in_archive": sum(1 for p in (folder / "images").iterdir() if p.suffix.lower() == ".jpg"),
        "checksum_index_sha256": file_hash(folder / "manifests/file_checksums.json"),
    }


# ─── 4.2 Đọc nhãn loài, kiểm tra ảnh, trimap và hộp bao vùng đầu ─────────────────────────────────────
def read_annotation_list(path):
    result = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        image_id, original_class, species, breed_id = line.split()
        result.append(
            {
                "image_id": image_id,
                "original_class": int(original_class),
                "species": int(species),
                "breed_id": int(breed_id),
                "breed": image_id.rsplit("_", 1)[0],
            }
        )
    return result


def inspect_sample(row, images_dir, annotations_dir):
    row = dict(row)
    image_path = images_dir / (row["image_id"] + ".jpg")
    with Image.open(image_path) as image:
        width, height = image.size  # Chỉ đọc header: ảnh test chỉ được giải mã ở pha B
    row.update(
        image_path=str(image_path),
        width=width,
        height=height,
        image_sha256=file_hash(image_path),
        label=CLASS_TO_IDX.get(row["breed"], -1),
        trimap_path="",
        has_valid_trimap=False,
        bbox_json="",
        has_bbox=False,
    )
    if row["species"] == 2:
        return row  # Chó chỉ làm OOD; không đọc trimap và XML
    mask_path = annotations_dir / "trimaps" / (row["image_id"] + ".png")
    if mask_path.exists():
        with Image.open(mask_path) as source:
            mask = np.array(source)
        row["trimap_path"] = str(mask_path)
        row["has_valid_trimap"] = bool(
            mask.shape == (height, width) and np.isin(mask, [1, 2, 3]).all() and (mask == 1).any()
        )
    xml_path = annotations_dir / "xmls" / (row["image_id"] + ".xml")
    if xml_path.exists():
        node = ET.parse(xml_path).find(".//bndbox")
        if node is not None:
            # PASCAL VOC dùng tọa độ 1-based inclusive; chuyển sang PIL 0-based exclusive.
            xmin, ymin, xmax, ymax = [
                int(node.findtext(key)) for key in ["xmin", "ymin", "xmax", "ymax"]
            ]
            box = [max(0, xmin - 1), max(0, ymin - 1), min(width, xmax), min(height, ymax)]
            if box[0] < box[2] and box[1] < box[3]:
                row["bbox_json"], row["has_bbox"] = json.dumps(box), True
    return row


# ─── 4.3 Chia train, validation, test một lần theo QĐ1 ───────────────────────────────────────────────
def fixed_split(cats, destination):
    path = Path(destination) / "fixed_split.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        mapping = cats[["image_id"]].copy()
        mapping["split"] = "test"
        pool = cats.index[cats.original_split == "trainval"]
        train, val = train_test_split(
            pool, test_size=0.2, stratify=cats.loc[pool, "label"], random_state=SPLIT_SEED
        )
        mapping.loc[train, "split"], mapping.loc[val, "split"] = "train", "val"
        temporary = path.with_suffix(".csv.tmp")
        mapping.sort_values("image_id").to_csv(temporary, index=False)
        temporary.replace(path)
    mapping = pd.read_csv(path).set_index("image_id")
    if not mapping.index.is_unique or set(mapping.index) != set(cats.image_id):
        raise ValueError("Manifest chia tập không phủ đúng 2.371 ảnh mèo.")
    result = cats.copy()
    result["split"] = result.image_id.map(mapping["split"])
    if result.split.value_counts().to_dict() != {"test": 1183, "train": 950, "val": 238}:
        raise ValueError("Số ảnh không khớp QĐ1: 950 train / 238 val / 1.183 test.")
    if set(result.loc[result.split == "test", "image_id"]) != set(
        cats.loc[cats.original_split == "test", "image_id"]
    ):
        raise ValueError("Không được thay đổi tập test chính thức.")
    return result


# ─── 4.4 Tập thăm dò, near-OOD và manifest ───────────────────────────────────────────────────────────
def sample_probe(dogs):
    # §2.3.1: lấy ngẫu nhiên phân tầng một lần, 20 ảnh mỗi giống chó trainval.
    # RandomState giữ nguyên dãy số giữa các phiên bản NumPy.
    rng = np.random.RandomState(PROBE_SEED)
    pool = dogs[dogs.original_split == "trainval"]
    chosen = []
    for breed in sorted(pool.breed.unique()):
        names = sorted(pool.loc[pool.breed == breed, "image_id"])
        chosen += sorted(rng.choice(names, PROBE_DOGS_PER_BREED, replace=False).tolist())
    return chosen


def make_manifests(folder, destination):
    images_dir, annotations_dir = folder / "images", folder / "annotations"
    rows = read_annotation_list(annotations_dir / "list.txt")
    prepared = pd.read_csv(folder / "manifests/all_labeled_images.csv").set_index("image_id")
    if not prepared.index.is_unique or set(prepared.index) != {row["image_id"] for row in rows}:
        raise ValueError("Manifest và list.txt không có cùng danh sách ảnh.")
    original_train = {r["image_id"] for r in read_annotation_list(annotations_dir / "trainval.txt")}
    original_test = {r["image_id"] for r in read_annotation_list(annotations_dir / "test.txt")}
    if original_train & original_test:
        raise ValueError("Official trainval/test overlap.")
    official = original_train | original_test
    for row in rows:
        if row["image_id"] not in official:
            raise ValueError("Ảnh list không có official split: " + row["image_id"])
        if row["species"] == 1 and row["breed"] not in CLASS_TO_IDX:
            raise ValueError("Giống mèo ngoài thiết kế.")
        row["original_split"] = "trainval" if row["image_id"] in original_train else "test"
    samples = parallel_map(
        lambda row: inspect_sample(row, images_dir, annotations_dir), rows, "Kiểm tra ảnh Oxford", leave=True
    )
    images = pd.DataFrame(samples)
    expected = prepared.loc[images.image_id]
    if not (
        np.array_equal(images.image_sha256.to_numpy(), expected.sha256.to_numpy())
        and np.array_equal(images.species.to_numpy(), expected.species_id.to_numpy())
    ):
        raise ValueError("Ảnh hoặc nhãn loài khác manifest đã kiểm tra.")
    cats = images[images.species == 1].reset_index(drop=True)
    dogs = images[images.species == 2].sort_values("image_id").reset_index(drop=True)
    if len(cats) != 2371 or cats.breed.value_counts().to_dict() != EXPECTED_CAT_COUNTS:
        raise ValueError("Không khớp 2.371 ảnh/12 giống mèo.")
    if len(dogs) != 4978:
        raise ValueError("Không khớp 4.978 ảnh chó chính thức.")
    if cats.image_id.duplicated().any() or cats.image_sha256.duplicated().any():
        raise ValueError("Phát hiện ảnh mèo lặp; cần quy tắc nhóm trước khi chia.")
    if set(cats.image_sha256) & set(dogs.image_sha256):
        raise ValueError("ID và near-OOD trùng nội dung.")
    cats = fixed_split(cats, destination).sort_values("image_id").reset_index(drop=True)
    split_id = json_hash(
        {
            "policy": SPLIT_POLICY,
            "seed": SPLIT_SEED,
            "assignments": cats[["image_id", "label", "split", "image_sha256"]].to_dict("records"),
        }
    )
    if split_id != EXPECTED_SPLIT_ID:
        raise ValueError("Danh sách chia tập khác manifest QĐ1 đã cố định.")
    probe_ids = sample_probe(dogs)
    probe_id = json_hash(probe_ids)
    if probe_id != EXPECTED_PROBE_ID:
        raise ValueError("Tập thăm dò khác danh sách đã cố định.")
    dogs["split"] = "unused"
    dogs.loc[dogs.original_split == "test", "split"] = "near_ood"  # §2.3.1: chó của split test
    dogs.loc[dogs.image_id.isin(probe_ids), "split"] = "probe_ood"
    near, probe = dogs[dogs.split == "near_ood"], dogs[dogs.split == "probe_ood"]
    if len(near) != 2486 or near.breed.nunique() != 25:
        raise ValueError("Near-OOD phải gồm 2.486 ảnh chó split test thuộc 25 giống.")
    if probe.groupby("breed").size().to_dict() != dict.fromkeys(sorted(near.breed.unique()), PROBE_DOGS_PER_BREED):
        raise ValueError("Tập thăm dò phải có đúng 20 ảnh chó trainval cho mỗi giống.")
    destination = Path(destination)
    cats.to_csv(destination / "cats_manifest.csv", index=False)
    dogs.to_csv(destination / "dogs_manifest.csv", index=False)
    probe.to_csv(destination / "probe_ood_manifest.csv", index=False)
    write_json(destination / "class_mapping.json", CLASS_TO_IDX)
    audit = {
        "split_id": split_id,
        "probe_id": probe_id,
        "split_policy": SPLIT_POLICY,
        "split_seed": SPLIT_SEED,
        "probe_seed": PROBE_SEED,
        "cat_count": len(cats),
        "dog_count": len(dogs),
        "near_ood_count": len(near),
        "near_ood_breeds": int(near.breed.nunique()),
        "probe_ood_count": len(probe),
        "fixed_split_sha256": file_hash(destination / "fixed_split.csv"),
        "split_counts": cats.split.value_counts().to_dict(),
        "empty_or_invalid_trimaps": cats.loc[~cats.has_valid_trimap, "image_id"].tolist(),
        "cats_with_bbox": int(cats.has_bbox.sum()),
        "bbox_by_split": cats.groupby("split").has_bbox.sum().astype(int).to_dict(),
        "trimap_by_split": cats.groupby("split").has_valid_trimap.sum().astype(int).to_dict(),
    }
    write_json(destination / "data_audit.json", audit)
    return cats, dogs, audit


def small_stratified(frame, count):
    return (
        frame.groupby("label", group_keys=False)
        .head(count)
        .sort_values("image_id")
        .reset_index(drop=True)
    )


# ─── 4.5 Far-OOD: DTD ────────────────────────────────────────────────────────────────────────────────
def make_far_manifest(config):
    # QĐ7: toàn bộ DTD r1.0.1, dựng từ archive chính thức, MD5 kiểm lúc chạy
    folder = Path(config["official_data_dir"]) / "dtd"
    archive, source = official_archive("dtd", config)
    extract_once(archive, folder)
    base = folder / "dtd" / "images"
    files = sorted(p for p in base.glob("*/*") if p.suffix.lower() in IMAGE_SUFFIXES)
    hashes = parallel_map(file_hash, files, "Kiểm tra SHA-256 ảnh DTD", leave=True)
    frame = pd.DataFrame(
        {
            "image_id": ["dtd/" + p.relative_to(base).as_posix() for p in files],
            "category": [p.parent.name for p in files],
            "image_path": [str(p) for p in files],
            "image_sha256": hashes,
        }
    )
    counts = frame.category.value_counts()
    if len(frame) != 5640 or len(counts) != 47 or not counts.eq(120).all() or frame.image_id.duplicated().any():
        raise ValueError("DTD phải có đủ 5.640 ảnh thuộc 47 nhóm, mỗi nhóm 120 ảnh.")
    frame["source"] = source["url"]
    frame["split"], frame["label"] = "far_ood", -1
    frame = frame.sort_values("image_id").reset_index(drop=True)
    manifest_path = folder / "far_ood_manifest.csv"
    frame.to_csv(manifest_path, index=False)
    frame.attrs["provenance"] = {
        "dataset": "DTD r1.0.1",
        "source": source["url"],
        "source_archive_md5": source["md5"],
        "source_archive_sha256": source["sha256"],
        "manifest_sha256": file_hash(manifest_path),
    }
    return frame


# ─── 4.6 Mèo ngoài 12 giống (tùy chọn, bỏ qua nếu không gắn) ─────────────────────────────────────────
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def make_unseen_cat_manifest(config, known_hashes):
    root = Path(config["unseen_cat_root"]).expanduser()
    classes = set(config["unseen_cat_classes"])
    files = sorted(
        path
        for path in (root.rglob("*") if root.is_dir() else [])
        if path.suffix.lower() in IMAGE_SUFFIXES and path.parent.name in classes
    )
    rows, seen, duplicates, rotated = [], set(known_hashes), 0, 0
    digests = parallel_map(file_hash, files, "Kiểm tra ảnh unseen_cat")
    for path, digest in zip(files, digests):
        if digest in seen:
            duplicates += 1  # Trùng Oxford, DTD hoặc ảnh khác trong tập
            continue
        seen.add(digest)
        with Image.open(path) as source:
            rotated += source.getexif().get(0x0112, 1) != 1  # Đọc như Oxford, không xoay theo EXIF
        rows.append(
            {
                "image_id": "unseen_cat/" + path.relative_to(root).as_posix(),
                "image_path": str(path),
                "category": path.parent.name,
                "image_sha256": digest,
                "split": "unseen_cat",
                "label": -1,
            }
        )
    if not rows:
        print("Không có ảnh trong thư mục", ", ".join(sorted(classes)), "→ bỏ qua unseen_cat, không sinh số.")
        log_event("Tập unseen_cat", "skipped", excluded_duplicates=duplicates)
        return None
    frame = pd.DataFrame(rows).sort_values("image_id").reset_index(drop=True)
    frame.attrs["provenance"] = {
        "root": str(root),
        "counts": frame.category.value_counts().to_dict(),
        "excluded_duplicates": duplicates,
        "exif_orientation_not_applied": rotated,
    }
    return frame


# ─── 4.7 Gom các tập dùng trong lượt chạy ────────────────────────────────────────────────────────────
def prepare_data(study):
    folder = study.root / "data_manifest"
    folder.mkdir(parents=True, exist_ok=True)
    dataset, provenance = run_step("Tìm dataset Oxford", locate_dataset, CONFIG)
    cats, dogs, audit = run_step("Kiểm tra Oxford và cố định split", make_manifests, dataset, folder)
    oxford_hashes = set(cats.image_sha256) | set(dogs.image_sha256)
    far = run_step("Kiểm tra DTD", make_far_manifest, CONFIG)
    if set(far.image_sha256) & oxford_hashes:
        raise ValueError("DTD trùng nội dung ảnh Oxford.")
    provenance["far_ood"] = far.attrs["provenance"]
    unseen = run_step(
        "Tìm tập unseen_cat", make_unseen_cat_manifest, CONFIG, oxford_hashes | set(far.image_sha256)
    )
    study.manifests = {"cats": cats, "dogs": dogs, "far_ood": far, "unseen_cat": unseen}
    frames = {split: cats[cats.split == split] for split in ["train", "val", "test"]}
    frames.update({split: dogs[dogs.split == split] for split in ["probe_ood", "near_ood"]})
    frames["far_ood"] = far
    if unseen is not None:
        frames["unseen_cat"], provenance["unseen_cat"] = unseen, unseen.attrs["provenance"]
    for split in ["far_ood", "unseen_cat"]:
        if split in frames:
            frames[split].to_csv(folder / (split + "_manifest.csv"), index=False)
    if CONFIG["profile"] == "smoke":
        for split in ["train", "val", "test"]:
            frames[split] = small_stratified(frames[split], CONFIG["smoke_images_per_class"])
        for split in ["probe_ood", *OOD_SPLITS]:
            if split in frames:
                frames[split] = frames[split].head(CONFIG["smoke_ood_images"])
    study.frames, study.audit, study.provenance = frames, audit, provenance
    study.info.update(split_id=audit["split_id"], probe_id=audit["probe_id"], data_version=provenance)
    save_info(study.directory, study.info)
    print(
        "Dữ liệu đã kiểm tra | split", audit["split_id"][:12], "| tập thăm dò", audit["probe_id"][:12],
        "| test và các tập OOD chỉ được giải mã ở pha B.",
        flush=True,
    )


# ─── Chạy ────────────────────────────────────────────────────────────────────────────────────────────
if AUTO_RUN:
    prepare_data(STUDY)


## 5. Khảo sát: số ảnh các tập và phân bố 12 giống

- **Xuất:** bảng các tập (nguồn, số ảnh, mục đích); bảng số ảnh mỗi giống theo train, validation, test.
- **Hình:** số ảnh các tập ID và OOD; số ảnh mỗi giống trong ba tập ID.


In [ ]:
# ─── 5.1 Các tập dữ liệu gồm bao nhiêu ảnh ───────────────────────────────────────────────────────────
def eda_overview(study):
    cats, dogs, far, unseen = (study.manifests[k] for k in ["cats", "dogs", "far_ood", "unseen_cat"])
    sets = [
        ("ID train", "Oxford-IIIT Pet trainval, mèo", (cats.split == "train").sum(), "Huấn luyện", "train"),
        ("ID validation", "Oxford-IIIT Pet trainval, mèo", (cats.split == "val").sum(), "Chọn checkpoint, khóa ngưỡng", "val"),
        ("ID test", "Oxford-IIIT Pet test, mèo", (cats.split == "test").sum(), "Đánh giá cuối (pha B)", "test"),
        ("Tập thăm dò", "Oxford-IIIT Pet trainval, chó", (dogs.split == "probe_ood").sum(), "Chọn phương pháp từ chối (pha A)", "probe_ood"),
        ("Near-OOD", "Oxford-IIIT Pet test, chó", (dogs.split == "near_ood").sum(), "Đánh giá cuối (pha B)", "near_ood"),
        ("Far-OOD", "DTD r1.0.1, toàn bộ", len(far), "Đánh giá cuối (pha B)", "far_ood"),
    ]
    if unseen is not None:
        sets.append(("Mèo ngoài 12 giống", "FeliBreed", len(unseen), "Mô tả, không đặt giả thuyết", "unseen_cat"))
    table = pd.DataFrame(
        [
            {"Tập": name, "Nguồn": source, "Số ảnh": int(count), "Mục đích": purpose, "Dùng trong lượt chạy": len(study.frames[key])}
            for name, source, count, purpose, key in sets
        ]
    )
    display(table)
    fig, axis = plt.subplots(figsize=(8.5, 0.5 * len(table) + 1.3))
    colors = [BLUE if key in ["train", "val", "test"] else ORANGE for *_, key in sets]
    bars = axis.barh(table["Tập"], table["Số ảnh"], height=0.62, color=colors)
    axis.bar_label(bars, labels=[thousands(count) for count in table["Số ảnh"]], padding=4)
    axis.invert_yaxis()
    axis.grid(axis="y", visible=False)
    axis.margins(x=0.1)
    axis.xaxis.set_major_formatter(lambda value, _: thousands(value))
    axis.set(xlabel="Số ảnh", title="Số ảnh của các tập ID và OOD")
    legend_above(axis, handles=[Patch(color=BLUE, label="ID: mèo thuộc 12 giống"), Patch(color=ORANGE, label="OOD: ảnh ngoài 12 giống")])
    show_figure(study, fig, "03_1_dataset_overview", "Chương 3", "Số ảnh của các tập ID và OOD")


# ─── 5.2 Ảnh chia thế nào giữa 12 giống và ba tập ID ─────────────────────────────────────────────────
def eda_breeds(study):
    cats = study.manifests["cats"]
    table = pd.crosstab(cats.breed, cats.split).reindex(index=CAT_BREEDS, columns=["train", "val", "test"])
    table["Tổng"] = table.sum(axis=1)
    table.loc["Tổng"] = table.sum()
    table.columns = ["Train", "Validation", "ID test", "Tổng"]
    display(table)
    counts = table.drop(index="Tổng")
    fig, axis = plt.subplots(figsize=(9, 6.2))
    positions = np.arange(len(counts))
    for offset, (column, color) in enumerate(zip(["Train", "Validation", "ID test"], [BLUE, ORANGE, AQUA])):
        axis.barh(positions + (offset - 1) * 0.27, counts[column], height=0.24, color=color, label=column)
    axis.set_yticks(positions, [name.replace("_", " ") for name in counts.index])
    axis.invert_yaxis()
    axis.grid(axis="y", visible=False)
    axis.set(xlabel="Số ảnh", title="Số ảnh mỗi giống trong ba tập ID")
    legend_above(axis)
    show_figure(study, fig, "03_2_breed_distribution", "Chương 3", "Phân bố ảnh theo 12 giống trong train, validation, test")


# ─── Chạy ────────────────────────────────────────────────────────────────────────────────────────────
if AUTO_RUN:
    eda_overview(STUDY)
    eda_breeds(STUDY)


## 6. Khảo sát: kích thước ảnh và diện tích con vật (chỉ trainval)

- **Xuất:** thống kê chiều rộng, chiều cao, tỷ lệ khung; tỷ lệ diện tích con vật theo giống; danh sách trimap không dùng được.
- **Hình:** histogram kích thước ảnh; hộp phân bố diện tích con vật theo giống.


In [ ]:
# ─── 6.1 Kích thước và tỷ lệ khung ảnh ───────────────────────────────────────────────────────────────
def trainval_cats(study):
    # Báo cáo cam kết chỉ khảo sát ảnh trainval trước bước khóa
    cats = study.manifests["cats"]
    return cats[cats.original_split == "trainval"]


def eda_image_sizes(study):
    frame = trainval_cats(study).assign(aspect=lambda f: f.width / f.height)
    short_side = frame[["width", "height"]].min(axis=1)
    display(frame[["width", "height", "aspect"]].describe().T.round(3))
    display(
        pd.Series(
            {
                "Số ảnh trainval": len(frame),
                "Cạnh ngắn < 256 (ảnh)": int((short_side < CONFIG["resize_size"]).sum()),
                "Cạnh ngắn < 256 (%)": round(100 * float((short_side < CONFIG["resize_size"]).mean()), 1),
                "Ảnh nằm ngang (%)": round(100 * float((frame.width > frame.height).mean()), 1),
                "Ảnh dọc (%)": round(100 * float((frame.width < frame.height).mean()), 1),
                "Ảnh vuông (%)": round(100 * float((frame.width == frame.height).mean()), 1),
            }
        ).to_frame("Giá trị")
    )
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), layout="constrained")
    for axis, column, label, reference in zip(
        axes, ["width", "height", "aspect"], ["Chiều rộng (pixel)", "Chiều cao (pixel)", "Tỷ lệ khung rộng/cao"], [256, 256, 1]
    ):
        axis.hist(frame[column], bins=30, color=BLUE, edgecolor="white", linewidth=0.6)
        axis.axvline(reference, color=INK, linewidth=1.2, linestyle="--")
        line_label(axis, reference, f"mốc {reference}")
        axis.set(xlabel=label, ylabel="Số ảnh")
    figure_title(fig, f"Kích thước {thousands(len(frame))} ảnh mèo trainval")
    show_figure(study, fig, "03_3_image_sizes", "Phụ lục C", "Phân bố chiều rộng, chiều cao, tỷ lệ khung ảnh mèo trainval")


# ─── 6.2 Con vật chiếm bao nhiêu diện tích ảnh theo trimap ───────────────────────────────────────────
def eda_trimap_area(study):
    frame = trainval_cats(study)
    frame = frame[frame.has_valid_trimap].copy()
    fractions = []
    for path in frame.trimap_path:
        with Image.open(path) as source:
            fractions.append(float((np.array(source) == 1).mean()))  # Nhãn 1 là con vật
    frame["foreground"] = fractions
    grouped = frame.groupby("breed").foreground
    table = pd.DataFrame(
        {
            "Số ảnh": grouped.size(),
            "Trung vị": grouped.median(),
            "Trung bình": grouped.mean(),
            "Nhỏ nhất": grouped.min(),
            "Lớn nhất": grouped.max(),
            "Dưới 10%": frame.assign(small=frame.foreground < 0.1).groupby("breed").small.sum(),
        }
    ).reindex(CAT_BREEDS)
    table.loc["Toàn bộ"] = [len(frame), frame.foreground.median(), frame.foreground.mean(), frame.foreground.min(), frame.foreground.max(), int((frame.foreground < 0.1).sum())]
    display(table.round(3))
    cats = study.manifests["cats"]
    display(cats.loc[~cats.has_valid_trimap, ["image_id", "breed", "split"]].rename(columns={"image_id": "Trimap không dùng được"}))
    order = table.drop(index="Toàn bộ").sort_values("Trung vị").index
    fig, axis = plt.subplots(figsize=(10, 4.8))
    axis.boxplot(
        [frame.loc[frame.breed == breed, "foreground"] for breed in order],
        patch_artist=True,
        widths=0.6,
        boxprops={"facecolor": "#b7d3f6", "edgecolor": BLUE},
        medianprops={"color": INK, "linewidth": 1.6},
        whiskerprops={"color": BLUE},
        capprops={"color": BLUE},
        flierprops={"marker": "o", "markersize": 3, "markerfacecolor": NEUTRAL, "markeredgecolor": "none"},
    )
    axis.set_xticks(range(1, len(order) + 1), [name.replace("_", " ") for name in order], rotation=35, ha="right")
    axis.axhline(0.1, color=INK, linewidth=1.2, linestyle="--")
    line_label(axis, 0.1, "10% diện tích ảnh", vertical=False)
    axis.grid(axis="x", visible=False)
    axis.set(ylabel="Tỷ lệ diện tích con vật", title="Con vật chiếm bao nhiêu diện tích ảnh (trimap, mèo trainval)")
    show_figure(study, fig, "03_4_trimap_area", "Phụ lục C", "Tỷ lệ diện tích con vật theo trimap của từng giống, trainval")


# ─── Chạy ────────────────────────────────────────────────────────────────────────────────────────────
if AUTO_RUN:
    eda_image_sizes(STUDY)
    eda_trimap_area(STUDY)
    eda_crop_retention(STUDY)


## 7. Khảo sát: trimap, hộp bao vùng đầu và các tập chó, DTD

- **Xuất:** bảng số ảnh chó mỗi giống (tập thăm dò, near-OOD); bảng tóm tắt DTD.
- **Hình:** ảnh, trimap, hộp bao của một ảnh train; mỗi giống một ảnh kèm hộp bao; số ảnh chó mỗi giống.


In [ ]:
def eda_crop_retention(study):
    # Tỷ lệ pixel con vật (nhãn 1 của trimap) còn lại sau Resize 256 → CenterCrop 224; chỉ ảnh trainval
    frame = trainval_cats(study)
    frame = frame[frame.has_valid_trimap]

    def retained(path):
        with Image.open(path) as source:
            resized = TF.resize(source, CONFIG["resize_size"], interpolation=InterpolationMode.NEAREST)
        full = np.array(resized)
        crop = np.array(TF.center_crop(resized, [CONFIG["image_size"], CONFIG["image_size"]]))
        return float((crop == 1).sum() / max(1, (full == 1).sum()))

    values = np.array(parallel_map(retained, list(frame.trimap_path), "Tỷ lệ tiền cảnh sau cắt giữa"))
    result = frame[["image_id", "breed"]].assign(foreground_retained=values)
    result.to_csv(study.directory / "crop_retention_trainval.csv", index=False)
    display(
        pd.Series(
            {
                "Số ảnh trainval có trimap": len(values),
                "Trung bình": round(float(values.mean()), 4),
                "Trung vị": round(float(np.median(values)), 4),
                "Phân vị 5%": round(float(np.quantile(values, 0.05)), 4),
                "Ảnh mất > 10% tiền cảnh (%)": round(100 * float((values < 0.9).mean()), 1),
                "Ảnh mất > 25% tiền cảnh (%)": round(100 * float((values < 0.75).mean()), 1),
            }
        ).to_frame("Giá trị")
    )


# ─── 7.1 Trimap và hộp bao vùng đầu ──────────────────────────────────────────────────────────────────
def trimap_display(trimap):
    # Con vật đen, biên xám, nền trắng để đọc được khi in trắng đen
    return np.select([trimap == 1, trimap == 3], [0.0, 0.5], 1.0)


def eda_annotations(study):
    train = study.manifests["cats"]
    train = train[(train.split == "train") & train.has_valid_trimap & train.has_bbox].sort_values("image_id")
    row = train.iloc[0]
    image = read_rgb(row.image_path)
    with Image.open(row.trimap_path) as source:
        trimap = np.array(source)
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), layout="constrained")
    axes[0].imshow(image)
    axes[0].set_title("Ảnh gốc")
    axes[1].imshow(trimap_display(trimap), cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("Trimap: đen con vật, xám biên, trắng nền")
    axes[2].imshow(image)
    draw_box(axes[2], json.loads(row.bbox_json))
    axes[2].set_title("Hộp bao vùng đầu")
    for axis in axes:
        image_axis(axis)
    figure_title(fig, row.image_id.replace("_", " "))
    show_figure(study, fig, "03_5_annotation_example", "Chương 3", "Ví dụ ảnh, trimap và hộp bao vùng đầu (train)")
    fig, axes = plt.subplots(3, 4, figsize=(13, 8.8), layout="constrained")
    for axis, breed in zip(axes.flat, CAT_BREEDS):
        sample = train[train.breed == breed].iloc[0]
        axis.imshow(read_rgb(sample.image_path))
        draw_box(axis, json.loads(sample.bbox_json))
        axis.set_title(breed.replace("_", " "))
        image_axis(axis)
    figure_title(fig, "Mỗi giống một ảnh train kèm hộp bao vùng đầu")
    show_figure(study, fig, "03_5_breed_grid", "Chương 3", "Mỗi giống một ảnh train kèm hộp bao vùng đầu")


# ─── 7.2 Tập thăm dò, near-OOD và DTD ────────────────────────────────────────────────────────────────
def eda_ood_sets(study):
    dogs, far, unseen = (study.manifests[k] for k in ["dogs", "far_ood", "unseen_cat"])
    table = pd.crosstab(dogs.breed, dogs.split)[["probe_ood", "near_ood"]]
    table.columns = ["Tập thăm dò (trainval)", "Near-OOD (test)"]
    table.loc["Tổng"] = table.sum()
    display(table)
    per_category = far.category.value_counts()
    display(
        pd.DataFrame(
            {
                "Số nhóm kết cấu": [far.category.nunique()],
                "Ảnh mỗi nhóm (nhỏ nhất)": [per_category.min()],
                "Ảnh mỗi nhóm (lớn nhất)": [per_category.max()],
                "Tổng ảnh DTD": [len(far)],
            }
        )
    )
    if unseen is not None:
        display(unseen.category.value_counts().rename("Số ảnh unseen_cat").to_frame())
    counts = table.drop(index="Tổng")
    fig, axis = plt.subplots(figsize=(8.5, 8))
    positions = np.arange(len(counts))
    axis.barh(positions, counts["Near-OOD (test)"], height=0.62, color=ORANGE, label="Near-OOD (chó test)")
    axis.plot(
        counts["Tập thăm dò (trainval)"], positions, linestyle="none", marker="D", markersize=6,
        color=INK, label="Tập thăm dò (chó trainval)",
    )
    axis.set_yticks(positions, [name.replace("_", " ") for name in counts.index])
    axis.invert_yaxis()
    axis.grid(axis="y", visible=False)
    axis.set(xlabel="Số ảnh", title="Số ảnh chó mỗi giống: near-OOD và tập thăm dò")
    legend_above(axis)
    show_figure(study, fig, "03_6_ood_dog_breeds", "Chương 3", "Số ảnh chó theo giống của tập thăm dò và near-OOD")


# ─── Chạy ────────────────────────────────────────────────────────────────────────────────────────────
if AUTO_RUN:
    eda_annotations(STUDY)
    eda_ood_sets(STUDY)


## 8. Tiền xử lý: tăng cường khi huấn luyện, Resize 256 và cắt giữa 224 khi đánh giá

- **Làm:** định nghĩa phép tăng cường, phép đánh giá (một kiểu nội suy bilinear chung cho mọi mô hình), ảnh xóa nền, ảnh cắt vùng đầu, DataLoader có seed, bộ giải mã ảnh đánh giá.
- **Hình:** từng phép tăng cường trên một ảnh train; ba bước tiền xử lý đánh giá.


In [ ]:
# ─── 8.1 Phép biến đổi ảnh huấn luyện, đánh giá và hình học hộp bao ──────────────────────────────────
NORMALIZE = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
# Một kiểu nội suy chung cho cả bốn mô hình, kể cả EfficientNet-B0 (trọng số ImageNet của nó được công bố kèm
# bicubic): mọi mô hình nhận cùng một ảnh đầu vào nên chênh lệch kết quả là do mô hình, không do tiền xử lý.
INTERPOLATION = InterpolationMode.BILINEAR


def augmentation_steps():
    # §2.4.1: cạnh ngắn 256 như lúc đánh giá, cắt ngẫu nhiên 224 (thay vì cắt giữa), lật ngang, xoay ±15°,
    # đổi sáng/tương phản/bão hòa 20%. Tỷ lệ khung và độ phóng giống ảnh đánh giá, nên lượt có và không
    # tăng cường chỉ khác nhau ở các biến đổi ngẫu nhiên.
    return [
        transforms.Resize(CONFIG["resize_size"], interpolation=INTERPOLATION, antialias=True),
        transforms.RandomCrop(CONFIG["image_size"]),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomRotation(15, interpolation=INTERPOLATION, fill=128),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    ]


def evaluation_geometry(interpolation=INTERPOLATION):
    # §2.4.2: cạnh ngắn 256 rồi cắt giữa 224; trimap dùng NEAREST (§2.8.2)
    return transforms.Compose(
        [
            transforms.Resize(CONFIG["resize_size"], interpolation=interpolation, antialias=True),
            transforms.CenterCrop(CONFIG["image_size"]),
        ]
    )


def evaluation_layout(width, height):
    # Kích thước sau Resize và góc trên-trái của CenterCrop, cùng công thức với torchvision
    short, long = sorted((width, height))
    resized_long = int(CONFIG["resize_size"] * long / short)
    new_width, new_height = (
        (CONFIG["resize_size"], resized_long) if width <= height else (resized_long, CONFIG["resize_size"])
    )
    left = int(round((new_width - CONFIG["image_size"]) / 2.0))
    top = int(round((new_height - CONFIG["image_size"]) / 2.0))
    return new_width, new_height, left, top


def transform_box(box, width, height):
    # Hộp [x0, y0, x1, y1] của ảnh gốc sang hệ tọa độ 224×224 sau tiền xử lý đánh giá
    new_width, new_height, left, top = evaluation_layout(width, height)
    x0, y0, x1, y1 = box
    scale_x, scale_y = new_width / width, new_height / height
    size = CONFIG["image_size"]
    return [
        float(np.clip(x0 * scale_x - left, 0, size)),
        float(np.clip(y0 * scale_y - top, 0, size)),
        float(np.clip(x1 * scale_x - left, 0, size)),
        float(np.clip(y1 * scale_y - top, 0, size)),
    ]


def make_transform(training=False, augmentation=True):
    geometry = augmentation_steps() if training and augmentation else [evaluation_geometry()]
    return transforms.Compose(geometry + [transforms.ToTensor(), NORMALIZE])


def model_input(images):
    return NORMALIZE(images.float() / 255)  # Ảnh uint8 đã cắt 224, khớp ToTensor + Normalize


# ─── 8.2 Ảnh đầy đủ, ảnh xóa nền và ảnh cắt vùng đầu ─────────────────────────────────────────────────
def read_view(row, view="full"):
    image = read_rgb(row.image_path)
    if view == "no_background":
        if not row.has_valid_trimap:
            raise ValueError("Không dùng trimap rỗng để xóa nền.")
        with Image.open(row.trimap_path) as source:
            keep = np.array(source) != 2  # QĐ5: giữ cả nhãn 1 và nhãn 3.
        pixels = np.array(image).copy()
        pixels[~keep] = 128
        return Image.fromarray(pixels)
    if view == "head":
        if not row.has_bbox:
            raise ValueError("Thiếu bounding box vùng đầu; không tự tạo annotation.")
        x0, y0, x1, y1 = json.loads(row.bbox_json)
        dx, dy = 0.1 * (x1 - x0), 0.1 * (y1 - y0)
        box = (
            max(0, math.floor(x0 - dx)),
            max(0, math.floor(y0 - dy)),
            min(image.width, math.ceil(x1 + dx)),
            min(image.height, math.ceil(y1 + dy)),
        )
        return image.crop(box)
    if view != "full":
        raise ValueError("Biến thể dữ liệu không hợp lệ: " + view)
    return image


# ─── 8.3 DataLoader huấn luyện với seed cố định ──────────────────────────────────────────────────────
class PetDataset(Dataset):
    def __init__(self, frame, augmentation):
        self.frame = frame.reset_index(drop=True)
        self.transform = make_transform(training=True, augmentation=augmentation)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        return self.transform(read_rgb(row.image_path)), int(row.label)


def seed_worker(worker_id):
    seed = torch.initial_seed() % 2**32
    np.random.seed(seed)
    random.seed(seed)


def make_train_loader(frame, seed, augmentation):
    return DataLoader(
        PetDataset(frame, augmentation),
        batch_size=CONFIG["batch_size"],
        shuffle=True,
        num_workers=CONFIG["workers"],
        worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(seed),
        pin_memory=torch.cuda.is_available(),
    )


# ─── 8.4 Giải mã một lần ảnh đánh giá về 224×224 ─────────────────────────────────────────────────────
EVALUATION_SETS = {  # tên: (tập nguồn, biến thể ảnh, cột lọc)
    "train": ("train", "full", None),
    "val": ("val", "full", None),
    "val_head": ("val", "head", "has_bbox"),
    "probe_ood": ("probe_ood", "full", None),
    "test": ("test", "full", None),
    "test_no_background": ("test", "no_background", "has_valid_trimap"),
    "near_ood": ("near_ood", "full", None),
    "far_ood": ("far_ood", "full", None),
    "unseen_cat": ("unseen_cat", "full", None),
}
PHASE_A_SETS = ["train", "val", "val_head", "probe_ood"]  # Trước bước khóa không giải mã test hay OOD test
PHASE_B_SETS = ["test", "test_no_background", *OOD_SPLITS]


def decode_images(frame, view, label):
    geometry = evaluation_geometry()
    size = CONFIG["image_size"]
    images = torch.empty((len(frame), 3, size, size), dtype=torch.uint8)

    def decode(item):
        index, row = item
        images[index] = TF.pil_to_tensor(geometry(read_view(row, view)))

    parallel_map(decode, list(enumerate(frame.itertuples(index=False))), "Giải mã " + label)
    return {
        "images": images,
        "labels": frame.label.to_numpy(np.int64, copy=True),
        "image_id": frame.image_id.to_numpy(str),
    }


def decode_trimaps(frame):
    geometry = evaluation_geometry(InterpolationMode.NEAREST)
    trimaps = []
    for row in frame.itertuples(index=False):
        with Image.open(row.trimap_path) as source:
            trimaps.append(np.array(geometry(source)))
    return np.stack(trimaps)


def evaluation_cache(frames, names):
    cache = {}
    for name in names:
        source, view, column = EVALUATION_SETS[name]
        if source not in frames:
            continue  # unseen_cat là tùy chọn
        frame = frames[source] if column is None else frames[source][frames[source][column]]
        cache[name] = decode_images(frame, view, name)
    return cache


def with_trimaps(entry, frame):
    keep = frame.has_valid_trimap.to_numpy(copy=True)
    return {
        **{key: value[keep] for key, value in entry.items()},
        "trimaps": decode_trimaps(frame[keep]),
        "frame": frame[keep].reset_index(drop=True),
    }


# ─── 8.5 Hình minh họa tăng cường và tiền xử lý đánh giá ─────────────────────────────────────────────
def example_train_image(study, landscape=False):
    train = study.manifests["cats"]
    train = train[(train.split == "train") & (train.breed == "Bengal")].sort_values("image_id")
    return train[train.width >= 1.3 * train.height].iloc[0] if landscape else train.iloc[0]


def show_augmentation(study):
    row = example_train_image(study)
    image = read_rgb(row.image_path)
    resize, crop, _, rotate, jitter = augmentation_steps()
    full = transforms.Compose(augmentation_steps())
    torch.manual_seed(42)  # Minh họa tái lập được
    panels = [
        ("Ảnh gốc", image),
        ("Cạnh ngắn 256, cắt ngẫu nhiên 224×224", crop(resize(image))),
        ("Lật ngang (xác suất 0,5)", TF.hflip(image)),
        ("Xoay ngẫu nhiên trong ±15°", rotate(image)),
        ("Đổi sáng, tương phản, bão hòa", jitter(image)),
    ] + [(f"Pipeline đầy đủ, mẫu {k}", full(image)) for k in range(1, 4)]
    fig, axes = plt.subplots(2, 4, figsize=(14, 7.6), layout="constrained")
    for axis, (title, picture) in zip(axes.flat, panels):
        axis.imshow(picture)
        axis.set_title(title)
        image_axis(axis)
    figure_title(fig, "Tăng cường dữ liệu trên một ảnh train: " + row.image_id.replace("_", " "))
    show_figure(study, fig, "04_1_augmentation", "Chương 3", "Minh họa các phép tăng cường trên một ảnh train")


def show_evaluation_preprocessing(study):
    row = example_train_image(study, landscape=True)
    image = read_rgb(row.image_path)
    new_width, new_height, left, top = evaluation_layout(image.width, image.height)
    size = CONFIG["image_size"]
    scale_x, scale_y = new_width / image.width, new_height / image.height
    resized = transforms.Resize(CONFIG["resize_size"], interpolation=INTERPOLATION, antialias=True)(image)
    fig, axes = plt.subplots(
        1, 3, figsize=(15, 4.4), layout="constrained", width_ratios=[image.width / image.height, new_width / new_height, 1]
    )
    axes[0].imshow(image)
    draw_box(axes[0], (left / scale_x, top / scale_y, (left + size) / scale_x, (top + size) / scale_y))
    axes[0].set_title(f"Ảnh gốc {image.width}×{image.height}, khung sẽ được giữ")
    axes[1].imshow(resized)
    draw_box(axes[1], (left, top, left + size, top + size))
    axes[1].set_title(f"Cạnh ngắn về {CONFIG['resize_size']}: {new_width}×{new_height}")
    axes[2].imshow(evaluation_geometry()(image))
    axes[2].set_title(f"Cắt giữa {size}×{size}")
    for axis in axes:
        image_axis(axis)
    show_figure(study, fig, "04_2_evaluation_preprocessing", "Chương 3", "Tiền xử lý đánh giá: cạnh ngắn 256 rồi cắt giữa 224")


# ─── Chạy ────────────────────────────────────────────────────────────────────────────────────────────
if AUTO_RUN:
    show_augmentation(STUDY)
    show_evaluation_preprocessing(STUDY)


## 9. Mô hình: CNN baseline, MobileNetV2, EfficientNet-B0, ResNet50

- **Làm:** dựng backbone, GAP, Dropout 0,3 và Linear 12 đầu ra; quy định phần đóng băng và phần mở ở mỗi giai đoạn.
- **Xuất:** bảng số tham số được cập nhật theo giai đoạn và dung lượng (Bảng 2.6).


In [ ]:
# ─── 9.1 Trọng số ImageNet và khối tích chập của CNN baseline ────────────────────────────────────────
BACKBONES = {
    "resnet50": models.resnet50,
    "mobilenet_v2": models.mobilenet_v2,
    "efficientnet_b0": models.efficientnet_b0,
}


PRETRAINED_WEIGHTS = {  # Giữ IMAGENET1K_V1; DEFAULT của ResNet50/MobileNetV2 là V2
    "resnet50": models.ResNet50_Weights.IMAGENET1K_V1,
    "mobilenet_v2": models.MobileNet_V2_Weights.IMAGENET1K_V1,
    "efficientnet_b0": models.EfficientNet_B0_Weights.IMAGENET1K_V1,
}


def ensure_pretrained_weights(name):
    weights = PRETRAINED_WEIGHTS[name]
    filename = weights.url.rsplit("/", 1)[-1]
    expected = filename.rsplit("-", 1)[-1].split(".")[0]
    target = Path(torch.hub.get_dir()) / "checkpoints" / filename
    if target.is_file() and file_hash(target).startswith(expected):
        return
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_suffix(".download")
    started = time.monotonic()
    try:
        with urlopen(weights.url, timeout=CONFIG["network_read_timeout_seconds"]) as response:
            size = int(response.headers.get("Content-Length", 0))
            chunks = iter(lambda: response.read(1024 * 1024), b"")
            count = 0
            with temporary.open("wb") as stream:
                for _, chunk in progress(
                    chunks,
                    "Tải ImageNet " + name,
                    total=math.ceil(size / (1024 * 1024)) if size else None,
                ):
                    if time.monotonic() - started > CONFIG["download_timeout_seconds"]:
                        raise TimeoutError("Tải trọng số ImageNet quá thời gian cấu hình.")
                    stream.write(chunk)
                    count += len(chunk)
            if (size and count != size) or not file_hash(temporary).startswith(expected):
                raise ValueError("Trọng số tải thiếu hoặc không đúng checksum.")
        temporary.replace(target)
    finally:
        temporary.unlink(missing_ok=True)


def baseline_features(batchnorm=None):
    batchnorm = CONFIG["baseline_batchnorm"] if batchnorm is None else batchnorm
    # §2.5.1: ba khối Conv3×3 + ReLU + MaxPool 32/64/128 kênh, không BatchNorm
    blocks, channels_in = [], 3
    for channels in [32, 64, 128]:
        norm = [nn.BatchNorm2d(channels)] if batchnorm else []
        blocks.append(
            nn.Sequential(nn.Conv2d(channels_in, channels, 3, padding=1), *norm, nn.ReLU(), nn.MaxPool2d(2))
        )
        channels_in = channels
    return nn.Sequential(*blocks), channels_in


# ─── 9.2 Mô hình phân loại: backbone, GAP, Dropout, Linear ───────────────────────────────────────────
class CatClassifier(nn.Module):
    def __init__(self, name, pretrained=True, dropout=None, head_hidden=None, baseline_batchnorm=None):
        super().__init__()
        self.name = name
        if name == "cnn_baseline":
            self.features, dimension = baseline_features(baseline_batchnorm)
        else:
            if pretrained:
                ensure_pretrained_weights(name)
            net = BACKBONES[name](weights=PRETRAINED_WEIGHTS[name] if pretrained else None)
            if name == "resnet50":
                self.features = nn.Sequential(*list(net.children())[:-2])
                dimension = net.fc.in_features
            else:
                self.features, dimension = net.features, net.classifier[-1].in_features
        self.pool = nn.AdaptiveAvgPool2d(1)
        rate = CONFIG["dropout"] if dropout is None else dropout
        hidden = CONFIG["head_hidden"] if head_hidden is None else head_hidden
        if hidden:
            self.head = nn.Sequential(
                nn.Dropout(rate),
                nn.Linear(dimension, hidden),
                nn.ReLU(),
                nn.Dropout(rate),
                nn.Linear(hidden, len(CAT_BREEDS)),
            )
        else:
            self.head = nn.Sequential(nn.Dropout(rate), nn.Linear(dimension, len(CAT_BREEDS)))
        self.tune_indices, self.tune_units = [], []

    def forward_features(self, x):
        return self.pool(self.features(x)).flatten(1)

    def forward(self, x):
        return self.head(self.forward_features(x))

    def configure_stage(self, stage):
        for parameter in self.features.parameters():
            parameter.requires_grad = stage == "scratch"
        for parameter in self.head.parameters():
            parameter.requires_grad = True
        self.tune_units = self.unfrozen_units() if stage == "finetuned" else []
        self.tune_indices = [name for name, _ in self.tune_units]  # ghi vào metrics.json
        for _, module in self.tune_units:
            for parameter in module.parameters():
                parameter.requires_grad = True

    def feature_units(self):
        # Đơn vị mở khóa: tách các nhóm nn.Sequential thuần (layer1–4 của ResNet50, các stage của
        # EfficientNet-B0) thành từng khối con (Bottleneck, MBConv); InvertedResidual của MobileNetV2 và
        # Conv2dNormActivation (lớp con của Sequential) giữ nguyên là một đơn vị
        units = []
        for index, block in enumerate(self.features):
            if type(block) is nn.Sequential:
                units += [(f"{index}.{child}", module) for child, module in block.named_children()]
            else:
                units.append((str(index), block))
        return units

    def unfrozen_units(self):
        # fine_tune_fraction = None: giữ cách cũ, mở fine_tune_blocks phần tử cuối của features (§2.5.2).
        # Có fine_tune_fraction: mở số khối con cuối có tỷ lệ tham số gần mục tiêu nhất (hòa thì ít khối hơn)
        if CONFIG["fine_tune_fraction"] is None:
            count = len(self.features)
            return [(str(i), self.features[i]) for i in range(count - CONFIG["fine_tune_blocks"], count)]
        units = self.feature_units()
        sizes = [sum(p.numel() for p in module.parameters()) for _, module in units]
        target, cumulative, best = CONFIG["fine_tune_fraction"] * sum(sizes), 0, None
        for count in range(1, len(units) + 1):
            cumulative += sizes[-count]
            gap = abs(cumulative - target)
            if best is None or gap < best[0]:
                best = (gap, count)
        return units[len(units) - best[1]:]

    def training_mode(self, stage):
        self.train()
        if stage != "scratch":
            self.features.eval()  # §2.5.2: BatchNorm của phần đóng băng giữ chế độ đánh giá
            for _, module in self.tune_units:
                module.train()

    def trainable_parameters(self):
        return sum(parameter.numel() for parameter in self.parameters() if parameter.requires_grad)


def state_megabytes(model):
    return sum(tensor.numel() * tensor.element_size() for tensor in model.state_dict().values()) / 1e6


# ─── 9.3 Bảng số tham số theo giai đoạn ──────────────────────────────────────────────────────────────
def show_model_parameters(study):
    rows = []
    for name in MODEL_NAMES:
        model = CatClassifier(name, pretrained=False)
        stages = ["scratch"] if name == "cnn_baseline" else ["frozen", "finetuned"]
        counts = []
        for stage in stages:
            model.configure_stage(stage)
            counts.append(model.trainable_parameters())
        total = sum(parameter.numel() for parameter in model.parameters())
        rows.append(
            {
                "model": name,
                "total_parameters": total,
                "stage1_trainable": counts[0],
                "stage2_trainable": counts[1] if len(counts) > 1 else None,
                "stage2_fraction": counts[1] / total if len(counts) > 1 else None,
                "size_megabytes": round(state_megabytes(model), 2),
            }
        )
    table = pd.DataFrame(rows)
    table.to_csv(study.directory / "report_tables/model_parameters.csv", index=False)
    display(table)


# ─── Chạy ────────────────────────────────────────────────────────────────────────────────────────────
if AUTO_RUN:
    show_model_parameters(STUDY)


## 10. Vòng huấn luyện: một epoch, chọn checkpoint, một giai đoạn

- **Làm:** định nghĩa huấn luyện một epoch, đo validation, quy tắc giữ checkpoint (QĐ9), lưu và nạp lại checkpoint, hình đường huấn luyện.
- Cell này chỉ định nghĩa hàm, chưa chạy gì.


In [ ]:
# ─── 10.1 Trích xuất đặc trưng, logits và lưu NPZ ────────────────────────────────────────────────────
@torch.no_grad()
def extract_arrays(model, entry, device, label):
    model.eval()
    features, logits = [], []
    size = CONFIG["batch_size"]
    for _, start in progress(range(0, len(entry["images"]), size), label):
        h = model.forward_features(model_input(entry["images"][start:start + size]).to(device))
        features.append(h.cpu())
        logits.append(model.head(h).cpu())
    return {
        "features": torch.cat(features).numpy(),
        "logits": torch.cat(logits).numpy(),
        "labels": entry["labels"],
        "image_id": entry["image_id"],
    }


def save_npz(path, arrays):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("wb") as output:
        np.savez_compressed(output, **arrays)
    temporary.replace(path)


# ─── 10.2 Huấn luyện một epoch và đo trên validation ─────────────────────────────────────────────────
def classification_metrics(labels, predictions):
    labels, predictions = np.asarray(labels), np.asarray(predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, labels=CLASS_INDICES, average="macro", zero_division=0
    )
    return {
        "accuracy": float(np.mean(labels == predictions)),
        "macro_precision": float(precision),
        "macro_recall": float(recall),
        "macro_f1": float(f1),
        "weighted_f1": float(f1_score(labels, predictions, average="weighted", zero_division=0)),
    }


def train_epoch(model, loader, criterion, optimizer, stage, device, label):
    model.training_mode(stage)
    total_loss, labels, predictions = 0.0, [], []
    for bar, (images, targets) in progress(loader, label):
        images, targets = images.to(device), targets.to(device)
        logits = model(images)
        loss = criterion(logits, targets)
        if not torch.isfinite(loss):
            raise FloatingPointError("Loss không hữu hạn; dừng trước khi lưu checkpoint.")
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.detach()) * len(targets)
        labels += targets.tolist()
        predictions += logits.argmax(1).tolist()
        running = classification_metrics(labels, predictions)
        bar.set_postfix(
            loss=f"{total_loss / len(labels):.4f}",
            acc=f"{running['accuracy']:.3f}",
            f1=f"{running['macro_f1']:.3f}",
            refresh=False,
        )
    return {"loss": total_loss / len(labels), **classification_metrics(labels, predictions)}


def validation_pass(model, entry, criterion, device, label):
    arrays = extract_arrays(model, entry, device, label)
    loss = criterion(torch.from_numpy(arrays["logits"]), torch.from_numpy(arrays["labels"]))
    return {"loss": float(loss), **classification_metrics(arrays["labels"], arrays["logits"].argmax(1))}


# ─── 10.3 Chọn checkpoint theo QĐ9 và đóng gói checkpoint ────────────────────────────────────────────
def is_better(candidate, current):
    # QĐ9: F1 macro; hòa khi lệch < 1e-4 thì Accuracy cao hơn, rồi epoch sớm hơn
    if current is None:
        return True
    delta = candidate["macro_f1"] - current["macro_f1"]
    if abs(delta) < 1e-4:
        if candidate["accuracy"] != current["accuracy"]:
            return candidate["accuracy"] > current["accuracy"]
        return candidate["epoch"] < current["epoch"]
    return delta > 0


def checkpoint_payload(model, info, stage, epoch, validation):
    pretrained = CONFIG["pretrained"] and model.name in PRETRAINED_WEIGHTS
    return {
        "model_name": model.name,
        "state_dict": model.state_dict(),
        "class_mapping": CLASS_TO_IDX,
        "config": dict(CONFIG),
        "pretrained_weights": PRETRAINED_WEIGHTS[model.name].name if pretrained else None,
        "augmentation": info["augmentation"],
        "run_id": info["run_id"],
        "split_id": info["split_id"],
        "stage": stage,
        "epoch": epoch,
        "validation": validation,
        "git_commit": info["git_commit"],
        "notebook_source_sha256": info["notebook_source_sha256"],
    }


# ─── 10.4 Huấn luyện một giai đoạn, tóm tắt từng epoch, lưu checkpoint tốt nhất ──────────────────────
STAGE_EPOCHS = {"frozen": "frozen_epochs", "finetuned": "finetuned_epochs", "scratch": "baseline_epochs"}


def make_optimizer(model, stage):
    if stage == "finetuned":
        groups = [
            {"params": list(model.head.parameters()), "lr": CONFIG["head_lr"]},
            {
                "params": [p for p in model.features.parameters() if p.requires_grad],
                "lr": CONFIG["backbone_lr"],
            },
        ]
    else:
        rate = CONFIG["baseline_lr"] if stage == "scratch" else CONFIG["frozen_lr"]
        groups = [{"params": [p for p in model.parameters() if p.requires_grad], "lr": rate}]
    return torch.optim.AdamW(groups, weight_decay=CONFIG["weight_decay"])


def run_label(info):
    return f"{MODEL_LABELS[info['model']]} s{info['training_seed']}" + ("" if info["augmentation"] else " không tăng cường")


def train_stage(model, stage, train_frame, val_entry, run_dir, info, device):
    model.configure_stage(stage)
    optimizer = make_optimizer(model, stage)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=CONFIG["scheduler_patience"]
    )
    criterion = nn.CrossEntropyLoss()  # QĐ2: không label smoothing
    loader = make_train_loader(train_frame, info["training_seed"], info["augmentation"])
    max_epochs = CONFIG[STAGE_EPOCHS[stage]]
    label = run_label(info) + " " + STAGE_LABELS[stage]
    folder = run_dir / stage
    folder.mkdir()
    best, stale = None, 0
    for epoch in range(1, max_epochs + 1):
        if device.type == "cuda":
            torch.cuda.synchronize(device)
        started = time.monotonic()
        learning_rate = float(optimizer.param_groups[0]["lr"])
        training = train_epoch(model, loader, criterion, optimizer, stage, device, f"{label} {epoch}/{max_epochs}")
        validation = validation_pass(model, val_entry, criterion, device, "validation")
        if device.type == "cuda":
            torch.cuda.synchronize(device)
        scheduler.step(validation["macro_f1"])
        record = {
            "stage": stage,
            "epoch": epoch,
            "seconds": time.monotonic() - started,
            "learning_rate": learning_rate,
            **{"train_" + key: value for key, value in training.items()},
            **{"val_" + key: value for key, value in validation.items()},
        }
        info["history"].append(record)
        pd.DataFrame(info["history"]).to_csv(run_dir / "history.csv", index=False)
        if is_better({**validation, "epoch": epoch}, best):
            best, stale = {**validation, "epoch": epoch}, 0
            temporary = folder / "checkpoint.pt.tmp"
            torch.save(checkpoint_payload(model, info, stage, epoch, validation), temporary)
            temporary.replace(folder / "checkpoint.pt")
        else:
            stale += 1
        log_event(label + " epoch", "metrics", **record, best_macro_f1=best["macro_f1"], stale_epochs=stale)
        print(
            f"{label} | epoch {epoch:>2}/{max_epochs}"
            f" | train loss {training['loss']:.4f} acc {training['accuracy']:.3f}"
            f" | val loss {validation['loss']:.4f} acc {validation['accuracy']:.3f} F1 {validation['macro_f1']:.4f}"
            f" | F1 tốt nhất {best['macro_f1']:.4f} | lr {learning_rate:.0e} | {record['seconds']:.1f} s",
            flush=True,
        )
        if stale >= CONFIG["early_stopping_patience"]:
            break
    metrics = {
        "selection_metric": "validation_macro_f1",
        "tie_f1_tolerance": 1e-4,
        "best_epoch": best["epoch"],
        "epochs_run": epoch,
        "validation": {key: value for key, value in best.items() if key != "epoch"},
        "trainable_parameters": model.trainable_parameters(),
        "total_parameters": sum(parameter.numel() for parameter in model.parameters()),
        "unfrozen_feature_blocks": model.tune_indices,
        "is_smoke_test": CONFIG["profile"] == "smoke",
    }
    write_json(folder / "metrics.json", metrics)
    saved = torch.load(folder / "checkpoint.pt", map_location="cpu", weights_only=True)
    model.load_state_dict(saved["state_dict"])
    model.eval()
    if GROUP_SAVE_STAGE is not None:
        info["sync_status"][stage + "_checkpoint"] = bool(
            GROUP_SAVE_STAGE(
                run_dir=run_dir,
                stage=stage,
                checkpoint=saved,
                metrics=metrics,
                repo_id=CONFIG["hf_repo_id"],
            )
        )
        save_info(run_dir, info)
    return folder / "checkpoint.pt"


# ─── 10.5 Nạp checkpoint độc lập từ đĩa ──────────────────────────────────────────────────────────────
def reload_checkpoint(path, device="cpu"):
    saved = torch.load(path, map_location=device, weights_only=True)
    if saved["class_mapping"] != CLASS_TO_IDX:
        raise RuntimeError("Class mapping không khớp.")
    model = CatClassifier(
        saved["model_name"],
        pretrained=False,
        dropout=saved["config"]["dropout"],
        head_hidden=saved["config"].get("head_hidden", 0),
        baseline_batchnorm=saved["config"].get("baseline_batchnorm", False),
    ).to(device)
    model.load_state_dict(saved["state_dict"])
    model.eval()
    return model, saved


# ─── 10.6 Hình đường huấn luyện của một lượt ─────────────────────────────────────────────────────────
def kept_epochs(history):
    # Vị trí trên trục epoch (nối hai giai đoạn) của checkpoint được giữ ở mỗi giai đoạn, theo đúng QĐ9
    kept, best, stage = [], None, None
    for position, row in enumerate(history, 1):
        if row["stage"] != stage:
            stage, best = row["stage"], None
            kept.append(position)
        candidate = {"macro_f1": row["val_macro_f1"], "accuracy": row["val_accuracy"], "epoch": row["epoch"]}
        if is_better(candidate, best):
            best, kept[-1] = candidate, position
    return kept


def training_figure(info):
    frame = pd.DataFrame(info["history"])
    epochs = np.arange(1, len(frame) + 1)
    stages = frame.stage.to_numpy()
    boundaries = np.flatnonzero(stages[1:] != stages[:-1]) + 1.5
    fig, axes = plt.subplots(1, 4, figsize=(17, 3.9), layout="constrained")
    for axis, metric, label in zip(
        axes, ["loss", "accuracy", "macro_f1"], ["Loss (cross-entropy)", "Accuracy", "F1 macro"]
    ):
        axis.plot(epochs, frame["train_" + metric], color=BLUE, marker="o", label="Train")
        axis.plot(epochs, frame["val_" + metric], color=ORANGE, marker="s", label="Validation")
        axis.set_title(label)
    kept = np.array(kept_epochs(info["history"]))
    axes[2].plot(
        kept, frame.val_macro_f1.to_numpy()[kept - 1], linestyle="none", marker="*", markersize=15,
        color=ORANGE, markeredgecolor=INK, label="Checkpoint được giữ", zorder=3,
    )
    axes[3].plot(epochs, frame.learning_rate, color=INK, marker="^")
    axes[3].set(title="Learning rate (nhóm tham số đầu)", yscale="log")
    for axis in axes:
        axis.set_xlabel("Epoch")
        axis.margins(y=0.12)
        axis.xaxis.set_major_locator(MaxNLocator(integer=True))
        for boundary in boundaries:  # Ranh giới giữa giai đoạn đóng băng và tinh chỉnh
            axis.axvline(boundary, color=NEUTRAL, linewidth=1, linestyle=":")
            for text, side in [("← đóng băng ", "right"), (" tinh chỉnh →", "left")]:
                axis.text(
                    boundary, 0.99, text, transform=axis.get_xaxis_transform(),
                    ha=side, va="top", fontsize=8.5, color=MUTED,
                )
    legend_above(axes[2], row=axes)
    figure_title(fig, "Đường huấn luyện: " + run_label(info))
    return fig


## 11. Năm điểm từ chối, ngưỡng τ, nhiệt độ T*, NLL và ECE

- **Làm:** định nghĩa MSP, MaxLogit, Energy, Mahalanobis, KNN; τ là phân vị 5% điểm ID validation; AUROC, FPR@95TPR; T* cực tiểu NLL.
- Cell này chỉ định nghĩa hàm, chưa chạy gì.


In [ ]:
# ─── 11.1 MSP, MaxLogit và Energy từ logits ──────────────────────────────────────────────────────────
def logit_scores(logits, temperature=1.0):
    # Phụ lục B: softmax và logsumexp của SciPy trừ logit lớn nhất trước khi lấy mũ
    logits = np.asarray(logits, dtype=np.float64)
    scaled = logits / temperature
    return {
        "MSP": softmax(scaled, axis=1).max(axis=1),
        "MaxLogit": logits.max(axis=1),
        "Energy": temperature * logsumexp(scaled, axis=1),
    }


# ─── 11.2 Mahalanobis và KNN từ vector đặc trưng ─────────────────────────────────────────────────────
def unit_norm(x):
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)


class DistanceScores:
    def fit(self, features, labels, k):
        x, y = np.asarray(features, dtype=np.float64), np.asarray(labels)
        if set(y.tolist()) != set(CLASS_INDICES):
            raise ValueError("Train bank phải có đủ 12 lớp.")
        if len(x) < k:
            raise ValueError("Train bank nhỏ hơn k; không tự đổi k.")
        self.means = np.stack([x[y == c].mean(axis=0) for c in CLASS_INDICES])
        covariance = LedoitWolf().fit(x - self.means[y])  # §2.6.3: hiệp phương sai chung của phần dư
        self.precision, self.shrinkage = covariance.precision_, float(covariance.shrinkage_)
        return self.index(unit_norm(x), k)

    def index(self, bank, k):
        self.bank, self.k = bank, k  # §2.6.4: ngân hàng train đã chuẩn hóa L2
        self.neighbors = NearestNeighbors(n_neighbors=k, metric="euclidean", algorithm="brute").fit(bank)
        return self

    def score(self, features):
        x = np.asarray(features, dtype=np.float64)
        xp, mp = x @ self.precision, self.means @ self.precision
        squared = (
            (xp * x).sum(axis=1)[:, None]
            - 2 * xp @ self.means.T
            + (mp * self.means).sum(axis=1)[None, :]
        )
        distances = self.neighbors.kneighbors(unit_norm(x), return_distance=True)[0]
        return {
            "Mahalanobis": -np.maximum(squared, 0).min(axis=1),
            "KNN": -distances[:, self.k - 1],
        }

    def save(self, path):
        np.savez_compressed(
            path,
            means=self.means,
            precision=self.precision,
            shrinkage=self.shrinkage,
            bank=self.bank,
            k=self.k,
        )

    @classmethod
    def load(cls, path):
        with np.load(path, allow_pickle=False) as stored:
            distance = cls()
            distance.means, distance.precision = stored["means"], stored["precision"]
            distance.shrinkage = float(stored["shrinkage"])
            return distance.index(stored["bank"], int(stored["k"]))


# ─── 11.3 Ngưỡng τ trên validation và các chỉ số OOD ─────────────────────────────────────────────────
METHODS = ["MSP", "MaxLogit", "Energy", "Mahalanobis", "KNN"]
LOGIT_METHODS, DISTANCE_METHODS = METHODS[:3], METHODS[3:]
TEMPERATURE_METHODS = ["MSP", "Energy"]  # §2.7.4: bảng OOD phụ với T*
THRESHOLD_QUANTILE = 0.05  # §2.7.1
METHOD_COLORS = dict(zip(METHODS, [BLUE, ORANGE, AQUA, YELLOW, MAGENTA]))
METHOD_LINES = {**dict.fromkeys(LOGIT_METHODS, "-"), **dict.fromkeys(DISTANCE_METHODS, "--")}  # họ khoảng cách: nét đứt


def rejection_scores(arrays, distance):
    return {**logit_scores(arrays["logits"]), **distance.score(arrays["features"])}


def score_matrix(scores):
    return np.column_stack([scores[name] for name in METHODS])


def lock_thresholds(validation_scores):
    # Phụ lục B: phân vị nội suy tuyến tính, mặc định của numpy.quantile
    return {
        name: float(np.quantile(values, THRESHOLD_QUANTILE, method="linear"))
        for name, values in validation_scores.items()
    }


def auroc(id_scores, ood_scores):
    # Thống kê Mann–Whitney (hạng trung bình khi hòa), bằng roc_auc_score; mỗi cột một phương pháp
    ranks = rankdata(np.concatenate([id_scores, ood_scores]), axis=0)
    n_id, n_ood = len(id_scores), len(ood_scores)
    return (ranks[:n_id].sum(axis=0) - n_id * (n_id + 1) / 2) / (n_id * n_ood)


def fpr_at_95_tpr(id_scores, ood_scores):
    labels = np.r_[np.ones(len(id_scores)), np.zeros(len(ood_scores))]
    fpr, tpr, _ = roc_curve(labels, np.r_[id_scores, ood_scores], drop_intermediate=False)
    first = int(np.argmax(tpr >= 0.95))  # §2.7.2: điểm đầu tiên có TPR ≥ 0,95, nội suy với điểm trước
    return float(np.interp(0.95, tpr[first - 1:first + 1], fpr[first - 1:first + 1]))


def ood_metrics(id_scores, ood_scores, threshold):
    return {
        "auroc": float(auroc(id_scores, ood_scores)),
        "fpr_at_95_tpr": fpr_at_95_tpr(id_scores, ood_scores),
        "threshold": float(threshold),
        "tpr_at_threshold": float(np.mean(id_scores >= threshold)),
        "fpr_at_threshold": float(np.mean(ood_scores >= threshold)),
        "n_ood_accepted": int(np.sum(ood_scores >= threshold)),
        "n_id": len(id_scores),
        "n_ood": len(ood_scores),
    }


def accuracy_coverage(arrays, scores):
    correct = arrays["logits"].argmax(1) == arrays["labels"]
    order = np.argsort(-np.asarray(scores), kind="stable")
    sorted_scores = np.asarray(scores)[order]
    # Chỉ báo điểm cuối mỗi nhóm hòa; không tùy ý giữ một phần nhóm cùng điểm.
    ends = np.r_[np.flatnonzero(sorted_scores[:-1] != sorted_scores[1:]), len(order) - 1]
    accepted = ends + 1
    return pd.DataFrame(
        {
            "threshold": sorted_scores[ends],
            "coverage": accepted / len(order),
            "accuracy": np.cumsum(correct[order])[ends] / accepted,
            "accepted": accepted,
        }
    )


# ─── 11.4 Nhiệt độ T*, NLL và ECE ────────────────────────────────────────────────────────────────────
def negative_log_likelihood(logits, labels, temperature=1.0):
    scaled = np.asarray(logits, dtype=np.float64) / temperature
    return float(np.mean(logsumexp(scaled, axis=1) - scaled[np.arange(len(scaled)), labels]))


def fit_temperature(logits, labels):
    # §2.7.4: T* cực tiểu NLL trên validation
    optimum = minimize_scalar(
        lambda log_t: negative_log_likelihood(logits, labels, np.exp(log_t)),
        bounds=(-4, 4),
        method="bounded",
    )
    if not optimum.success:
        raise RuntimeError("Không tối ưu được nhiệt độ.")
    return float(np.exp(optimum.x))


def reliability_bins(logits, labels, temperature=1.0):
    probabilities = softmax(np.asarray(logits, dtype=np.float64) / temperature, axis=1)
    confidence, prediction = probabilities.max(axis=1), probabilities.argmax(axis=1)
    count = CONFIG["ece_bins"]  # §2.7.4: 15 bin đều
    edges = np.linspace(0, 1, count + 1)
    bins = np.minimum(np.searchsorted(edges, confidence, side="right") - 1, count - 1)
    rows, ece = [], 0.0
    for b in range(count):
        chosen = bins == b
        if not chosen.any():
            continue
        accuracy = float(np.mean(prediction[chosen] == labels[chosen]))
        mean_confidence = float(confidence[chosen].mean())
        ece += chosen.sum() / len(labels) * abs(accuracy - mean_confidence)
        rows.append({"bin": b, "n": int(chosen.sum()), "accuracy": accuracy, "confidence": mean_confidence})
    return float(ece), pd.DataFrame(rows)


def calibration_summary(logits, labels, temperature):
    ece_before, bins_before = reliability_bins(logits, labels)
    ece_after, bins_after = reliability_bins(logits, labels, temperature)
    summary = {
        "temperature": temperature,
        "n_images": len(labels),
        "accuracy": float(np.mean(np.argmax(logits, axis=1) == labels)),  # Không đổi theo T
        "nll_before": negative_log_likelihood(logits, labels),
        "nll_after": negative_log_likelihood(logits, labels, temperature),
        "ece_before": ece_before,
        "ece_after": ece_after,
    }
    return summary, bins_before, bins_after


## 12. Grad-CAM, IoU, R, f và các phép đánh giá phân loại

- **Làm:** định nghĩa Grad-CAM, IoU với trimap, tỷ lệ năng lượng R, báo cáo phân loại, CI bootstrap, McNemar, so sánh ảnh đầy đủ với ảnh cắt vùng đầu hoặc xóa nền.
- Cell này chỉ định nghĩa hàm, chưa chạy gì.


In [ ]:
# ─── 12.1 Bản đồ Grad-CAM cho một batch ──────────────────────────────────────────────────────────────
def gradcam(model, batch):
    model.eval()
    activations = []
    handle = model.features[-1].register_forward_hook(
        lambda module, inputs, output: activations.append(output)
    )
    try:
        logits = model(batch.requires_grad_(True))
        prediction = logits.argmax(1)
        # QĐ5: gradient logit lớp dự đoán tại tầng trước GAP; BN ở chế độ eval nên các ảnh độc lập
        target = logits.gather(1, prediction[:, None]).sum()
        gradient = torch.autograd.grad(target, activations[0])[0]
        heat = torch.relu((gradient.mean((2, 3), keepdim=True) * activations[0]).sum(1, keepdim=True))
        heat = nn.functional.interpolate(
            heat, size=batch.shape[-2:], mode="bilinear", align_corners=False
        )[:, 0]
        low = heat.amin((1, 2), keepdim=True)
        span = heat.amax((1, 2), keepdim=True) - low
        heat = torch.where(span > 0, (heat - low) / span.clamp_min(1e-12), torch.zeros_like(heat))
        return heat.detach().cpu().numpy(), prediction.cpu().numpy()
    finally:
        handle.remove()


# ─── 12.2 IoU, tỷ lệ năng lượng R và tỷ lệ diện tích f theo QĐ5 ──────────────────────────────────────
def cam_overlap(heat, trimap, threshold=0.5):
    if heat.shape != trimap.shape or not np.isfinite(heat).all():
        raise ValueError("Heatmap/trimap sai kích thước hoặc có giá trị không hữu hạn.")
    valid = trimap != 3
    if not valid.any():
        raise ValueError("Trimap không có điểm ảnh hợp lệ.")
    foreground = trimap == 1
    f = float(foreground.sum() / valid.sum())
    degenerate = bool(heat.max() == heat.min())
    attention = (heat >= threshold) & valid
    union = np.logical_or(attention, foreground).sum()
    iou = (
        0.0
        if degenerate or not union
        else float(np.logical_and(attention, foreground).sum() / union)
    )
    total = float(heat[valid].sum())
    ratio = None if degenerate or total == 0 else float(heat[foreground].sum() / total)
    return {"iou": iou, "R": ratio, "f": f, "degenerate": degenerate}


# ─── 12.3 Grad-CAM theo ảnh, mIoU, R, nhóm lỗi và kết quả theo giống ─────────────────────────────────
CAM_THRESHOLDS = [0.3, 0.5, 0.7]  # QĐ5: θ = 0,5 là ngưỡng chính


def gradcam_rows(model, entry, device, label):
    rows = []
    size = CONFIG["batch_size"]
    for _, start in progress(range(0, len(entry["image_id"]), size), label):
        heats, predictions = gradcam(model, model_input(entry["images"][start:start + size]).to(device))
        for index, heat, prediction in zip(range(start, start + len(heats)), heats, predictions):
            scores = {theta: cam_overlap(heat, entry["trimaps"][index], theta) for theta in CAM_THRESHOLDS}
            rows.append(
                {
                    "image_id": entry["image_id"][index],
                    "label": int(entry["labels"][index]),
                    "prediction": int(prediction),
                    "correct": bool(prediction == entry["labels"][index]),
                    **scores[0.5],
                    "iou_0.3": scores[0.3]["iou"],
                    "iou_0.7": scores[0.7]["iou"],
                }
            )
    result = pd.DataFrame(rows)
    result["R"] = result.R.astype(float)
    return result


def mean_or_none(values):
    return float(values.mean()) if values.notna().any() else None


def error_group(correct, iou, boundary):
    # QĐ5: ca sai chia theo ranh giới IoU khóa trên validation
    if correct:
        return "correct"
    if boundary is None:
        return "unavailable_boundary"
    return "low_iou" if iou < boundary else "high_iou"


def gradcam_report(model, entry, device, folder, split, iou_boundary=None):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    result = gradcam_rows(model, entry, device, f"Grad-CAM {split}")
    if split == "validation":
        correct_iou = result.loc[result.correct, "iou"]
        iou_boundary = float(correct_iou.median()) if len(correct_iou) else None
    result["error_group"] = [
        error_group(correct, iou, iou_boundary) for correct, iou in zip(result.correct, result.iou)
    ]
    result.to_csv(folder / "gradcam_per_image.csv", index=False)
    defined = result[result.R.notna()]
    summary = {
        "split": split,
        "n_evaluated": len(result),
        "miou": mean_or_none(result.iou),
        "miou_correct": mean_or_none(result.loc[result.correct, "iou"]),
        "miou_incorrect": mean_or_none(result.loc[~result.correct, "iou"]),
        "miou_0.3": mean_or_none(result["iou_0.3"]),
        "miou_0.7": mean_or_none(result["iou_0.7"]),
        "mean_R": mean_or_none(result.R),
        "mean_f": mean_or_none(result.f),
        "R_defined_count": len(defined),
        "mean_f_on_R_defined": mean_or_none(defined.f),
        "R_above_uniform_fraction": float((defined.R > defined.f).mean()) if len(defined) else None,
        "degenerate_count": int(result.degenerate.sum()),
        "iou_boundary": iou_boundary,
        "boundary_source": "median IoU of correct ID validation, theta=0.5",
        "R_definition": "CAM energy on label 1 / CAM energy on labels 1 and 2",
        "f_definition": "label 1 area / labels 1 and 2 area",
        "is_smoke_test": CONFIG["profile"] == "smoke",
    }
    write_json(folder / "gradcam_summary.json", summary)
    return summary, result


def gradcam_by_breed(result):
    rows = []
    for label, group in result.groupby("label"):
        rows.append(
            {
                "breed": CAT_BREEDS[label],
                "n_images": len(group),
                "accuracy": float(group.correct.mean()),
                "miou": float(group.iou.mean()),
                "miou_correct": mean_or_none(group.iou[group.correct]),
                "miou_incorrect": mean_or_none(group.iou[~group.correct]),
                "mean_R": mean_or_none(group.R),
                "mean_f": float(group.f.mean()),
            }
        )
    return pd.DataFrame(rows)


# ─── 12.4 Đánh giá phân loại, cặp giống dễ nhầm, khoảng tin cậy và McNemar ───────────────────────────
def confused_pairs(matrix):
    rows = [
        {
            "breed_a": CAT_BREEDS[a],
            "breed_b": CAT_BREEDS[b],
            "a_predicted_as_b": int(matrix[a, b]),
            "b_predicted_as_a": int(matrix[b, a]),
            "errors": int(matrix[a, b] + matrix[b, a]),
        }
        for a, b in combinations(CLASS_INDICES, 2)
    ]
    return pd.DataFrame(rows).sort_values("errors", ascending=False, kind="stable").reset_index(drop=True)


def classification_outputs(arrays, folder):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    labels, predictions = arrays["labels"], arrays["logits"].argmax(1)
    report = classification_report(
        labels, predictions, labels=CLASS_INDICES, target_names=CAT_BREEDS, output_dict=True, zero_division=0
    )
    pd.DataFrame({name: row for name, row in report.items() if isinstance(row, dict)}).T.to_csv(
        folder / "classification_report.csv"
    )
    pd.DataFrame(
        {
            "image_id": arrays["image_id"],
            "true_label": labels,
            "predicted_label": predictions,
            "confidence_T1": softmax(arrays["logits"], axis=1).max(1),
        }
    ).to_csv(folder / "predictions.csv", index=False)
    matrix = confusion_matrix(labels, predictions, labels=CLASS_INDICES)
    pd.DataFrame(matrix, index=CAT_BREEDS, columns=CAT_BREEDS).to_csv(folder / "confusion_matrix.csv")
    pairs = confused_pairs(matrix)
    pairs.to_csv(folder / "confused_pairs.csv", index=False)
    summary = {**classification_metrics(labels, predictions), "n_images": len(labels)}
    write_json(folder / "summary.json", summary)
    log_event("/".join(folder.parts[-3:]), "metrics", **summary)
    return summary, pairs


def classification_confidence(arrays):
    # §2.7.5: CI 95% bootstrap theo ảnh cho Accuracy và F1 macro
    labels, predictions = arrays["labels"], arrays["logits"].argmax(1)

    def metrics(index):
        return [
            np.mean(labels[index] == predictions[index]),
            f1_score(labels[index], predictions[index], labels=CLASS_INDICES, average="macro", zero_division=0),
        ]

    rng = np.random.default_rng(CONFIG["bootstrap_seed"])
    samples = np.array(
        [metrics(rng.integers(len(labels), size=len(labels))) for _ in range(CONFIG["bootstrap_repeats"])]
    )
    point = metrics(np.arange(len(labels)))
    rows = []
    for column, name in enumerate(["accuracy", "macro_f1"]):
        low, high = np.quantile(samples[:, column], [0.025, 0.975])
        rows.append(
            {
                "metric": name,
                "estimate": float(point[column]),
                "ci_low": float(low),
                "ci_high": float(high),
                "bootstrap_repeats": len(samples),
            }
        )
    return rows


def mcnemar_exact(correct_a, correct_b):
    # Kiểm định McNemar chính xác (nhị thức hai phía trên các ảnh bất đồng)
    only_a = int(np.sum(correct_a & ~correct_b))
    only_b = int(np.sum(~correct_a & correct_b))
    discordant = only_a + only_b
    return {
        "n_images": len(correct_a),
        "accuracy_a": float(np.mean(correct_a)),
        "accuracy_b": float(np.mean(correct_b)),
        "a_correct_b_wrong": only_a,
        "a_wrong_b_correct": only_b,
        "p_value": float(binomtest(min(only_a, only_b), discordant, 0.5).pvalue) if discordant else 1.0,
    }


def mcnemar_table(entries, test_correct):
    rows = []
    for a, b in combinations(entries, 2):
        if (a["seed"], a["stage"]) == (b["seed"], b["stage"]):
            rows.append(
                {
                    "seed": a["seed"],
                    "stage": a["stage"],
                    "model_a": a["model"],
                    "model_b": b["model"],
                    **mcnemar_exact(test_correct[a["name"]], test_correct[b["name"]]),
                }
            )
    return pd.DataFrame(rows)


# ─── 12.5 So sánh ảnh đầy đủ với ảnh cắt vùng đầu hoặc xóa nền ───────────────────────────────────────
def view_comparison(model, full_arrays, view_entry, device, label):
    # Chỉ suy luận, không huấn luyện lại
    view = extract_arrays(model, view_entry, device, label)
    index = pd.Index(full_arrays["image_id"]).get_indexer(view["image_id"])
    if (index < 0).any() or not np.array_equal(full_arrays["labels"][index], view["labels"]):
        raise AssertionError("Biến thể phải dùng đúng ảnh và nhãn của tập đầy đủ.")
    full = classification_metrics(view["labels"], full_arrays["logits"][index].argmax(1))
    changed = classification_metrics(view["labels"], view["logits"].argmax(1))
    return {
        "n_images": len(index),
        "excluded": len(full_arrays["image_id"]) - len(index),
        "full_accuracy": full["accuracy"],
        "view_accuracy": changed["accuracy"],
        "accuracy_delta": changed["accuracy"] - full["accuracy"],
        "full_macro_f1": full["macro_f1"],
        "view_macro_f1": changed["macro_f1"],
        "macro_f1_delta": changed["macro_f1"] - full["macro_f1"],
    }


## 13. Pha A: quy trình một lượt huấn luyện và giải mã ảnh

- **Làm:** định nghĩa một lượt (huấn luyện hai giai đoạn, rồi đo checkpoint tốt nhất trên validation và tập thăm dò); giải mã ảnh train, validation, tập thăm dò. Ảnh test và OOD chưa được mở.
- **Xuất:** thanh tiến độ giải mã và số ảnh mỗi tập.


In [ ]:
# ─── 13.1 Đo một checkpoint trên validation và tập thăm dò ───────────────────────────────────────────
def checkpoint_keys(record):
    return {key: record[key] for key in ["model", "seed", "stage", "augmentation"]}


def add_keys(rows, keys):
    frame = pd.DataFrame(rows).copy()
    for position, (column, value) in enumerate(keys.items()):
        frame.insert(position, column, value)
    return frame


def evaluate_validation(model, record, study):
    stage_dir, name = study.root / record["stage_dir"], record["name"]
    val = extract_arrays(model, study.cache["val"], study.device, name + " val")
    save_npz(stage_dir / "val.npz", val)
    classification_outputs(val, stage_dir / "validation_classification")
    if record["evaluation"] != "full":
        return
    train = extract_arrays(model, study.cache["train"], study.device, name + " train")
    save_npz(stage_dir / "train.npz", train)
    distance = run_step(
        "Fit Mahalanobis và KNN", DistanceScores().fit, train["features"], train["labels"], CONFIG["knn_k"]
    )
    distance.save(stage_dir / "distance_parameters.npz")
    val_scores = rejection_scores(val, distance)
    pd.DataFrame({"image_id": val["image_id"], **val_scores}).to_csv(
        stage_dir / "validation_scores.csv", index=False
    )
    temperature = run_step("Học T* trên validation", fit_temperature, val["logits"], val["labels"])
    calibration, _, _ = calibration_summary(val["logits"], val["labels"], temperature)
    write_json(stage_dir / "calibration.json", {**calibration, "scope": "ID validation, in-sample"})
    scaled = logit_scores(val["logits"], temperature)
    thresholds = {  # §2.7.1, §2.7.4: τ chỉ từ ID validation, không dùng ảnh chó
        "T1": lock_thresholds(val_scores),
        "T_star": lock_thresholds({method: scaled[method] for method in TEMPERATURE_METHODS}),
    }
    write_json(
        stage_dir / "thresholds.json",
        {
            "checkpoint_sha256": record["checkpoint_sha256"],
            "quantile": THRESHOLD_QUANTILE,
            "quantile_method": "linear",
            "selected_from": "ID validation",
            **thresholds,
        },
    )
    probe = extract_arrays(model, study.cache["probe_ood"], study.device, name + " probe_ood")
    save_npz(stage_dir / "probe_ood.npz", probe)
    probe_scores = rejection_scores(probe, distance)
    pd.DataFrame({"image_id": probe["image_id"], **probe_scores}).to_csv(
        stage_dir / "probe_scores.csv", index=False
    )
    probe_results = pd.DataFrame(
        [
            {"method": method, **ood_metrics(val_scores[method], probe_scores[method], thresholds["T1"][method])}
            for method in METHODS
        ]
    )
    probe_results.to_csv(stage_dir / "probe_results.csv", index=False)
    gradcam_report(model, study.cache["val_cam"], study.device, stage_dir / "gradcam", "validation")
    head = view_comparison(model, val, study.cache["val_head"], study.device, name + " val_head")
    write_json(stage_dir / "head_crop.json", head)
    keys = checkpoint_keys(record)
    study.tables["probe_results"].append(add_keys(probe_results, keys))
    study.tables["head_crop"].append(add_keys([head], keys))


# ─── 13.2 Huấn luyện một mô hình với một seed và lưu hồ sơ lượt chạy ─────────────────────────────────
def checkpoint_name(model_name, seed, stage, augmentation):
    return f"{model_name}_s{seed}_{stage}" + ("" if augmentation else "_noaug")


def stage_record(info, stage, checkpoint, model, root):
    metrics = read_json(checkpoint.parent / "metrics.json")
    return {
        "name": checkpoint_name(info["model"], info["training_seed"], stage, info["augmentation"]),
        "model": info["model"],
        "seed": info["training_seed"],
        "stage": stage,
        "augmentation": info["augmentation"],
        "evaluation": info["evaluation"],
        "run_id": info["run_id"],
        "stage_dir": checkpoint.parent.relative_to(root).as_posix(),
        "checkpoint_sha256": file_hash(checkpoint),
        "best_epoch": metrics["best_epoch"],
        "epochs_run": metrics["epochs_run"],
        "stage_seconds": sum(row["seconds"] for row in info["history"] if row["stage"] == stage),
        "trainable_parameters": metrics["trainable_parameters"],
        "total_parameters": metrics["total_parameters"],
        "size_megabytes": state_megabytes(model),
        **{"val_" + key: value for key, value in metrics["validation"].items()},
        "is_smoke_test": CONFIG["profile"] == "smoke",
    }


def train_one(model_name, seed, augmentation, study, sweep_point=None):
    # Ghi đè CONFIG trong phạm vi một lượt: model_overrides rồi overrides của điểm sweep (nếu có)
    overrides = {
        **CONFIG["model_overrides"].get(model_name, {}),
        **(sweep_point or {}).get("overrides", {}),
    }
    saved = {key: CONFIG[key] for key in overrides}
    CONFIG.update(overrides)
    try:
        return train_one_with_config(model_name, seed, augmentation, study, sweep_point, overrides)
    finally:
        CONFIG.update(saved)


def train_one_with_config(model_name, seed, augmentation, study, sweep_point, overrides):
    seed_everything(seed)
    started = time.monotonic()
    suffix = "" if augmentation else "_noaug"
    if sweep_point is not None:
        suffix += "_" + sweep_point["name"]
    run_id = f"{run_stamp()}_{model_name}_s{seed}{suffix}_{uuid4().hex[:8]}"
    run_dir = study.root / run_id
    run_dir.mkdir(parents=True, exist_ok=False)
    info = {
        "run_id": run_id,
        "owner": CONFIG.get("hf_authenticated_owner", CONFIG["owner"]),
        "display_name": CONFIG["owner"],
        "started_at_utc": utc_now(),
        "status": "running",
        "is_smoke_test": CONFIG["profile"] == "smoke",
        "model": model_name,
        "training_seed": seed,
        "augmentation": augmentation,
        "evaluation": (
            "full"
            if model_name != "cnn_baseline" and augmentation and CONFIG["profile"] != "sweep"
            else "classification"
        ),
        "sweep_point": None if sweep_point is None else sweep_point["name"],
        "config_overrides": overrides,
        "study_id": study.directory.name,
        "split_id": study.audit["split_id"],
        "data_version": study.provenance,
        "protocol_id": CONFIG["protocol_id"],
        "git_commit": SUBMISSION.get("git_commit"),
        "notebook_source_sha256": SUBMISSION.get("notebook_source_sha256"),
        "runner_provenance": dict(RUNNER_PROVENANCE),
        "source_submission": SUBMISSION,
        "sync_status": {},
        "history": [],
    }
    write_json(run_dir / "class_mapping.json", CLASS_TO_IDX)
    write_json(run_dir / "config.yaml", CONFIG)  # JSON hợp lệ cũng là YAML
    (run_dir / "requirements_freeze.txt").write_text(study.requirements, encoding="utf-8")
    save_info(run_dir, info)
    for split, frame in study.frames.items():
        frame.to_csv(run_dir / (split + "_manifest.csv"), index=False)
    train_frame = study.frames["train"]
    if CONFIG["train_fraction"] < 1:
        train_frame = train_frame.groupby("label", group_keys=False).sample(
            frac=CONFIG["train_fraction"], random_state=CONFIG["train_fraction_seed"]
        )
    info["train_images"] = len(train_frame)
    save_info(run_dir, info)
    try:
        model = run_step(
            "Khởi tạo " + model_name, CatClassifier, model_name, pretrained=CONFIG["pretrained"]
        ).to(study.device)
        sample = model_input(study.cache["val"]["images"][:1]).to(study.device)
        for stage in ["scratch"] if model_name == "cnn_baseline" else ["frozen", "finetuned"]:
            checkpoint = train_stage(model, stage, train_frame, study.cache["val"], run_dir, info, study.device)
            restored, _ = reload_checkpoint(checkpoint, study.device)
            with torch.no_grad():
                if not torch.allclose(model.eval()(sample), restored(sample), rtol=1e-4, atol=1e-5):
                    raise RuntimeError("Checkpoint độc lập cho logits khác.")
            model = restored  # §2.5.2: tinh chỉnh tiếp tục từ checkpoint đóng băng tốt nhất
            record = stage_record(info, stage, checkpoint, model, study.root)
            record["sweep_point"] = info["sweep_point"]
            record["train_images"] = info["train_images"]
            record["config_overrides"] = json.dumps(overrides, sort_keys=True)
            study.tables["validation_summary"].append(pd.DataFrame([record]))
            evaluate_validation(model, record, study)
            if study.device.type == "cuda":
                torch.cuda.empty_cache()
        info["status"] = "completed"
    except Exception:
        info["status"] = "failed"
        (run_dir / "error.txt").write_text(traceback.format_exc(), encoding="utf-8")
        raise
    finally:
        info["finished_at_utc"] = utc_now()
        info["training_validation_seconds"] = sum(row["seconds"] for row in info["history"])
        info["seconds_before_final_sync"] = time.monotonic() - started
        save_info(run_dir, info)
        final_ok = sync_run(run_dir, study.api, "final" if info["status"] == "completed" else "failed")
        write_json(
            run_dir / "sync_receipt.json",
            {
                "final_upload_success": final_ok,
                "checked_at_utc": utc_now(),
                "total_seconds": time.monotonic() - started,
            },
        )
    section = "Mục 4.2" if augmentation else "Mục 4.5"
    show_figure(study, training_figure(info), f"06_training_{model_name}_s{seed}{suffix}", section, "Đường huấn luyện " + run_label(info))


# ─── 13.3 Giải mã ảnh pha A và huấn luyện mọi seed của một mô hình ───────────────────────────────────
def prepare_phase_a(study):
    # Giải mã một lần các ảnh pha A về 224×224; ảnh test và các tập OOD của pha B chưa được mở
    study.cache = run_step("Giải mã ảnh pha A", evaluation_cache, study.frames, PHASE_A_SETS)
    study.cache["val_cam"] = with_trimaps(study.cache["val"], study.frames["val"])
    print(
        "Ảnh đã giải mã:",
        ", ".join(f"{name} {len(entry['image_id'])}" for name, entry in study.cache.items()),
        flush=True,
    )


def run_sweep(study):
    # Mỗi điểm × mô hình × seed là một lượt pha A; chỉ ghi chỉ số validation, không khóa, không pha B
    for point in CONFIG["sweep_grid"]:
        for model_name in point.get("models", CONFIG["models"]):
            for seed in CONFIG["training_seeds"]:
                train_one(model_name, seed, True, study, sweep_point=point)
    write_tables(study)
    summary = collected(study, "validation_summary")
    summary.to_csv(study.directory / "sweep_results.csv", index=False)
    display(
        summary.pivot_table(
            index=["sweep_point", "model"], columns="stage", values="val_macro_f1", aggfunc=["mean", "std", "count"]
        ).round(4)
    )
    study.info.update(status="sweep_completed", finished_at_utc=utc_now(), sweep_grid=CONFIG["sweep_grid"])
    save_info(study.directory, study.info)
    sync_run(study.directory, study.api, "Sweep")


def train_model(study, model_name):
    # Một mô hình: mỗi seed một lượt có tăng cường, rồi mỗi seed một lượt không tăng cường (trừ CNN baseline)
    if model_name not in CONFIG["models"]:
        print(MODEL_LABELS[model_name], "không nằm trong CONFIG['models'] nên được bỏ qua.", flush=True)
        return
    plan = [(seed, True) for seed in CONFIG["training_seeds"]]
    if CONFIG["run_augmentation_ablation"] and model_name != "cnn_baseline":
        plan += [(seed, False) for seed in CONFIG["training_seeds"]]
    for seed, augmentation in plan:
        train_one(model_name, seed, augmentation, study)
    write_tables(study)
    columns = {
        "seed": "Seed",
        "stage": "Giai đoạn",
        "augmentation": "Tăng cường",
        "best_epoch": "Epoch được giữ",
        "epochs_run": "Số epoch đã chạy",
        "val_accuracy": "Accuracy validation",
        "val_macro_f1": "F1 macro validation",
        "stage_seconds": "Thời gian (giây)",
    }
    summary = collected(study, "validation_summary")
    rows = summary.loc[summary.model == model_name, list(columns)].assign(stage=lambda f: f.stage.map(STAGE_LABELS))
    display(rows.rename(columns=columns).round(4))


# ─── Chạy ────────────────────────────────────────────────────────────────────────────────────────────
if AUTO_RUN:
    prepare_phase_a(STUDY)

if AUTO_RUN and CONFIG["profile"] == "sweep":
    run_sweep(STUDY)
    AUTO_RUN = False  # Dừng mọi cell sau: không huấn luyện chính thức, không khóa, không pha B
    print("Sweep xong. Kết quả: sweep_results.csv trong thư mục study trên Hugging Face.", flush=True)


## 14. Pha A: huấn luyện ResNet50

- **Làm:** 3 seed có tăng cường (đóng băng tối đa 10 epoch, rồi tinh chỉnh tối đa 20 epoch), sau đó 3 seed không tăng cường; mỗi checkpoint tốt nhất được đo năm điểm từ chối, τ, T* và Grad-CAM trên validation.
- **Xuất:** thanh tiến độ, một dòng tóm tắt mỗi epoch, bảng validation của mô hình.
- **Hình:** đường huấn luyện mỗi lượt (loss, accuracy, F1, learning rate).


In [ ]:
if AUTO_RUN:
    train_model(STUDY, "resnet50")


## 15. Pha A: huấn luyện MobileNetV2

- **Làm:** cùng quy trình với cell 14.
- **Xuất:** thanh tiến độ, một dòng tóm tắt mỗi epoch, bảng validation của mô hình.
- **Hình:** đường huấn luyện mỗi lượt.


In [ ]:
if AUTO_RUN:
    train_model(STUDY, "mobilenet_v2")


## 16. Pha A: huấn luyện EfficientNet-B0

- **Làm:** cùng quy trình với cell 14.
- **Xuất:** thanh tiến độ, một dòng tóm tắt mỗi epoch, bảng validation của mô hình.
- **Hình:** đường huấn luyện mỗi lượt.


In [ ]:
if AUTO_RUN:
    train_model(STUDY, "efficientnet_b0")


## 17. Pha A: huấn luyện CNN baseline

- **Làm:** 3 seed, học từ đầu tối đa 30 epoch; chỉ đo phân loại, không đo điểm từ chối.
- **Xuất:** thanh tiến độ, một dòng tóm tắt mỗi epoch, bảng validation của mô hình.
- **Hình:** đường huấn luyện mỗi lượt.


In [ ]:
if AUTO_RUN:
    train_model(STUDY, "cnn_baseline")


## 18. Pha A: kết quả validation của các cấu hình

- **Xuất:** bảng trung bình ± độ lệch chuẩn qua seed của từng cấu hình.
- **Hình:** F1 macro và accuracy validation của từng cấu hình.


In [ ]:
VALIDATION_METRICS = ["val_accuracy", "val_macro_f1", "val_macro_precision", "val_macro_recall"]


def show_validation_summary(study):
    summary = collected(study, "validation_summary")
    stats = mean_std(summary[summary.augmentation], ["model", "stage"], VALIDATION_METRICS)
    display(mean_std_text(stats, VALIDATION_METRICS))
    labels = [configuration_label(model, stage) for model, stage in zip(stats.model, stats.stage)]
    fig, axis = plt.subplots(figsize=(9, 0.55 * len(stats) + 1.6))
    dot_rows(
        axis,
        labels,
        [
            ("F1 macro", stats.val_macro_f1_mean, stats.val_macro_f1_std, {"color": BLUE, "marker": "o"}),
            ("Accuracy", stats.val_accuracy_mean, stats.val_accuracy_std, {"color": ORANGE, "marker": "s"}),
        ],
    )
    axis.set(
        xlabel="Giá trị trên ID validation, trung bình ± độ lệch chuẩn qua seed",
        title=f"{len(stats)} cấu hình trên ID validation",
    )
    legend_above(axis)
    show_figure(study, fig, "06_validation_comparison", "Mục 4.3", "F1 macro và Accuracy validation của các cấu hình")


if AUTO_RUN:
    show_validation_summary(STUDY)


## 19. Bước khóa: checkpoint triển khai, phương pháp từ chối, τ và T*

- **Làm:** chọn checkpoint triển khai (F1 validation cao nhất, không quá 25 MB); chọn phương pháp từ chối (tỷ lệ chó thăm dò lọt qua τ thấp nhất, chênh trong 2 điểm phần trăm thì xét AUROC); ghi SHA-256 các tệp vào `locked_selection.json`. Bước này không huấn luyện gì.
- **Xuất:** checkpoint và phương pháp được chọn, bảng so sánh năm phương pháp, bảng dung lượng và chất lượng.


In [ ]:
# ─── 19.1 Tệp được khóa và kiểm tra SHA-256 ──────────────────────────────────────────────────────────
LOCKED_FILES = {
    "full": [
        "checkpoint.pt",
        "thresholds.json",
        "calibration.json",
        "distance_parameters.npz",
        "gradcam/gradcam_summary.json",
    ],
    "classification": ["checkpoint.pt"],
}


LOCKED_KEYS = [
    "name",
    "model",
    "seed",
    "stage",
    "stage_dir",
    "checkpoint_sha256",
    "val_macro_f1",
    "val_accuracy",
    "best_epoch",
    "size_megabytes",
]


def locked_entry(record, root):
    stage_dir = root / record["stage_dir"]
    return {
        **{
            key: record[key]
            for key in ["name", "model", "seed", "stage", "augmentation", "evaluation", "run_id", "stage_dir"]
        },
        "sha256": {name: file_hash(stage_dir / name) for name in LOCKED_FILES[record["evaluation"]]},
    }


def verify_locked_files(stage_dir, hashes):
    for name, digest in hashes.items():
        if file_hash(stage_dir / name) != digest:
            raise RuntimeError("Tệp đã khóa bị thay đổi trước pha B: " + str(stage_dir / name))


# ─── 19.2 Chọn checkpoint triển khai và phương pháp từ chối ──────────────────────────────────────────
def best_checkpoint(records):
    best = None
    for record in records.to_dict("records"):
        candidate = {
            "macro_f1": record["val_macro_f1"],
            "accuracy": record["val_accuracy"],
            "epoch": record["best_epoch"],
            "record": record,
        }
        if is_better(candidate, best):
            best = candidate
    return best["record"]


REJECTION_TIE_MARGIN = 0.02  # Chênh không quá 2 điểm phần trăm (10 trong 500 ảnh thăm dò) thì coi là hòa


def choose_rejection_method(configuration):
    # Tiêu chí chính: tỷ lệ ảnh thăm dò lọt qua τ thấp nhất, trung bình qua seed của cấu hình triển khai.
    # Các phương pháp trong biên hòa so tiếp bằng AUROC thăm dò trung bình; vẫn hòa thì theo thứ tự METHODS.
    stats = configuration.groupby("method")[["fpr_at_threshold", "auroc"]].mean().reindex(METHODS)
    tied = stats[stats.fpr_at_threshold <= stats.fpr_at_threshold.min() + REJECTION_TIE_MARGIN + 1e-12]
    return tied.auroc.idxmax(), stats, list(tied.index)


def lock_selection(study):
    if CONFIG["profile"] == "sweep":
        raise RuntimeError("Profile sweep không được khóa lựa chọn.")
    # Bước khóa thứ hai: mọi lựa chọn chỉ dựa trên ID validation và tập thăm dò
    summary = collected(study, "validation_summary")
    probe = collected(study, "probe_results")
    full = summary[summary.evaluation == "full"]
    candidates = full[full.size_megabytes <= CONFIG["deployment_max_megabytes"]]
    if candidates.empty:
        raise RuntimeError("Không có checkpoint học chuyển giao nào trong giới hạn dung lượng triển khai.")
    chosen, unconstrained = best_checkpoint(candidates), best_checkpoint(full)
    configuration = probe[(probe.model == chosen["model"]) & (probe.stage == chosen["stage"])]
    method, probe_stats, tied = choose_rejection_method(configuration)
    stage_dir = study.root / chosen["stage_dir"]
    study.selection = {
        "locked_at_utc": utc_now(),
        "protocol_id": CONFIG["protocol_id"],
        "split_id": study.audit["split_id"],
        "probe_id": study.audit["probe_id"],
        "is_smoke_test": CONFIG["profile"] == "smoke",
        "deployment": {key: chosen[key] for key in LOCKED_KEYS},
        "best_unconstrained": {key: unconstrained[key] for key in LOCKED_KEYS},
        "deployment_rule": (
            "QĐ9 trên ID validation (F1 macro; lệch < 1e-4 thì Accuracy cao hơn, rồi epoch sớm hơn; "
            "vẫn hòa thì giữ checkpoint huấn luyện trước) trong các checkpoint học chuyển giao có tham số "
            f"≤ {CONFIG['deployment_max_megabytes']} MB (§2.10.1); best_unconstrained bỏ giới hạn dung lượng"
        ),
        "rejection": {
            "method": method,
            "threshold": read_json(stage_dir / "thresholds.json")["T1"][method],
            "temperature": read_json(stage_dir / "calibration.json")["temperature"],
            "probe_fpr_at_threshold_mean": {name: float(value) for name, value in probe_stats.fpr_at_threshold.items()},
            "probe_auroc_mean": {name: float(value) for name, value in probe_stats.auroc.items()},
            "tie_margin": REJECTION_TIE_MARGIN,
            "tied_methods": tied,
            "probe_seeds": sorted(int(seed) for seed in configuration.seed.unique()),
            "rule": (
                "Tỷ lệ ảnh thăm dò có điểm ≥ τ (lọt qua) thấp nhất, trung bình qua seed của cấu hình triển khai; "
                "τ là ngưỡng T = 1 khóa trên ID validation"
            ),
            "tie_rule": (
                f"Chênh không quá {100 * REJECTION_TIE_MARGIN:g} điểm phần trăm thì coi là hòa và chọn AUROC thăm dò "
                "trung bình cao hơn; vẫn hòa thì chọn phương pháp đứng trước: " + ", ".join(METHODS)
            ),
            "threshold_rule": "τ là phân vị 5% điểm ID validation, T* cực tiểu NLL trên ID validation; không dùng ảnh chó",
        },
        "checkpoints": [locked_entry(record, study.root) for record in summary.to_dict("records")],
    }
    write_json(study.directory / "locked_selection.json", study.selection)
    log_event("Khóa lựa chọn", "completed", deployment=chosen["name"], rejection_method=method)


# ─── 19.3 Bảng dung lượng và chất lượng, in kết quả khóa ─────────────────────────────────────────────
def deployment_dir(study):
    return study.root / study.selection["deployment"]["stage_dir"]


def deployment_tradeoff(study):
    keys = ["model", "stage"]
    summary = collected(study, "validation_summary")
    full = summary[summary.evaluation == "full"]
    table = mean_std(full, keys, ["val_accuracy", "val_macro_f1"])
    table.insert(2, "size_megabytes", full.groupby(keys, sort=False).size_megabytes.first().to_numpy())
    table.insert(3, "within_size_limit", table.size_megabytes <= CONFIG["deployment_max_megabytes"])
    if "test_classification" in study.tables:
        test = collected(study, "test_classification")
        test = test[test.augmentation & (test.model != "cnn_baseline")]
        test_stats = mean_std(test, keys, ["accuracy", "macro_f1"]).drop(columns="n_seeds")
        table = table.merge(test_stats.rename(columns=lambda c: c if c in keys else "test_" + c), on=keys)
    roles = {role: study.selection[role] for role in ["deployment", "best_unconstrained"]}
    table["locked_role"] = [
        ", ".join(f"{role} (seed {chosen['seed']})" for role, chosen in roles.items() if (chosen["model"], chosen["stage"]) == key)
        for key in zip(table.model, table.stage)
    ]
    return table


def publish_lock(study):
    write_tables(study)
    tradeoff = deployment_tradeoff(study)
    tradeoff.to_csv(study.directory / "report_tables/deployment_tradeoff.csv", index=False)
    study.info.update(status="locked", locked_at_utc=study.selection["locked_at_utc"])
    save_info(study.directory, study.info)
    sync_run(study.directory, study.api, "Khóa lựa chọn")
    for role, title in [("deployment", "Checkpoint triển khai"), ("best_unconstrained", "Tốt nhất không giới hạn dung lượng")]:
        chosen = study.selection[role]
        print(
            f"{title}: {configuration_label(chosen['model'], chosen['stage'])} seed {chosen['seed']}"
            f" | {chosen['size_megabytes']:.1f} MB | F1 val {chosen['val_macro_f1']:.4f} | Accuracy val {chosen['val_accuracy']:.4f}"
        )
    rejection = study.selection["rejection"]
    method = rejection["method"]
    print(
        f"Phương pháp từ chối: {method} | chó thăm dò lọt qua τ {rejection['probe_fpr_at_threshold_mean'][method]:.1%}"
        f" | AUROC thăm dò {rejection['probe_auroc_mean'][method]:.4f}"
        f" | τ = {rejection['threshold']:.4f} | T* = {rejection['temperature']:.4f}"
    )
    comparison = pd.DataFrame(
        {
            "Chó thăm dò lọt qua τ": rejection["probe_fpr_at_threshold_mean"],
            "AUROC thăm dò": rejection["probe_auroc_mean"],
        }
    ).round(4)
    comparison["Trong biên hòa"] = comparison.index.isin(rejection["tied_methods"])
    comparison["Được chọn"] = comparison.index == method
    display(comparison)
    display(mean_std_text(tradeoff, ["val_accuracy", "val_macro_f1"]))


# ─── Chạy ────────────────────────────────────────────────────────────────────────────────────────────
if AUTO_RUN:
    lock_selection(STUDY)
    publish_lock(STUDY)


## 20. Tập thăm dò: năm phương pháp từ chối trên validation và 500 ảnh chó

- **Xuất:** bảng AUROC, FPR@95TPR, số chó nhận nhầm theo từng seed và theo từng cấu hình.
- **Hình:** histogram điểm ID validation và chó thăm dò kèm ngưỡng τ; đường ROC năm phương pháp.


In [ ]:
# ─── 20.1 Bảng thăm dò năm phương pháp ───────────────────────────────────────────────────────────────
PROBE_COLUMNS = {
    "auroc": "AUROC",
    "fpr_at_95_tpr": "FPR@95TPR",
    "n_ood_accepted": "Chó nhận nhầm",
    "fpr_at_threshold": "Tỷ lệ chó nhận nhầm",
    "tpr_at_threshold": "Tỷ lệ ID giữ lại",
}


def show_probe_tables(study):
    probe = collected(study, "probe_results")
    deployment = study.selection["deployment"]
    rows = probe[(probe.model == deployment["model"]) & (probe.stage == deployment["stage"])]
    print("Cấu hình triển khai:", configuration_label(deployment["model"], deployment["stage"]), "| từng seed")
    display(rows[["seed", "method", *PROBE_COLUMNS]].rename(columns=PROBE_COLUMNS).round(4))
    print("Mọi cấu hình học chuyển giao | trung bình ± độ lệch chuẩn qua seed")
    stats = mean_std(probe, ["model", "stage", "method"], list(PROBE_COLUMNS))
    display(mean_std_text(stats, list(PROBE_COLUMNS), 3).rename(columns=PROBE_COLUMNS))


# ─── 20.2 Histogram điểm và đường ROC ────────────────────────────────────────────────────────────────
def score_histogram_figure(id_scores, ood_sets, thresholds, id_label):
    fig, axes = plt.subplots(
        len(ood_sets), len(METHODS), figsize=(3.6 * len(METHODS), 3.1 * len(ood_sets)), squeeze=False, layout="constrained"
    )
    for axes_row, (name, scores) in zip(axes, ood_sets.items()):
        for axis, method in zip(axes_row, METHODS):
            edges = np.histogram_bin_edges(np.r_[id_scores[method], scores[method]], bins=40)
            axis.hist(id_scores[method], bins=edges, density=True, color=BLUE, alpha=0.6, label=id_label)
            axis.hist(scores[method], bins=edges, density=True, color=ORANGE, alpha=0.6, label="Tập OOD ghi ở đầu hàng")
            axis.axvline(thresholds[method], color=INK, linewidth=1.5, label="Ngưỡng τ: điểm bên trái bị từ chối")
            accepted = float(np.mean(scores[method] >= thresholds[method]))
            axis.set(title=f"{method}: {accepted:.1%} OOD lọt qua τ", xlabel="Điểm (càng lớn càng giống ID)", yticks=[])
            axis.xaxis.set_major_locator(MaxNLocator(4))
        axes_row[0].set_ylabel(name + "\nmật độ")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=3, bbox_to_anchor=(0.5, 1.0))
    return fig


def roc_figure(id_scores, ood_scores, title):
    labels = np.r_[np.ones(len(id_scores)), np.zeros(len(ood_scores))]
    fig, axis = plt.subplots(figsize=(6.4, 6))
    axis.plot([0, 1], [0, 1], color=NEUTRAL, linewidth=1, linestyle=":")
    axis.axhline(0.95, color=NEUTRAL, linewidth=1)
    axis.annotate(
        "TPR = 0,95", (1.02, 0.95), xycoords=("axes fraction", "data"), va="center", fontsize=9, color=MUTED,
        annotation_clip=False,
    )
    for method in METHODS:
        fpr, tpr, _ = roc_curve(labels, np.r_[id_scores[method], ood_scores[method]])
        value = auroc(id_scores[method].to_numpy(), ood_scores[method].to_numpy())
        axis.plot(fpr, tpr, color=METHOD_COLORS[method], linestyle=METHOD_LINES[method], label=f"{method}: AUROC {value:.3f}")
    axis.set(
        xlabel="FPR: tỷ lệ ảnh OOD bị nhận", ylabel="TPR: tỷ lệ ảnh ID được giữ", title=title,
        xlim=(-0.01, 1.01), ylim=(-0.01, 1.01), aspect="equal",
    )
    axis.legend(loc="upper left", bbox_to_anchor=(1.02, 0.88))
    return fig


def show_probe_figures(study):
    folder, name = deployment_dir(study), study.selection["deployment"]["name"]
    val = pd.read_csv(folder / "validation_scores.csv")
    probe = pd.read_csv(folder / "probe_scores.csv")
    thresholds = read_json(folder / "thresholds.json")["T1"]
    show_figure(
        study,
        score_histogram_figure(val, {OOD_LABELS["probe_ood"]: probe}, thresholds, "ID validation"),
        "07_probe_histograms",
        "Mục 4.6",
        "Histogram điểm ID validation và tập thăm dò, checkpoint triển khai " + name,
    )
    show_figure(
        study,
        roc_figure(val, probe, "ROC giữa ID validation và tập thăm dò\n" + name),
        "07_probe_roc",
        "Mục 4.6",
        "Đường ROC năm phương pháp trên tập thăm dò, checkpoint triển khai " + name,
    )


# ─── Chạy ────────────────────────────────────────────────────────────────────────────────────────────
if AUTO_RUN:
    show_probe_tables(STUDY)
    show_probe_figures(STUDY)


## 21. Grad-CAM trên validation: ảnh, trimap, hộp bao, bản đồ nhiệt

- **Hình:** một ảnh bốn ô (ảnh, trimap, hộp bao vùng đầu, Grad-CAM); mỗi giống một hàng bốn ô kèm IoU và R.


In [ ]:
# ─── 21.1 Grad-CAM của checkpoint triển khai và cách vẽ một hàng ảnh ─────────────────────────────────
def deployment_model(study):
    return reload_checkpoint(deployment_dir(study) / "checkpoint.pt", study.device)[0]


def cam_details(model, entry, indices, device):
    heats, predictions = gradcam(model, model_input(entry["images"][indices]).to(device))
    return [
        {"index": index, "heat": heat, "prediction": int(prediction), **cam_overlap(heat, entry["trimaps"][index])}
        for index, heat, prediction in zip(indices, heats, predictions)
    ]


def cam_caption(entry, detail):
    true = CAT_BREEDS[entry["labels"][detail["index"]]].replace("_", " ")
    predicted = CAT_BREEDS[detail["prediction"]].replace("_", " ")
    ratio = "không xác định" if detail["R"] is None else f"{detail['R']:.2f}"
    return f"Thật: {true}\nDự đoán: {predicted}", f"IoU {detail['iou']:.2f}, R {ratio}"


def draw_cam_row(axes, image, trimap, heat, box=None):
    # §2.8.2: mọi ô dùng ảnh, trimap và hộp bao đã qua cùng phép Resize 256 + cắt giữa 224
    picture = image.permute(1, 2, 0).numpy()
    axes[0].imshow(picture)
    axes[1].imshow(trimap_display(trimap), cmap="gray", vmin=0, vmax=1)
    if box is not None:
        axes[2].imshow(picture)
        draw_box(axes[2], box)
    axes[-1].imshow(picture)
    axes[-1].imshow(heat, cmap="inferno", alpha=0.5, vmin=0, vmax=1)
    # Hai vùng dùng để tính IoU: con vật theo trimap (viền trắng) và vùng Grad-CAM ≥ 0,5 (viền xanh nét đứt)
    for mask, color, style in [(trimap == 1, "white", "solid"), (heat >= 0.5, "#7fe3ff", "dashed")]:
        if 0 < mask.mean() < 1:
            axes[-1].contour(mask.astype(float), levels=[0.5], colors=[color], linewidths=1.4, linestyles=style)
    for axis in axes:
        image_axis(axis)


def cam_note(fig):
    fig.supxlabel(
        "Ô Grad-CAM: viền trắng là con vật theo trimap, viền xanh nét đứt là vùng Grad-CAM ≥ 0,5; IoU đo phần chung của hai vùng.",
        x=0.01, ha="left", fontsize=9.5, color=MUTED,
    )


def head_box(row):
    return transform_box(json.loads(row.bbox_json), row.width, row.height)


# ─── 21.2 Một ảnh bốn ô và mỗi giống một hàng ────────────────────────────────────────────────────────
def show_gradcam_example(study):
    model, entry = deployment_model(study), study.cache["val_cam"]
    rows = entry["frame"]
    index = int(np.flatnonzero(rows.has_bbox.to_numpy())[0])
    detail = cam_details(model, entry, [index], study.device)[0]
    fig, axes = plt.subplots(1, 4, figsize=(14, 4.1), layout="constrained")
    draw_cam_row(axes, entry["images"][index], entry["trimaps"][index], detail["heat"], head_box(rows.iloc[index]))
    for axis, title in zip(axes, ["Ảnh sau tiền xử lý 224×224", "Trimap: đen con vật, xám biên", "Hộp bao vùng đầu", "Grad-CAM"]):
        axis.set_title(title)
    labels, scores = cam_caption(entry, detail)
    figure_title(fig, f"{rows.image_id[index]} | {labels.replace(chr(10), ' | ')} | {scores}")
    cam_note(fig)
    show_figure(study, fig, "08_1_gradcam_validation_example", "Mục 4.8", "Grad-CAM một ảnh validation: ảnh, trimap, hộp bao, bản đồ nhiệt")


def show_gradcam_grid(study):
    model, entry = deployment_model(study), study.cache["val_cam"]
    rows = entry["frame"]
    indices = [
        int(np.flatnonzero(((rows.breed == breed) & rows.has_bbox).to_numpy())[0])
        for breed in CAT_BREEDS
        if ((rows.breed == breed) & rows.has_bbox).any()
    ]
    details = cam_details(model, entry, indices, study.device)
    fig, axes = plt.subplots(len(details), 4, figsize=(9.6, 2.6 * len(details)), squeeze=False, layout="constrained")
    for axes_row, detail in zip(axes, details):
        index = detail["index"]
        draw_cam_row(axes_row, entry["images"][index], entry["trimaps"][index], detail["heat"], head_box(rows.iloc[index]))
        labels, scores = cam_caption(entry, detail)
        axes_row[0].set_title(labels, fontsize=9)
        axes_row[3].set_title(scores, fontsize=9)
    axes[0][1].set_title("Trimap", fontsize=9)
    axes[0][2].set_title("Hộp bao vùng đầu", fontsize=9)
    figure_title(fig, "Grad-CAM mỗi giống một ảnh ID validation, checkpoint " + study.selection["deployment"]["name"])
    cam_note(fig)
    show_figure(study, fig, "08_2_gradcam_validation_breeds", "Mục 4.8", "Grad-CAM mỗi giống một ảnh validation với trimap và hộp bao")


# ─── Chạy ────────────────────────────────────────────────────────────────────────────────────────────
if AUTO_RUN:
    show_gradcam_example(STUDY)
    show_gradcam_grid(STUDY)


## 22. Công cụ pha B: bootstrap AUROC, bộ kiểm tra độ bền, dự đoán một ảnh

- **Làm:** định nghĩa bootstrap ghép cặp cho AUROC và Δ, năm nhóm biến đổi ảnh, hàm dự đoán một ảnh với ngưỡng đã khóa.
- Cell này chỉ định nghĩa hàm, chưa chạy gì.


In [ ]:
# ─── 22.1 Bootstrap AUROC ghép cặp và Δ gộp các checkpoint ───────────────────────────────────────────
def bootstrap_auroc(id_scores, ood_scores, label):
    # §2.7.5, Phụ lục B: lấy lại riêng ID và OOD, cùng chỉ số cho mọi phương pháp;
    # cùng seed và cùng n nên mọi checkpoint dùng cùng chỉ số lấy lại.
    id_matrix, ood_matrix = score_matrix(id_scores), score_matrix(ood_scores)
    rng = np.random.default_rng(CONFIG["bootstrap_seed"])
    samples = np.empty((CONFIG["bootstrap_repeats"], len(METHODS)))
    for _, repeat in progress(range(len(samples)), label):
        id_index = rng.integers(len(id_matrix), size=len(id_matrix))
        ood_index = rng.integers(len(ood_matrix), size=len(ood_matrix))
        samples[repeat] = auroc(id_matrix[id_index], ood_matrix[ood_index])
    return samples


def family_delta(values):
    # §2.7.5: Δ = trung bình AUROC (Mahalanobis, KNN) − trung bình AUROC (MSP, MaxLogit, Energy)
    distance = [METHODS.index(name) for name in DISTANCE_METHODS]
    logit = [METHODS.index(name) for name in LOGIT_METHODS]
    return values[..., distance].mean(axis=-1) - values[..., logit].mean(axis=-1)


def interval(samples):
    low, high = np.quantile(samples, [0.025, 0.975])
    return float(low), float(high)


def paired_confidence(point, samples, split):
    comparisons = {name: (point[i], samples[:, i]) for i, name in enumerate(METHODS)}
    for distance in DISTANCE_METHODS:
        for logit in LOGIT_METHODS:
            d, j = METHODS.index(distance), METHODS.index(logit)
            comparisons[distance + " - " + logit] = (point[d] - point[j], samples[:, d] - samples[:, j])
    comparisons["Delta_c"] = (family_delta(point), family_delta(samples))
    rows = []
    for name, (estimate, values) in comparisons.items():
        low, high = interval(values)
        rows.append(
            {
                "ood_split": split,
                "comparison": name,
                "estimate": float(estimate),
                "ci_low": low,
                "ci_high": high,
                "bootstrap_repeats": len(samples),
                "role": "hypothesis" if split == "near_ood" else "descriptive",  # QĐ6
            }
        )
    return pd.DataFrame(rows)


def pooled_delta(paths):
    points, samples, settings = [], [], set()
    for path in paths:
        with np.load(path, allow_pickle=False) as stored:
            if stored["methods"].tolist() != METHODS:
                raise ValueError("Thứ tự phương pháp trong mẫu bootstrap không khớp.")
            settings.add(
                (int(stored["n_id"]), int(stored["n_ood"]), int(stored["seed"]), len(stored["samples"]))
            )
            points.append(family_delta(stored["point"]))
            samples.append(family_delta(stored["samples"]))
    if len(settings) != 1:
        raise ValueError("Các checkpoint phải dùng cùng chỉ số bootstrap (cùng seed, cùng n).")
    low, high = interval(np.mean(samples, axis=0))  # Trọng số bằng nhau cho mỗi checkpoint
    return {
        "n_checkpoints": len(paths),
        "estimate": float(np.mean(points)),
        "ci_low": low,
        "ci_high": high,
        "bootstrap_repeats": len(samples[0]),
    }


# ─── 22.2 Bộ kiểm tra độ bền theo Bảng 2.8 và Phụ lục B ──────────────────────────────────────────────
ROBUSTNESS_LEVELS = {
    "rotation": [5, 10, 15, 30, 45],  # độ, trung bình +θ và −θ
    "photometric": [0.1, 0.2, 0.4, 0.6, 0.8],  # lệch sáng và tương phản, trung bình tăng và giảm
    "gaussian_blur": [0.5, 1.0, 1.5, 2.0, 3.0],  # σ, kernel 2⌈3σ⌉+1
    "gaussian_noise": [0.02, 0.04, 0.06, 0.08, 0.10],  # độ lệch chuẩn trên ảnh [0, 1]
    "jpeg": [90, 70, 50, 30, 10],  # chất lượng JPEG qua PIL
}
TWO_SIDED = {"rotation", "photometric"}
AUGMENTATION_BOUNDS = {"rotation": 15, "photometric": 0.2}  # QĐ10: ±15°, ±20%
ROTATION_FILL = [128 / 255] * 3  # Cùng nền xám với RandomRotation khi huấn luyện


def in_augmentation_range(family, value):
    return family in AUGMENTATION_BOUNDS and value <= AUGMENTATION_BOUNDS[family]


def jpeg_roundtrip(image, quality):
    buffer = BytesIO()
    Image.fromarray(image.permute(1, 2, 0).numpy()).save(buffer, format="JPEG", quality=quality)
    buffer.seek(0)
    with Image.open(buffer) as decoded:
        return TF.pil_to_tensor(decoded.convert("RGB"))


def image_noise(image_ids, shape):
    # §2.9.1: seed cố định theo ảnh (CRC32 của image_id)
    return torch.stack(
        [
            torch.randn(shape, generator=torch.Generator().manual_seed(zlib.crc32(name.encode())))
            for name in image_ids
        ]
    )


def corrupt(images, image_ids, family, value, direction=1):
    # §2.9.1: ảnh uint8 đã đưa về 224×224, trả về float [0, 1] trước chuẩn hóa
    if family == "jpeg":
        return torch.stack([jpeg_roundtrip(image, value) for image in images]).float() / 255
    pixels = images.float() / 255
    if family == "rotation":
        return TF.rotate(pixels, direction * value, interpolation=InterpolationMode.BILINEAR, fill=ROTATION_FILL)
    if family == "photometric":
        factor = 1 + direction * value  # Phụ lục B: độ sáng trước, độ tương phản sau
        return TF.adjust_contrast(TF.adjust_brightness(pixels, factor), factor)
    if family == "gaussian_blur":
        size = 2 * math.ceil(3 * value) + 1
        return TF.gaussian_blur(pixels, [size, size], [value, value])
    if family == "gaussian_noise":
        return (pixels + value * image_noise(image_ids, pixels.shape[1:])).clamp(0, 1)
    raise ValueError("Nhóm biến đổi không hợp lệ: " + family)


@torch.no_grad()
def robustness_suite(entries, study, test_correct):
    # Mỗi biến thể chỉ tạo một lần rồi đo trên mọi checkpoint đã khóa
    models = {
        entry["name"]: reload_checkpoint(study.root / entry["stage_dir"] / "checkpoint.pt", study.device)[0]
        for entry in entries
    }
    test = study.cache["test"]
    labels = torch.from_numpy(test["labels"])
    size = CONFIG["batch_size"]
    conditions = [
        (family, level, value)
        for family, values in ROBUSTNESS_LEVELS.items()
        for level, value in enumerate(values, 1)
    ]
    rows = []
    for _, (family, level, value) in progress(conditions, "Độ bền", leave=True):
        directions = (1, -1) if family in TWO_SIDED else (1,)
        correct = dict.fromkeys(models, 0)
        for direction in directions:
            for start in range(0, len(labels), size):
                pixels = corrupt(
                    test["images"][start:start + size], test["image_id"][start:start + size], family, value, direction
                )
                batch = NORMALIZE(pixels).to(study.device)
                for name, model in models.items():
                    correct[name] += int((model(batch).argmax(1).cpu() == labels[start:start + size]).sum())
        for entry in entries:
            accuracy = correct[entry["name"]] / (len(labels) * len(directions))
            clean = float(np.mean(test_correct[entry["name"]]))
            rows.append(
                {
                    **checkpoint_keys(entry),
                    "family": family,
                    "level": level,
                    "parameter": value,
                    "in_augmentation_range": in_augmentation_range(family, value),
                    "n_images": len(labels),
                    "accuracy": accuracy,
                    "clean_accuracy": clean,
                    "accuracy_drop": clean - accuracy,
                }
            )
    return pd.DataFrame(rows)


# ─── 22.3 Dự đoán một ảnh và áp dụng ngưỡng đã khóa ──────────────────────────────────────────────────
def load_inference_bundle(stage_dir, device="cpu"):
    stage_dir = Path(stage_dir)
    model, saved = reload_checkpoint(stage_dir / "checkpoint.pt", device)
    thresholds = read_json(stage_dir / "thresholds.json")
    if thresholds["checkpoint_sha256"] != file_hash(stage_dir / "checkpoint.pt"):
        raise ValueError("Ngưỡng không thuộc checkpoint này.")
    return {
        "model": model,
        "distance": DistanceScores.load(stage_dir / "distance_parameters.npz"),
        "thresholds": thresholds["T1"],
        "temperature": read_json(stage_dir / "calibration.json")["temperature"],
        "device": device,
        "is_smoke_test": saved["config"]["profile"] == "smoke",
    }


def predict_image(bundle, image_path, method, show_cam=True):
    tensor = make_transform()(read_rgb(image_path)).unsqueeze(0).to(bundle["device"])
    model = bundle["model"]
    with torch.no_grad():
        features = model.forward_features(tensor)
        logits = model.head(features).cpu().numpy()
    scores = rejection_scores({"features": features.cpu().numpy(), "logits": logits}, bundle["distance"])
    index = int(logits.argmax(1)[0])
    threshold = bundle["thresholds"][method]
    accepted = bool(scores[method][0] >= threshold)
    result = {
        "label": CAT_BREEDS[index] if accepted else "không xác định",
        "predicted_class_before_rejection": CAT_BREEDS[index],
        "accepted": accepted,
        "method": method,
        "score": float(scores[method][0]),
        "threshold": threshold,
        "confidence": float(softmax(logits / bundle["temperature"], axis=1).max()),
        "is_smoke_test": bundle["is_smoke_test"],
    }
    if show_cam:
        result["heatmap"] = gradcam(model, tensor)[0][0]
    return result


## 23. Pha B: đánh giá test và các tập OOD bằng lựa chọn đã khóa

- **Làm:** kiểm tra SHA-256 các tệp đã khóa, rồi đo từng checkpoint trên ID test, near-OOD, far-OOD: phân loại, năm phương pháp từ chối, hiệu chỉnh, Grad-CAM, độ bền, McNemar. Pha này không chọn gì.
- **Xuất:** thanh tiến độ, accuracy và F1 test của từng checkpoint.


In [ ]:
# ─── 23.1 Đánh giá một checkpoint đã khóa trên test và OOD ───────────────────────────────────────────
def ood_on_test(model, entry, test, locked, study):
    stage_dir, keys = study.root / entry["stage_dir"], checkpoint_keys(entry)
    distance, thresholds, temperature = locked["distance"], locked["thresholds"], locked["temperature"]
    test_scores = rejection_scores(test, distance)
    test_scaled = logit_scores(test["logits"], temperature)
    pd.DataFrame({"image_id": test["image_id"], **test_scores}).to_csv(stage_dir / "test_scores.csv", index=False)
    results, scaled_results, confidence = [], [], []
    for split in OOD_SPLITS:
        if split not in study.cache:
            continue
        label = entry["name"] + " " + split
        outside = extract_arrays(model, study.cache[split], study.device, label)
        save_npz(stage_dir / (split + ".npz"), outside)
        out_scores = rejection_scores(outside, distance)
        pd.DataFrame({"image_id": outside["image_id"], **out_scores}).to_csv(
            stage_dir / (split + "_scores.csv"), index=False
        )
        out_scaled = logit_scores(outside["logits"], temperature)
        results += [
            {"ood_split": split, "method": m, **ood_metrics(test_scores[m], out_scores[m], thresholds["T1"][m])}
            for m in METHODS
        ]
        scaled_results += [
            {
                "ood_split": split,
                "method": m,
                "temperature": temperature,
                **ood_metrics(test_scaled[m], out_scaled[m], thresholds["T_star"][m]),
            }
            for m in TEMPERATURE_METHODS
        ]
        point = auroc(score_matrix(test_scores), score_matrix(out_scores))
        samples = bootstrap_auroc(test_scores, out_scores, "Bootstrap " + label)
        np.savez(
            stage_dir / f"bootstrap_auroc_{split}.npz",
            samples=samples,
            point=point,
            methods=np.array(METHODS),
            n_id=len(test["labels"]),
            n_ood=len(outside["labels"]),
            seed=CONFIG["bootstrap_seed"],
        )
        confidence.append(paired_confidence(point, samples, split))
    outputs = {
        "ood_results": pd.DataFrame(results),
        "ood_results_temperature": pd.DataFrame(scaled_results),
        "ood_paired_confidence": pd.concat(confidence, ignore_index=True),
    }
    for name, frame in outputs.items():
        frame.to_csv(stage_dir / (name + ".csv"), index=False)
        study.tables[name].append(add_keys(frame, keys))
    delta = outputs["ood_paired_confidence"]
    study.tables["delta_by_checkpoint"].append(add_keys(delta[delta.comparison == "Delta_c"], keys))
    return test_scores


def calibration_on_test(entry, test, temperature, study):
    stage_dir = study.root / entry["stage_dir"]
    summary, before, after = calibration_summary(test["logits"], test["labels"], temperature)
    write_json(stage_dir / "test_calibration.json", summary)
    before.to_csv(stage_dir / "test_reliability_before.csv", index=False)
    after.to_csv(stage_dir / "test_reliability_after.csv", index=False)
    study.tables["test_calibration"].append(add_keys([summary], checkpoint_keys(entry)))


def coverage_on_test(stage_dir, test, test_scores):
    for method, scores in test_scores.items():
        accuracy_coverage(test, scores).to_csv(stage_dir / f"accuracy_coverage_{method}.csv", index=False)


def evaluate_rejection_test(model, entry, test, study):
    stage_dir, keys = study.root / entry["stage_dir"], checkpoint_keys(entry)
    locked = {
        "distance": DistanceScores.load(stage_dir / "distance_parameters.npz"),
        "thresholds": read_json(stage_dir / "thresholds.json"),
        "temperature": read_json(stage_dir / "calibration.json")["temperature"],
    }
    test_scores = ood_on_test(model, entry, test, locked, study)
    calibration_on_test(entry, test, locked["temperature"], study)
    coverage_on_test(stage_dir, test, test_scores)
    boundary = read_json(stage_dir / "gradcam/gradcam_summary.json")["iou_boundary"]  # QĐ5.7
    cam_summary, cam_result = gradcam_report(
        model, study.cache["test_cam"], study.device, stage_dir / "test_gradcam", "test", boundary
    )
    breeds = gradcam_by_breed(cam_result)
    breeds.to_csv(stage_dir / "test_gradcam/gradcam_by_breed.csv", index=False)
    background = view_comparison(
        model, test, study.cache["test_no_background"], study.device, entry["name"] + " test_no_background"
    )
    write_json(stage_dir / "background_removal.json", background)
    study.tables["gradcam_summary"].append(add_keys([cam_summary], keys))
    study.tables["gradcam_by_breed"].append(add_keys(breeds, keys))
    study.tables["background_removal"].append(add_keys([background], keys))


def evaluate_locked_checkpoint(entry, study):
    stage_dir, keys = study.root / entry["stage_dir"], checkpoint_keys(entry)
    verify_locked_files(stage_dir, entry["sha256"])
    model, _ = reload_checkpoint(stage_dir / "checkpoint.pt", study.device)
    test = extract_arrays(model, study.cache["test"], study.device, entry["name"] + " test")
    save_npz(stage_dir / "test.npz", test)
    summary, pairs = classification_outputs(test, stage_dir / "test_classification")
    study.tables["test_classification"].append(add_keys([summary], keys))
    study.tables["test_confidence"].append(add_keys(classification_confidence(test), keys))
    study.tables["confused_pairs"].append(add_keys(pairs, keys))
    if entry["evaluation"] == "full":
        evaluate_rejection_test(model, entry, test, study)
    if study.device.type == "cuda":
        torch.cuda.empty_cache()
    print(f"Pha B {entry['name']}: accuracy test {summary['accuracy']:.4f} | F1 macro {summary['macro_f1']:.4f}", flush=True)
    return test["logits"].argmax(1) == test["labels"]


def hypothesis_outputs(entries, study):
    pooled = {
        split: pooled_delta(
            [study.root / entry["stage_dir"] / f"bootstrap_auroc_{split}.npz" for entry in entries]
        )
        for split in OOD_SPLITS
        if split in study.cache
    }
    near = pooled.pop("near_ood")
    if near["ci_low"] > 0:  # §2.7.5
        near["conclusion"] = "ủng hộ giả thuyết"
    elif near["ci_high"] < 0:
        near["conclusion"] = "ngược chiều giả thuyết"
    else:
        near["conclusion"] = "chưa đủ bằng chứng"
    write_json(
        study.directory / "hypothesis_near_ood.json",
        {
            "definition": "mean over checkpoints of mean(AUROC Mahalanobis, KNN) - mean(AUROC MSP, MaxLogit, Energy)",
            "checkpoints": [entry["name"] for entry in entries],
            "near_ood": near,
            "descriptive": pooled,
        },
    )
    log_event("Kiểm định near-OOD gộp", "metrics", **near)


# ─── 23.2 Gộp bảng kết quả, trung bình ± độ lệch chuẩn qua 3 seed ────────────────────────────────────
VIEW_METRICS = [
    "full_accuracy",
    "view_accuracy",
    "accuracy_delta",
    "full_macro_f1",
    "view_macro_f1",
    "macro_f1_delta",
]
OOD_METRICS = ["auroc", "fpr_at_95_tpr", "tpr_at_threshold", "fpr_at_threshold", "n_ood_accepted"]
REPORT_TABLES = {  # tệp: (bảng nguồn, khóa nhóm, chỉ số)
    "classification_validation": ("validation_summary", ["model", "stage"], VALIDATION_METRICS + ["val_weighted_f1"]),
    "classification_test": (
        "test_classification",
        ["model", "stage"],
        ["accuracy", "macro_f1", "macro_precision", "macro_recall", "weighted_f1"],
    ),
    "confused_pairs": ("confused_pairs", ["model", "stage", "breed_a", "breed_b"], ["errors"]),
    "probe": ("probe_results", ["model", "stage", "method"], OOD_METRICS),
    "ood": ("ood_results", ["ood_split", "model", "stage", "method"], OOD_METRICS),
    "ood_temperature": ("ood_results_temperature", ["ood_split", "model", "stage", "method"], OOD_METRICS),
    "delta_by_checkpoint": ("delta_by_checkpoint", ["ood_split", "model", "stage"], ["estimate"]),
    "calibration": (
        "test_calibration",
        ["model", "stage"],
        ["temperature", "accuracy", "nll_before", "nll_after", "ece_before", "ece_after"],
    ),
    "gradcam": (
        "gradcam_summary",
        ["model", "stage"],
        ["miou", "miou_correct", "miou_incorrect", "miou_0.3", "miou_0.7", "mean_R", "mean_f", "R_above_uniform_fraction"],
    ),
    "gradcam_by_breed": (
        "gradcam_by_breed",
        ["model", "stage", "breed"],
        ["accuracy", "miou", "miou_correct", "miou_incorrect", "mean_R", "mean_f"],
    ),
    "head_crop": ("head_crop", ["model", "stage"], VIEW_METRICS),
    "background_removal": ("background_removal", ["model", "stage"], VIEW_METRICS),
    "robustness": (
        "robustness",
        ["model", "stage", "family", "level", "parameter", "in_augmentation_range"],
        ["accuracy", "accuracy_drop"],
    ),
    "augmentation_ablation": (
        "augmentation_ablation",
        ["model", "stage", "augmentation"],
        ["val_accuracy", "val_macro_f1", "test_accuracy", "test_macro_f1"],
    ),
}


def ablation_table(study):
    keys = ["model", "seed", "stage", "augmentation"]
    validation = collected(study, "validation_summary")
    test = collected(study, "test_classification").rename(
        columns={"accuracy": "test_accuracy", "macro_f1": "test_macro_f1"}
    )
    table = validation[keys + ["val_accuracy", "val_macro_f1"]].merge(
        test[keys + ["test_accuracy", "test_macro_f1"]], on=keys
    )
    return table[table.model != "cnn_baseline"].reset_index(drop=True)


def write_report_tables(study):
    folder = study.directory / "report_tables"
    for name, (source, keys, metrics) in REPORT_TABLES.items():
        frame = collected(study, source)
        if name != "augmentation_ablation":
            frame = frame[frame.augmentation]  # Bảng chính chỉ gồm checkpoint có tăng cường
        table = mean_std(frame, keys, metrics)
        if name == "confused_pairs":
            table = table.sort_values(["model", "stage", "errors_mean"], ascending=[True, True, False])
        table.to_csv(folder / (name + ".csv"), index=False)
    deployment_tradeoff(study).to_csv(folder / "deployment_tradeoff.csv", index=False)


# ─── 23.3 Điều phối pha B từ lựa chọn đã khóa trên đĩa ───────────────────────────────────────────────
def run_phase_b(study):
    if CONFIG["profile"] == "sweep":
        raise RuntimeError("Profile sweep không được giải mã test hay OOD test.")
    # Chỉ đọc lựa chọn đã khóa từ đĩa; không chọn gì trên test hay OOD
    entries = read_json(study.directory / "locked_selection.json")["checkpoints"]
    study.cache = run_step("Giải mã ảnh pha B", evaluation_cache, study.frames, PHASE_B_SETS)
    study.cache["test_cam"] = with_trimaps(study.cache["test"], study.frames["test"])
    test_correct = {entry["name"]: evaluate_locked_checkpoint(entry, study) for entry in entries}
    full = [entry for entry in entries if entry["evaluation"] == "full"]
    study.tables["robustness"].append(
        run_step("Kiểm tra độ bền", robustness_suite, full, study, test_correct)
    )
    study.tables["mcnemar"].append(mcnemar_table(full, test_correct))
    study.tables["augmentation_ablation"].append(ablation_table(study))
    hypothesis_outputs(full, study)
    write_tables(study)
    write_report_tables(study)
    for run_id in dict.fromkeys(entry["run_id"] for entry in entries):
        sync_run(study.root / run_id, study.api, "Pha B " + run_id)
    study.info.update(status="phase_b_completed", phase_b_finished_at_utc=utc_now())
    save_info(study.directory, study.info)


# ─── Chạy ────────────────────────────────────────────────────────────────────────────────────────────
if AUTO_RUN:
    run_phase_b(STUDY)


## 24. Kết quả phân loại trên ID test

- **Xuất:** bảng trung bình ± độ lệch chuẩn, CI 95%, bảng dung lượng và chất lượng, cặp giống dễ nhầm, kiểm định McNemar giữa các backbone.
- **Hình:** ma trận nhầm lẫn của cấu hình đã khóa.


In [ ]:
# ─── 24.1 Bảng phân loại và CI 95% ───────────────────────────────────────────────────────────────────
def configuration_entries(study, model, stage):
    return [
        entry
        for entry in study.selection["checkpoints"]
        if (entry["model"], entry["stage"], entry["augmentation"]) == (model, stage, True)
    ]


def locked_configurations(study):
    return list(
        dict.fromkeys((study.selection[role]["model"], study.selection[role]["stage"]) for role in ["deployment", "best_unconstrained"])
    )


def show_classification_results(study):
    metrics = ["accuracy", "macro_f1", "macro_precision", "macro_recall", "weighted_f1"]
    display(mean_std_text(report_table(study, "classification_test"), metrics))
    confidence = collected(study, "test_confidence")
    confidence = confidence[confidence.augmentation].assign(
        ci=lambda f: [f"{e:.4f} [{low:.4f}; {high:.4f}]" for e, low, high in zip(f.estimate, f.ci_low, f.ci_high)]
    )
    display(confidence.pivot_table(index=["model", "stage", "seed"], columns="metric", values="ci", aggfunc="first"))
    tradeoff = report_table(study, "deployment_tradeoff")
    display(mean_std_text(tradeoff, ["val_accuracy", "val_macro_f1", "test_accuracy", "test_macro_f1"]))


# ─── 24.2 Ma trận nhầm lẫn cộng các seed và cặp giống dễ nhầm ────────────────────────────────────────
def confusion_figure(matrix, title):
    fig, axis = plt.subplots(figsize=(8.8, 7.6))
    graphic = axis.imshow(matrix, cmap="Blues")
    names = [name.replace("_", " ") for name in CAT_BREEDS]
    axis.set_xticks(CLASS_INDICES, names, rotation=45, ha="right")
    axis.set_yticks(CLASS_INDICES, names)
    axis.grid(False)
    for spine in axis.spines.values():
        spine.set_visible(False)
    for (row, column), value in np.ndenumerate(matrix):
        if value:
            color = "white" if value > matrix.max() / 2 else INK
            axis.text(column, row, int(value), ha="center", va="center", fontsize=9, color=color)
    axis.set(xlabel="Giống dự đoán", ylabel="Giống thật", title=title)
    fig.colorbar(graphic, ax=axis, fraction=0.046, pad=0.03, label="Số ảnh")
    return fig


def show_confusion_matrices(study):
    pairs = report_table(study, "confused_pairs")
    for model, stage in locked_configurations(study):
        entries = configuration_entries(study, model, stage)
        matrix = sum(
            pd.read_csv(study.root / entry["stage_dir"] / "test_classification/confusion_matrix.csv", index_col=0).to_numpy()
            for entry in entries
        )
        label = configuration_label(model, stage)
        show_figure(
            study,
            confusion_figure(matrix, f"Ma trận nhầm lẫn ID test, {label}, cộng {len(entries)} seed"),
            f"10_02_confusion_{model}_{stage}",
            "Mục 4.4",
            "Ma trận nhầm lẫn ID test cộng các seed, " + label,
        )
        top = pairs[(pairs.model == model) & (pairs.stage == stage)].nlargest(10, "errors_mean")
        display(top[["breed_a", "breed_b", "errors_mean", "errors_std"]].round(2))


# ─── 24.3 Kiểm định McNemar giữa các backbone ────────────────────────────────────────────────────────
def show_mcnemar(study):
    table = collected(study, "mcnemar")
    table = table.assign(
        model_a=table.model_a.map(MODEL_LABELS), model_b=table.model_b.map(MODEL_LABELS), stage=table.stage.map(STAGE_LABELS)
    )
    display(table.round(4))


# ─── Chạy ────────────────────────────────────────────────────────────────────────────────────────────
if AUTO_RUN:
    show_classification_results(STUDY)
    show_confusion_matrices(STUDY)
    show_mcnemar(STUDY)


## 25. Kết quả phát hiện OOD

- **Xuất:** bảng năm phương pháp trên near-OOD và far-OOD (T = 1 và T*); bảng kiểm định Δ.
- **Hình:** ba miền dữ liệu; AUROC theo phương pháp; histogram điểm kèm τ; accuracy–coverage; Δ AUROC từng checkpoint.


In [ ]:
# ─── 25.1 Ba miền dữ liệu: mèo, chó và kết cấu DTD ───────────────────────────────────────────────────
def show_domains(study):
    fig, axes = plt.subplots(3, 4, figsize=(12, 9.6), layout="constrained")
    for axes_row, (label, name) in zip(
        axes, [("Mèo, ID test", "test"), ("Chó, near-OOD", "near_ood"), ("Kết cấu DTD, far-OOD", "far_ood")]
    ):
        images = study.cache[name]["images"]
        for axis, index in zip(axes_row, np.linspace(0, len(images) - 1, 4).astype(int)):
            axis.imshow(images[index].permute(1, 2, 0).numpy())
            image_axis(axis)
        axes_row[0].set_title(label)
    figure_title(fig, "Ba miền dữ liệu sau tiền xử lý đánh giá 224×224")
    show_figure(study, fig, "10_13_domains", "Chương 3", "So sánh trực quan mèo (ID), chó (near-OOD) và kết cấu DTD (far-OOD)")


# ─── 25.2 Bảng chỉ số và AUROC theo phương pháp, cấu hình ────────────────────────────────────────────
def show_ood_results(study):
    for name in ["ood", "ood_temperature"]:
        print("Bảng", name, "| trung bình ± độ lệch chuẩn qua seed")
        display(mean_std_text(report_table(study, name), OOD_METRICS, 3))
    table = report_table(study, "ood")
    splits = [split for split in OOD_SPLITS if split in set(table.ood_split)]
    configurations = list(dict.fromkeys(zip(table.model, table.stage)))
    step = 0.8 / len(configurations)
    positions = np.arange(len(METHODS))
    fig, axes = plt.subplots(
        1, len(splits), figsize=(6.2 * len(splits), 4.6), sharey=True, squeeze=False, layout="constrained"
    )
    for axis, split in zip(axes[0], splits):
        part = table[table.ood_split == split]
        for k, (model, stage) in enumerate(configurations):
            rows = part[(part.model == model) & (part.stage == stage)].set_index("method").reindex(METHODS)
            mean_markers(
                axis,
                positions + (k - (len(configurations) - 1) / 2) * step,
                rows.auroc_mean,
                rows.auroc_std.fillna(0),
                configuration_style(model, stage),
                configuration_label(model, stage),
            )
        axis.axhline(0.5, color=NEUTRAL, linewidth=1, linestyle=":")
        for boundary in positions[:-1] + 0.5:
            axis.axvline(boundary, color=GRID, linewidth=0.8)
        axis.set_xticks(positions, METHODS)
        axis.grid(axis="x", visible=False)
        axis.set_title(OOD_LABELS[split])
    lowest = float((table.auroc_mean - table.auroc_std.fillna(0)).min())
    axes[0][0].set(ylabel="AUROC, trung bình ± độ lệch chuẩn qua seed", ylim=(max(0.0, min(0.45, lowest - 0.03)), 1.02))
    axes[0][-1].annotate(
        "0,5: đoán ngẫu nhiên", (1.01, 0.5), xycoords=("axes fraction", "data"), va="center", fontsize=8.5, color=MUTED,
        annotation_clip=False,
    )
    handles, labels = axes[0][0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=min(len(labels), 3), bbox_to_anchor=(0.5, 1.0))
    show_figure(study, fig, "10_03_ood_auroc", "Mục 4.6", "AUROC theo phương pháp và chế độ trên từng tập OOD, trung bình ± độ lệch chuẩn")


# ─── 25.3 Histogram điểm và accuracy–coverage của checkpoint triển khai ──────────────────────────────
def show_score_histograms(study):
    folder = deployment_dir(study)
    test = pd.read_csv(folder / "test_scores.csv")
    sets = {
        OOD_LABELS[split]: pd.read_csv(folder / (split + "_scores.csv"))
        for split in OOD_SPLITS
        if (folder / (split + "_scores.csv")).is_file()
    }
    thresholds = read_json(folder / "thresholds.json")["T1"]
    show_figure(
        study,
        score_histogram_figure(test, sets, thresholds, "ID test"),
        "10_04_score_histograms",
        "Mục 4.6",
        "Histogram điểm ID test và từng tập OOD, checkpoint triển khai " + study.selection["deployment"]["name"],
    )


def show_coverage(study):
    folder, name = deployment_dir(study), study.selection["deployment"]["name"]
    fig, axis = plt.subplots(figsize=(7, 5))
    for method in METHODS:
        curve = pd.read_csv(folder / f"accuracy_coverage_{method}.csv")
        axis.plot(curve.coverage, curve.accuracy, color=METHOD_COLORS[method], linestyle=METHOD_LINES[method], label=method)
    axis.set(
        xlabel="Coverage: tỷ lệ ảnh ID test được chấp nhận",
        ylabel="Accuracy trên ảnh được chấp nhận",
        title="Accuracy–coverage trên ID test, " + name,
    )
    legend_above(axis)
    show_figure(study, fig, "10_05_accuracy_coverage", "Mục 4.6", "Accuracy–coverage năm phương pháp, checkpoint triển khai")


# ─── 25.4 Kiểm định near-OOD: Δ từng checkpoint và trung bình gộp ────────────────────────────────────
def show_hypothesis(study):
    hypothesis = read_json(study.directory / "hypothesis_near_ood.json")
    pooled = {"near_ood": hypothesis["near_ood"], **hypothesis["descriptive"]}
    display(pd.DataFrame([{"Tập": OOD_LABELS[split], **values} for split, values in pooled.items()]))
    delta = collected(study, "delta_by_checkpoint")
    splits = [split for split in OOD_SPLITS if split in pooled]
    rows = delta.ood_split.value_counts().max() + 1
    fig, axes = plt.subplots(
        1, len(splits), figsize=(6 * len(splits), 0.36 * rows + 2), sharey=True, squeeze=False, layout="constrained"
    )
    for axis, split in zip(axes[0], splits):
        part = delta[delta.ood_split == split].sort_values(["model", "stage", "seed"])
        positions = np.arange(len(part))
        axis.errorbar(
            part.estimate, positions, xerr=[part.estimate - part.ci_low, part.ci_high - part.estimate],
            fmt="o", color=MUTED, markersize=5, elinewidth=1.2, capsize=2, label="Từng checkpoint",
        )
        summary = pooled[split]
        axis.errorbar(
            [summary["estimate"]], [len(part) + 0.5],
            xerr=[[summary["estimate"] - summary["ci_low"]], [summary["ci_high"] - summary["estimate"]]],
            fmt="D", color=BLUE, markersize=9, elinewidth=2, capsize=4, label="Gộp, trọng số bằng nhau",
        )
        axis.axvline(0, color=INK, linewidth=1)
        axis.grid(axis="y", visible=False)
        labels = [f"{configuration_label(m, s)} s{seed}" for m, s, seed in zip(part.model, part.stage, part.seed)]
        axis.set_yticks([*positions, len(part) + 0.5], [*labels, f"Gộp {summary['n_checkpoints']} checkpoint"])
        axis.set(xlabel="Δ AUROC (khoảng cách − logits), CI 95%", title=OOD_LABELS[split])
    axes[0][0].invert_yaxis()
    fig.legend(*axes[0][0].get_legend_handles_labels(), loc="lower center", ncol=2, bbox_to_anchor=(0.5, 1.0))
    show_figure(study, fig, "10_11_hypothesis_forest", "Mục 4.6", "Δ AUROC từng checkpoint và trung bình gộp kèm CI 95% (§2.7.5)")


# ─── Chạy ────────────────────────────────────────────────────────────────────────────────────────────
if AUTO_RUN:
    show_domains(STUDY)
    show_ood_results(STUDY)
    show_score_histograms(STUDY)
    show_coverage(STUDY)
    show_hypothesis(STUDY)


## 26. Kết quả hiệu chỉnh nhiệt độ

- **Xuất:** bảng T*, NLL và ECE trước và sau khi chia logits cho T*.
- **Hình:** reliability diagram trước và sau T* của checkpoint triển khai.


In [ ]:
def show_calibration(study):
    metrics = ["temperature", "accuracy", "nll_before", "nll_after", "ece_before", "ece_after"]
    display(mean_std_text(report_table(study, "calibration"), metrics))
    folder = deployment_dir(study)
    summary = read_json(folder / "test_calibration.json")
    width = 1 / CONFIG["ece_bins"]
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 5.2), sharex=True, sharey=True, layout="constrained")
    for axis, when, title in zip(axes, ["before", "after"], ["T = 1", f"T* = {summary['temperature']:.3f}"]):
        bins = pd.read_csv(folder / f"test_reliability_{when}.csv")
        axis.bar((bins.bin + 0.5) * width, bins.accuracy, width=0.92 * width, color=BLUE, label="Accuracy của từng khoảng")
        axis.plot([0, 1], [0, 1], color=INK, linewidth=1.2, linestyle="--", label="Hiệu chỉnh hoàn hảo")
        axis.set(
            xlabel="Độ tin cậy (xác suất lớn nhất)",
            title=f"{title}: ECE {summary['ece_' + when]:.3f}, NLL {summary['nll_' + when]:.3f}",
            xlim=(0, 1), ylim=(0, 1), aspect="equal",
        )
    axes[0].set_ylabel("Accuracy")
    legend_above(axes[0], row=axes)
    figure_title(fig, "Reliability diagram ID test, " + study.selection["deployment"]["name"])
    show_figure(study, fig, "10_06_reliability", "Mục 4.7", "Reliability diagram ID test trước và sau T*, checkpoint triển khai")


if AUTO_RUN:
    show_calibration(STUDY)


## 27. Kết quả Grad-CAM trên ID test

- **Xuất:** bảng mIoU, R, f theo cấu hình và theo giống.
- **Hình:** phân bố IoU và R so với f; ví dụ ca dự đoán đúng và ca sai.


In [ ]:
# ─── 27.1 Bảng mIoU, R, f và hình phân bố ────────────────────────────────────────────────────────────
GRADCAM_METRICS = ["miou", "miou_correct", "miou_incorrect", "miou_0.3", "miou_0.7", "mean_R", "mean_f"]


def show_gradcam_results(study):
    display(mean_std_text(report_table(study, "gradcam"), GRADCAM_METRICS + ["R_above_uniform_fraction"], 3))
    deployment = study.selection["deployment"]
    breeds = report_table(study, "gradcam_by_breed")
    breeds = breeds[(breeds.model == deployment["model"]) & (breeds.stage == deployment["stage"])]
    metrics = ["accuracy", "miou", "mean_R", "mean_f"]
    columns = ["breed", "n_seeds"] + [f"{metric}_{s}" for metric in metrics for s in ["mean", "std"]]
    display(mean_std_text(breeds[columns], metrics, 3))
    result = pd.read_csv(deployment_dir(study) / "test_gradcam/gradcam_per_image.csv")
    groups = [(True, BLUE, "o", 16, 0.6, "Dự đoán đúng"), (False, ORANGE, "X", 44, 0.95, "Dự đoán sai")]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), layout="constrained")
    axes[0].hist(
        [result.loc[result.correct == correct, "iou"] for correct, *_ in groups],
        bins=np.linspace(0, 1, 11),
        color=[color for _, color, *_ in groups],
        label=[label for *_, label in groups],
    )
    axes[0].set(xlabel="IoU giữa vùng Grad-CAM ≥ 0,5 và con vật", ylabel="Số ảnh", title="Phân bố IoU trên ID test")
    legend_above(axes[0])
    axes[1].plot([0, 1], [0, 1], color=NEUTRAL, linewidth=1, linestyle=":", label="R = f: bản đồ trải đều")
    for correct, color, marker, size, alpha, label in groups:
        chosen = result[(result.correct == correct) & result.R.notna()]
        axes[1].scatter(
            chosen.f, chosen.R, s=size, marker=marker, color=color, alpha=alpha, edgecolors="white", linewidths=0.4, label=label
        )
    axes[1].set(
        xlabel="Tỷ lệ diện tích con vật f", ylabel="Tỷ lệ năng lượng Grad-CAM trên con vật R",
        xlim=(0, 1), ylim=(0, 1), title="R so với f trên ID test",
    )
    legend_above(axes[1])
    figure_title(fig, "Grad-CAM, checkpoint " + deployment["name"])
    show_figure(study, fig, "10_07_iou_r_f", "Mục 4.8", "Phân bố IoU và R so với f trên ID test, checkpoint triển khai")
    show_test_cam_examples(study, result)


# ─── 27.2 Ví dụ ca dự đoán đúng và ca sai ────────────────────────────────────────────────────────────
def show_test_cam_examples(study, result):
    entry = study.cache["test_cam"]
    chosen = pd.concat([result[result.correct].head(3), result[~result.correct].head(3)])
    indices = pd.Index(entry["image_id"]).get_indexer(chosen.image_id).tolist()
    details = cam_details(deployment_model(study), entry, indices, study.device)
    fig, axes = plt.subplots(len(details), 3, figsize=(7.8, 2.7 * len(details)), squeeze=False, layout="constrained")
    for axes_row, detail in zip(axes, details):
        draw_cam_row(axes_row, entry["images"][detail["index"]], entry["trimaps"][detail["index"]], detail["heat"])
        labels, scores = cam_caption(entry, detail)
        axes_row[0].set_title(labels, fontsize=9)
        axes_row[2].set_title(scores, fontsize=9)
    axes[0][1].set_title("Trimap", fontsize=9)
    figure_title(fig, "Grad-CAM trên ID test: ca dự đoán đúng rồi ca sai")
    cam_note(fig)
    show_figure(study, fig, "10_07_gradcam_test_examples", "Mục 4.8", "Grad-CAM ID test: ảnh, trimap, bản đồ nhiệt cho ca đúng và sai")


# ─── Chạy ────────────────────────────────────────────────────────────────────────────────────────────
if AUTO_RUN:
    show_gradcam_results(STUDY)


## 28. Kết quả độ bền, tăng cường dữ liệu, cắt vùng đầu và xóa nền

- **Xuất:** bảng mức giảm accuracy theo biến đổi; bảng có và không có tăng cường; bảng cắt vùng đầu và xóa nền.
- **Hình:** accuracy theo mức biến đổi; accuracy có và không có tăng cường.


In [ ]:
# ─── 28.1 Độ bền theo mức biến đổi ───────────────────────────────────────────────────────────────────
ROBUSTNESS_LABELS = {
    "rotation": ("Xoay nghiêng", "{:g}°"),
    "photometric": ("Sáng và tương phản", "±{:.0%}"),
    "gaussian_blur": ("Gaussian blur", "σ {:g}"),
    "gaussian_noise": ("Nhiễu Gauss", "std {:g}"),
    "jpeg": ("Nén JPEG", "q{:g}"),
}


def show_robustness(study):
    table = report_table(study, "robustness")
    clean = report_table(study, "classification_test").set_index(["model", "stage"]).accuracy_mean
    display(table.pivot_table(index=["model", "stage"], columns=["family", "level"], values="accuracy_drop_mean").round(3))
    configurations = list(dict.fromkeys(zip(table.model, table.stage)))
    fig, axes = plt.subplots(
        1, len(ROBUSTNESS_LEVELS), figsize=(3.9 * len(ROBUSTNESS_LEVELS), 4.4), sharey=True, layout="constrained"
    )
    for axis, (family, values) in zip(axes, ROBUSTNESS_LEVELS.items()):
        title, pattern = ROBUSTNESS_LABELS[family]
        in_range = [level for level, value in enumerate(values, 1) if in_augmentation_range(family, value)]
        if in_range:
            axis.axvspan(0.5, max(in_range) + 0.5, color="#f0efec")
            axis.text(
                0.5 + max(in_range) / 2, 0.02, "trong phạm vi\ntăng cường", transform=axis.get_xaxis_transform(),
                ha="center", va="bottom", fontsize=8.5, color=MUTED,
            )
        for model, stage in configurations:
            rows = table[(table.family == family) & (table.model == model) & (table.stage == stage)].sort_values("level")
            axis.plot(
                [0, *rows.level],
                [clean[(model, stage)], *rows.accuracy_mean],
                label=configuration_label(model, stage),
                **configuration_style(model, stage),
            )
        axis.set_xticks(range(len(values) + 1), ["sạch"] + [pattern.format(value) for value in values], rotation=35, ha="right")
        axis.set(title=title, xlabel="Mức biến đổi")
    axes[0].set_ylabel("Accuracy ID test, trung bình qua seed")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=min(len(labels), 4), bbox_to_anchor=(0.5, 1.0))
    show_figure(study, fig, "10_08_robustness", "Mục 4.9", "Accuracy ID test theo mức của năm nhóm biến đổi, vùng xám là mức trong phạm vi tăng cường")


# ─── 28.2 Ảnh hưởng của tăng cường dữ liệu ───────────────────────────────────────────────────────────
def show_ablation(study):
    table = report_table(study, "augmentation_ablation")
    display(mean_std_text(table, ["val_accuracy", "val_macro_f1", "test_accuracy", "test_macro_f1"]))
    configurations = list(dict.fromkeys(zip(table.model, table.stage)))
    series = []
    for augmentation, label, style in [
        (True, "Có tăng cường", {"color": BLUE, "marker": "o"}),
        (False, "Không tăng cường", {"color": NEUTRAL, "marker": "o", "markerfacecolor": "white", "markeredgewidth": 1.5}),
    ]:
        rows = table[table.augmentation == augmentation].set_index(["model", "stage"]).reindex(configurations)
        series.append((label, rows.test_accuracy_mean.to_numpy(), rows.test_accuracy_std.to_numpy(), style))
    fig, axis = plt.subplots(figsize=(9, 0.55 * len(configurations) + 1.6))
    dot_rows(axis, [configuration_label(model, stage) for model, stage in configurations], series)
    axis.set(xlabel="Accuracy ID test, trung bình ± độ lệch chuẩn qua seed", title="Có và không có tăng cường dữ liệu")
    legend_above(axis)
    show_figure(study, fig, "10_09_augmentation_ablation", "Mục 4.5", "Accuracy ID test có và không có tăng cường dữ liệu")


# ─── 28.3 Cắt vùng đầu và xóa nền ────────────────────────────────────────────────────────────────────
def show_view_experiments(study):
    for name, caption in [("head_crop", "Cắt vùng đầu, ID validation"), ("background_removal", "Xóa nền, ID test có trimap hợp lệ")]:
        print(caption, "| chỉ suy luận, so với ảnh đầy đủ trên cùng ảnh")
        display(mean_std_text(report_table(study, name), VIEW_METRICS))


# ─── Chạy ────────────────────────────────────────────────────────────────────────────────────────────
if AUTO_RUN:
    show_robustness(STUDY)
    show_ablation(STUDY)
    show_view_experiments(STUDY)


## 29. Xuất tệp zip kết quả

- **Xuất:** danh sách tệp trong zip (hình, bảng, tệp khóa) và liên kết tải về.


In [ ]:
def export_results(study):
    pd.DataFrame(study.figures).to_csv(study.directory / "figure_index.csv", index=False)
    members = [
        *sorted((study.directory / "figures").glob("*.png")),
        *sorted((study.directory / "report_tables").glob("*.csv")),
        *sorted(study.directory.glob("*.csv")),  # bảng theo checkpoint, CI, McNemar, mục lục hình
        *(study.directory / name for name in ["locked_selection.json", "hypothesis_near_ood.json"]),
    ]
    target = Path(CONFIG["output_root"]).resolve().parent / (study.directory.name + "_results.zip")
    with zipfile.ZipFile(target, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in members:
            archive.write(path, path.relative_to(study.directory).as_posix())
    with zipfile.ZipFile(target) as archive:
        listing = pd.DataFrame(
            [{"Tệp": item.filename, "KB": round(item.file_size / 1024, 1)} for item in archive.infolist()]
        )
    display(listing)
    print(f"{target} | {len(listing)} tệp | {target.stat().st_size / 1e6:.2f} MB", flush=True)
    study.info.update(status="completed", finished_at_utc=utc_now(), results_zip=target.name)
    save_info(study.directory, study.info)
    sync_run(study.directory, study.api, "Tổng hợp")
    display(FileLink(os.path.relpath(target)))
    return target


if AUTO_RUN:
    RESULTS_ZIP = export_results(STUDY)


## 30. Dọn Output trên Kaggle

- **Làm:** xóa khỏi Output các thư mục đã đồng bộ lên Hugging Face và bản clone mã nguồn; thư mục chưa đồng bộ được giữ lại để upload lại.
- **Xuất:** danh sách những gì còn lại trong Output (tệp zip kết quả là thứ cần tải về).


In [ ]:
def clean_kaggle_output(study):
    # Output của Kaggle chỉ cần tệp zip kết quả. Ở lượt official, checkpoint và hồ sơ từng lượt đã nằm trên
    # Hugging Face; thư mục nào chưa đồng bộ thành công thì được giữ lại để upload lại (quy tắc của nhóm).
    # Ở lượt smoke, kết quả chỉ để kiểm tra code nên xóa hết.
    if not IS_KAGGLE or not CONFIG["clean_kaggle_output"]:
        print("Giữ nguyên các thư mục kết quả.", flush=True)
        return
    root, kept = Path(CONFIG["output_root"]), []
    for folder in sorted(path for path in root.iterdir() if path.is_dir()):
        group_check = "_smoke_" in folder.name  # thư mục kiểm tra workflow của cell gốc thứ 4, đã tự đồng bộ
        disposable = CONFIG["profile"] == "smoke" or folder.name == "data_manifest" or group_check
        if disposable or SYNC_STATUS.get(folder.name):
            shutil.rmtree(folder)
        else:
            kept.append(folder.name)
    clone = globals().get("WORK_DIR")  # bản clone mã nguồn do cell gốc thứ 2 tạo
    if clone is not None and Path(clone).is_dir():
        shutil.rmtree(clone)
    print("Output còn lại:", ", ".join(sorted(path.name for path in root.parent.iterdir())), flush=True)
    if kept:
        print("Giữ lại vì chưa đồng bộ lên Hugging Face:", ", ".join(kept), flush=True)


if AUTO_RUN:
    clean_kaggle_output(STUDY)
